In [ ]:
# ============================================================
# Portable Research Platform Bootstrap - Canonical v1.2
# ============================================================
"""
MANDATORY FIRST CELL.

One notebook version works in:
- local VS Code / Jupyter;
- Google Drive desktop sync;
- Google Colab with Drive mounted;
- Colab transient clone under /content.

Best practice: keep the full research_platform_definitive folder on Google Drive at
MyDrive/machine-learning-for-trading/research_platform_definitive or
MyDrive/GitHub/machine-learning-for-trading/research_platform_definitive.
If Colab cannot find it, this cell can clone the GitHub repo into /content as a fallback.
"""

from pathlib import Path
import os
import subprocess
import sys


DEFAULT_GIT_URL = os.environ.get(
    "RESEARCH_PLATFORM_GIT_URL",
    "https://github.com/TheGenesisAIStory/ml-trading-thesis-bot.git",
)


def _has_platform_sentinel(path):
    path = Path(path).expanduser()
    return (
        (path / "src" / "research_platform_core").exists()
        or (path / "src" / "research_platform_core.py").exists()
    )


def _candidate_roots():
    cwd = Path.cwd().resolve()
    candidates = []

    env_root = os.environ.get("RESEARCH_PLATFORM_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    drive_desktop_candidates = [
        Path.home() / "Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/GitHub/machine-learning-for-trading/research_platform_definitive",
        Path.home() / "Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/machine-learning-for-trading/research_platform_definitive",
    ]

    local_mirror_candidates = []
    for p in [cwd, *cwd.parents]:
        local_mirror_candidates.append(p)
        local_mirror_candidates.append(p / "research_platform_definitive")
    local_mirror_candidates.append(Path.home() / "GitHub/machine-learning-for-trading/research_platform_definitive")

    colab_candidates = [
        Path("/content/drive/MyDrive/GitHub/machine-learning-for-trading/research_platform_definitive"),
        Path("/content/drive/MyDrive/machine-learning-for-trading/research_platform_definitive"),
        Path("/content/drive/MyDrive/research_platform_definitive"),
        Path("/content/machine-learning-for-trading/research_platform_definitive"),
        Path("/content/ml-trading-thesis-bot/research_platform_definitive"),
        Path("/content/research_platform_definitive"),
    ]

    prefer_drive = os.environ.get("RESEARCH_PLATFORM_STORAGE_MODE", "drive").strip().lower() != "local"
    if _is_colab():
        candidates.extend(colab_candidates)
        candidates.extend(drive_desktop_candidates)
        candidates.extend(local_mirror_candidates)
    elif prefer_drive:
        candidates.extend(drive_desktop_candidates)
        candidates.extend(local_mirror_candidates)
        candidates.extend(colab_candidates)
    else:
        candidates.extend(local_mirror_candidates)
        candidates.extend(drive_desktop_candidates)
        candidates.extend(colab_candidates)

    deduped = []
    seen = set()
    for p in candidates:
        key = str(p.expanduser())
        if key not in seen:
            deduped.append(p)
            seen.add(key)
    return deduped


def _find_project_root():
    for candidate in _candidate_roots():
        candidate = candidate.expanduser()
        if _has_platform_sentinel(candidate):
            return candidate.resolve()
        nested = candidate / "research_platform_definitive"
        if _has_platform_sentinel(nested):
            return nested.resolve()
    return None


def _is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _mount_drive_if_colab(verbose=True):
    if not _is_colab():
        return
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            if verbose:
                print("Mounting Google Drive...")
            drive.mount("/content/drive")
    except Exception as exc:
        if verbose:
            print(f"Drive mount skipped/failed: {exc}")


def _clone_repo_fallback(verbose=True):
    if not _is_colab():
        return None
    if os.environ.get("RESEARCH_PLATFORM_AUTO_CLONE", "1") in {"0", "false", "False"}:
        return None

    target = Path(os.environ.get("RESEARCH_PLATFORM_CLONE_ROOT", "/content/machine-learning-for-trading"))
    if _has_platform_sentinel(target / "research_platform_definitive"):
        return (target / "research_platform_definitive").resolve()

    if target.exists() and not (target / ".git").exists():
        return None

    try:
        if target.exists():
            if verbose:
                print(f"Updating existing clone: {target}")
            subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=False)
        else:
            if verbose:
                print(f"Cloning research platform repo into {target}...")
            subprocess.run(["git", "clone", "--depth", "1", DEFAULT_GIT_URL, str(target)], check=True)
    except Exception as exc:
        if verbose:
            print(f"Git clone fallback failed: {exc}")
        return None

    root = target / "research_platform_definitive"
    return root.resolve() if _has_platform_sentinel(root) else None


def _first_existing_path(candidates, default):
    for candidate in candidates:
        candidate = Path(candidate).expanduser()
        if candidate.exists():
            return candidate
    return default


def _ensure_writable_dir(path, fallback):
    for candidate in [Path(path).expanduser(), Path(fallback).expanduser(), Path("/tmp/research_platform_output")]:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink(missing_ok=True)
            return candidate
        except Exception:
            continue
    raise OSError("No writable output/cache directory available.")


def setup_colab_environment(verbose=True):
    _mount_drive_if_colab(verbose=verbose)
    project_root = _find_project_root()
    if project_root is None:
        project_root = _clone_repo_fallback(verbose=verbose)

    if project_root is None:
        searched = "\n".join(f"- {p.expanduser()}" for p in _candidate_roots())
        raise FileNotFoundError(
            "PROJECT_ROOT not found. This notebook needs the full research_platform_definitive folder, not only the notebook.\n\n"
            "Best fix: sync this folder to Google Drive:\n"
            "  MyDrive/machine-learning-for-trading/research_platform_definitive\n\n"
            "Alternative: set RESEARCH_PLATFORM_GIT_URL and let Colab clone the repo into /content.\n\n"
            f"Searched:\n{searched}"
        )

    for rel in ["", "src", "company_valuation/src", "portfolio_analysis/src"]:
        path = str(project_root / rel)
        if path not in sys.path:
            sys.path.insert(0, path)

    financial_db_root = _first_existing_path(
        [
            Path(os.environ.get("FINANCIAL_DB_ROOT", "")) if os.environ.get("FINANCIAL_DB_ROOT") else Path("__missing__"),
            Path("/content/drive/MyDrive/Database Finanziario"),
            Path.home() / "Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/Database Finanziario",
        ],
        Path("/content/drive/MyDrive/Database Finanziario") if _is_colab() else project_root / "local_databases_not_on_drive" / "database",
    )

    output_root = _ensure_writable_dir(
        Path(os.environ.get("RESEARCH_PLATFORM_OUTPUT_ROOT", project_root / "output")),
        Path("/content/research_platform_output") if _is_colab() else project_root / "output",
    )
    local_cache = _ensure_writable_dir(
        Path(os.environ.get("RESEARCH_PLATFORM_LOCAL_CACHE", output_root / "data_cache")),
        Path("/content/research_platform_cache") if _is_colab() else output_root / "data_cache",
    )

    config = {
        "environment": "colab" if _is_colab() else "local",
        "PROJECT_ROOT": project_root,
        "FINANCIAL_DB_ROOT": financial_db_root,
        "DB_BASE": financial_db_root,
        "DATA_PATH": financial_db_root,
        "OUTPUTROOT": output_root,
        "OUTPUT_ROOT": output_root,
        "LOCAL_CACHE_ROOT": local_cache,
        "DATA_LOCAL": local_cache,
    }

    for key in ["FINANCIAL_DB_ROOT", "DB_BASE", "DATA_PATH", "RESEARCH_PLATFORM_OUTPUT_ROOT", "RESEARCH_PLATFORM_LOCAL_CACHE", "DATA_LOCAL", "COMPANY_VALUATION_DATA_LOCAL"]:
        if key in {"RESEARCH_PLATFORM_OUTPUT_ROOT"}:
            os.environ[key] = str(output_root)
        elif key in {"RESEARCH_PLATFORM_LOCAL_CACHE", "DATA_LOCAL", "COMPANY_VALUATION_DATA_LOCAL"}:
            os.environ[key] = str(local_cache)
        else:
            os.environ[key] = str(financial_db_root)

    globals().update(config)

    if verbose:
        print(f"PROJECT_ROOT: {project_root}")
        print(f"Environment: {config['environment']}")
        print(f"FINANCIAL_DB_ROOT: {financial_db_root} | exists={financial_db_root.exists()}")
        print(f"OUTPUTROOT: {output_root}")
        print(f"LOCAL_CACHE_ROOT: {local_cache}")
        print("sys.path project entries inserted: OK")
    return config


CONFIG = setup_colab_environment(verbose=True)

try:
    from research_platform_core import read_dataset, resolve_dataset_path
    from ml_stock_lab import features, valuation
    from smart_money_engine import run_smart_money_engine
    print("Core imports: OK")
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Core imports failed after bootstrap: {exc}. Confirm PROJECT_ROOT contains src/research_platform_core and src/ml_stock_lab."
    ) from exc


In [ ]:
# Parameters
portfolio_name = None
benchmark = None
risk_profile = None
refresh_cache = None
rerun_exports_only = None


In [ ]:
# 0.0 Data Platform Bootstrap - Drive-first canonical data root
from pathlib import Path
import os, sys

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/content/machine-learning-for-trading'), Path('/content/drive/MyDrive/GitHub/machine-learning-for-trading')]
for _root in PROJECT_ROOT_CANDIDATES:
    if (_root / 'src' / 'research_platform_core').exists() and str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
        break

try:
    from src.research_platform_core import (
        resolve_data_platform_roots, dataset_status, write_data_platform_status,
        read_dataset, read_dataset_drive_first, should_refresh, resolve_dataset_path, provider_fallback_plan
    )
except Exception:
    from research_platform_core import (
        resolve_data_platform_roots, dataset_status, write_data_platform_status,
        read_dataset, read_dataset_drive_first, should_refresh, resolve_dataset_path, provider_fallback_plan
    )

DATA_PLATFORM_ROOTS = resolve_data_platform_roots(
    financial_db_root=os.environ.get('FINANCIAL_DB_ROOT') or os.environ.get('DB_BASE') or os.environ.get('DATA_PATH'),
    repo_output_root=globals().get('OUTPUTROOT', Path.cwd() / 'output'),
)
FINANCIAL_DB_ROOT = DATA_PLATFORM_ROOTS.financial_db
DB_BASE = FINANCIAL_DB_ROOT
DATA_PATH = FINANCIAL_DB_ROOT
os.environ['FINANCIAL_DB_ROOT'] = str(FINANCIAL_DB_ROOT)
os.environ['DB_BASE'] = str(DB_BASE)
os.environ['DATA_PATH'] = str(DATA_PATH)

def load_price_history_drive_first(ticker, max_age_hours=24*7, allow_stale=True):
    return read_dataset_drive_first(FINANCIAL_DB_ROOT, 'prices', ticker, max_age_hours=max_age_hours, allow_stale=allow_stale)

DATA_PLATFORM_STATUS = dataset_status(FINANCIAL_DB_ROOT, max_files=5000) if DATA_PLATFORM_ROOTS.available else {}
PROVIDER_FALLBACK_PLAN = provider_fallback_plan(FINANCIAL_DB_ROOT) if DATA_PLATFORM_ROOTS.available else None
print(f"Data platform root: {FINANCIAL_DB_ROOT}")
print(f"Data platform available: {DATA_PLATFORM_ROOTS.available} ({DATA_PLATFORM_ROOTS.source})")
if DATA_PLATFORM_STATUS.get('summary') is not None and not DATA_PLATFORM_STATUS['summary'].empty:
    display(DATA_PLATFORM_STATUS['summary'].head(12))


# Investment Research Platform Pro

## Start Here: Control Center

This notebook is designed to behave like a Colab research platform, not a linear worksheet. The first operational workspace is the **Control Center**: configure the target company, benchmark, peers, universe, data layers, model depth and reporting detail there.

Core workflow: **Input Setup -> Apply configuration -> Diagnostics -> Run research -> Results Dashboard -> Saved Outputs**.

Open the **Input Setup** tab, click **Apply configuration**, then click **Run research**. The final dashboard is intentionally placed after the research run so the notebook first explains and configures the analysis, then presents results as a navigable visual layer.


## Portfolio Notebook Standard Map

| Canonical Section | Portfolio Implementation | Output |
|---|---|---|
| 0. Setup & Config | Environment, theme, paths, logger, `MASTERREQUEST`, config blocks | Runtime, logs, config snapshot |
| 1. Data Ingestion | Project DB/cache, Database Finanziario, yfinance, synthetic fallback | Price/fundamental layers, data source summary |
| 2. Cleaning / Integration | Ticker normalization, coverage, required-column checks | Data quality and diagnostics tables |
| 3. Features | Value, quality, momentum, risk, growth, SWS-style axes | Factor and SWS scorecards |
| 4. Targets | Scenario targets and base upside | Bear/base/bull framework |
| 5. Descriptive Statistics | Portfolio/SWS snapshot, source mix, missingness | Summary and feature missingness tables |
| 6. Exploration | Equity curves, drawdown, peer comparison | Plotly dashboard charts |
| 7. Diagnostics | Enterprise diagnostics, governance checks | PASS/WARN/FAIL tables |
| 8. Models | Model factory and leaderboard | Model comparison and explainability |
| 9. ML Walk-Forward / Lab | Time-safe model comparison proxy | Leaderboard, feature importance |
| 10. Ablation / Sensitivity | Valuation sensitivity and factor exposure | Heatmaps and scenario tables |
| 11. Backtest | Top-ranked strategy vs benchmark | Gross/net curve and performance |
| 12. Interpretability | Feature importance and factor decomposition | Explainability chart/table |
| 13. Robustness Checks | Data/model/valuation/risk checks | Robustness table |
| 14. Final Dashboard / Conclusion | Interactive dashboard + static HTML export | Dashboard, CSV/HTML/Markdown artifacts |

The notebook is intentionally compact, but each dashboard tab and exported table maps to this standard.


In [ ]:
# 0. Essential setup
import sys, os, json, math, warnings, subprocess, platform, logging
from pathlib import Path
from datetime import datetime, date, timedelta

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

def install_if_missing(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except Exception as exc:
            print(f"[warning] install failed for {pip_name}: {exc}")

for import_name, pip_name in [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("plotly", "plotly"),
    ("ipywidgets", "ipywidgets"),
    ("yfinance", "yfinance"),
    ("sklearn", "scikit-learn"),
    ("cvxpy", "cvxpy"),
    ("pypfopt", "pyportfolioopt"),
    ("riskfolio", "riskfolio-lib"),
    ("cvxportfolio", "cvxportfolio"),
]:
    install_if_missing(import_name, pip_name)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    WIDGETS_AVAILABLE = True
except Exception:
    from IPython.display import display, HTML, clear_output
    WIDGETS_AVAILABLE = False

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.metrics import mean_absolute_error, r2_score
except Exception:
    StandardScaler = KMeans = PCA = TimeSeriesSplit = None

np.random.seed(42)

if IN_COLAB:
    try:
        from google.colab import output, drive
        output.enable_custom_widget_manager()
        drive.mount("/content/drive")
    except Exception:
        pass

print("Environment ready:", platform.platform())

In [ ]:
# 0.1 Professional UI theme
COLORS = {
    "primary": "#01696f",
    "primary_dark": "#004f54",
    "accent": "#da7101",
    "danger": "#c0392b",
    "warning": "#b36b00",
    "green": "#2e7d32",
    "blue": "#006494",
    "neutral": "#7a7974",
    "bg": "#f7f6f2",
    "card": "#ffffff",
    "border": "#e5e1d8",
    "text": "#1f2933",
    "muted": "#687076",
}
PLOTLY_TEMPLATE = "plotly_white"

display(HTML(f"""
<style>
body, .jp-Notebook {{ background: {COLORS['bg']} !important; color: {COLORS['text']}; }}
h1, h2, h3 {{ color: {COLORS['primary']}; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; letter-spacing: -0.02em; }}
.ir-hero {{ background: linear-gradient(135deg, {COLORS['primary']} 0%, {COLORS['primary_dark']} 100%); color: white; border-radius: 22px; padding: 28px 32px; margin: 8px 0 20px 0; box-shadow: 0 12px 32px rgba(1,105,111,0.23); }}
.ir-hero h1 {{ color: white; margin: 0; font-size: 31px; }}
.ir-hero p {{ margin: 8px 0 0 0; color: rgba(255,255,255,0.86); font-size: 15px; }}
.ir-grid-4 {{ display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 14px; margin: 14px 0 20px 0; }}
.ir-grid-3 {{ display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 14px; margin: 14px 0 20px 0; }}
.ir-card {{ background: {COLORS['card']}; border: 1px solid {COLORS['border']}; border-radius: 16px; padding: 16px 18px; box-shadow: 0 6px 20px rgba(35,35,35,0.05); }}
.ir-card-title {{ font-size: 12px; text-transform: uppercase; color: {COLORS['muted']}; font-weight: 700; letter-spacing: 0.05em; margin-bottom: 8px; }}
.ir-card-value {{ font-size: 23px; font-weight: 800; color: {COLORS['primary']}; }}
.ir-section {{ border-left: 5px solid {COLORS['accent']}; background: rgba(255,255,255,0.75); border-radius: 12px; padding: 12px 16px; margin: 22px 0 12px 0; font-weight: 800; color: {COLORS['primary']}; }}
.ir-status-pass {{ color: #1b5e20; background: #e8f5e9; border: 1px solid #c8e6c9; border-radius: 10px; padding: 5px 9px; font-weight: 700; }}
.ir-status-warn {{ color: #8a4b00; background: #fff4df; border: 1px solid #ffd699; border-radius: 10px; padding: 5px 9px; font-weight: 700; }}
.ir-status-fail {{ color: #7f1d1d; background: #fdeaea; border: 1px solid #f3b9b9; border-radius: 10px; padding: 5px 9px; font-weight: 700; }}
.ir-pill {{ display: inline-block; border-radius: 999px; padding: 4px 10px; background: #f0ede5; color: {COLORS['primary']}; font-weight: 700; margin: 2px; }}
.rendered_html table {{ border-collapse: collapse; width: 100%; background: white; border-radius: 12px; overflow: hidden; }}
.rendered_html th {{ background: #f0ede5; color: {COLORS['primary']}; font-weight: 800; }}
.rendered_html td, .rendered_html th {{ border: 1px solid #e7e2d6; padding: 8px 10px; }}
.widget-label {{ font-weight: 650 !important; }}

.ir-control-hero {{ background: linear-gradient(135deg, #013f43 0%, #01696f 58%, #0b7f73 100%); border-radius: 18px; padding: 26px 30px; margin: 16px 0 14px 0; color: white; box-shadow: 0 18px 42px rgba(1,105,111,0.25); }}
.ir-control-hero h2 {{ color: white; margin: 0; font-size: 30px; letter-spacing: 0; }}
.ir-control-hero p {{ color: rgba(255,255,255,0.88); margin: 8px 0 0 0; font-size: 14px; max-width: 960px; }}
.ir-nav-strip {{ display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 10px; margin: 10px 0 16px 0; }}
.ir-nav-item {{ background: white; border: 1px solid #e5e1d8; border-radius: 12px; padding: 12px 14px; box-shadow: 0 5px 16px rgba(31,41,51,0.05); }}
.ir-nav-num {{ display: inline-flex; align-items: center; justify-content: center; width: 24px; height: 24px; border-radius: 999px; background: #01696f; color: white; font-weight: 800; margin-right: 8px; }}
.ir-nav-title {{ font-weight: 800; color: #01696f; }}
.ir-nav-copy {{ color: #687076; font-size: 12px; margin-top: 5px; line-height: 1.35; }}
.ir-action-bar {{ background: #fff; border: 1px solid #d9d2c4; border-radius: 14px; padding: 12px; margin: 10px 0 14px 0; box-shadow: 0 8px 22px rgba(31,41,51,0.06); }}
.ir-dashboard-hero {{ background: linear-gradient(135deg, #ffffff 0%, #f7f6f2 100%); border: 1px solid #d9d2c4; border-left: 7px solid #da7101; border-radius: 16px; padding: 18px 20px; margin: 16px 0; box-shadow: 0 8px 28px rgba(31,41,51,0.07); }}
.ir-dashboard-hero h2 {{ margin: 0; color: #01696f; letter-spacing: 0; }}
.ir-dashboard-hero p {{ color: #687076; margin: 7px 0 0 0; }}
.ir-chart-panel {{ background: white; border: 1px solid #e5e1d8; border-radius: 14px; padding: 12px 14px; margin: 12px 0 18px 0; box-shadow: 0 8px 24px rgba(31,41,51,0.05); }}
.ir-chart-title {{ font-weight: 800; color: #01696f; margin-bottom: 4px; }}
.ir-chart-copy {{ color: #687076; font-size: 12px; margin-bottom: 8px; }}

.ir-setup-hero {{ background: linear-gradient(135deg, #012f34 0%, #01696f 58%, #0a8f7c 100%); color: white; border-radius: 22px; padding: 30px 34px; margin: 18px 0 18px 0; box-shadow: 0 20px 52px rgba(1,105,111,0.28); }}
.ir-setup-hero h1 {{ color: white; margin: 0; font-size: 32px; letter-spacing: 0; }}
.ir-setup-hero p {{ color: rgba(255,255,255,0.90); margin: 10px 0 0 0; max-width: 1100px; font-size: 15px; line-height: 1.5; }}
.ir-step-ribbon {{ display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 12px; margin: 12px 0 18px 0; }}
.ir-step-card {{ background: #ffffff; border: 1px solid #e5e1d8; border-radius: 14px; padding: 14px 16px; box-shadow: 0 8px 24px rgba(31,41,51,0.06); }}
.ir-step-card b {{ color: #01696f; }}
.ir-step-card span {{ display: inline-flex; width: 26px; height: 26px; align-items: center; justify-content: center; border-radius: 99px; background: #01696f; color: white; font-weight: 800; margin-right: 8px; }}
.ir-widget-frame {{ border: 1px solid #d9d2c4; border-radius: 18px; background: #fff; padding: 16px; box-shadow: 0 10px 28px rgba(31,41,51,0.07); margin: 10px 0 18px 0; }}
.ir-dashboard-empty {{ background: #ffffff; border: 1px dashed #d9d2c4; border-radius: 16px; padding: 20px; margin: 10px 0; }}
.ir-dashboard-empty h3 {{ margin-top: 0; color: #01696f; }}
.ir-big-button-note {{ background: #f7f6f2; border-left: 5px solid #da7101; border-radius: 10px; padding: 10px 12px; margin: 8px 0; }}
.ir-control-center-title {{ background: linear-gradient(135deg, #011f24 0%, #01696f 62%, #da7101 140%); color: white; border-radius: 24px; padding: 34px 38px; margin: 18px 0 16px 0; box-shadow: 0 24px 60px rgba(1,105,111,0.30); }}
.ir-control-center-title h1 {{ color: white; margin: 0; font-size: 40px; letter-spacing: 0; line-height: 1.08; }}
.ir-control-center-title p {{ color: rgba(255,255,255,0.92); max-width: 1120px; font-size: 16px; line-height: 1.5; margin: 12px 0 0 0; }}
.ir-control-mini {{ background: white; border: 2px solid #01696f; border-radius: 18px; padding: 14px 16px; margin: 12px 0; box-shadow: 0 12px 30px rgba(31,41,51,0.08); }}
.ir-control-mini strong {{ color: #01696f; }}
.ir-dashboard-shell {{ background: #ffffff; border: 1px solid #d9d2c4; border-radius: 20px; padding: 16px; margin: 14px 0 18px 0; box-shadow: 0 14px 38px rgba(31,41,51,0.08); }}
.ir-dashboard-navhint {{ display: grid; grid-template-columns: repeat(8, minmax(0, 1fr)); gap: 8px; margin-top: 12px; }}
.ir-dashboard-navhint div {{ background: #f7f6f2; border: 1px solid #e5e1d8; border-radius: 10px; padding: 9px; text-align: center; color: #01696f; font-weight: 800; font-size: 12px; }}
.ir-chart-stage {{ border: 1px solid #e5e1d8; border-radius: 16px; padding: 12px; margin: 10px 0 16px 0; background: #fff; box-shadow: inset 0 0 0 1px rgba(255,255,255,0.55); }}
@media (max-width: 900px) {{ .ir-step-ribbon, .ir-grid-4, .ir-grid-3, .ir-dashboard-navhint {{ grid-template-columns: 1fr !important; }} }}
</style>

<div class="ir-hero">
  <h1>Investment Research Platform Pro</h1>
  <p>Company analytics · data quality · peer intelligence · valuation lab · model lab · backtest · risk dashboard</p>
</div>
<div class="ir-grid-4">
  <div class="ir-card"><div class="ir-card-title">Workspace</div><div class="ir-card-value">Research OS</div></div>
  <div class="ir-card"><div class="ir-card-title">Data Quality</div><div class="ir-card-value">Audited</div></div>
  <div class="ir-card"><div class="ir-card-title">Peer Engine</div><div class="ir-card-value">Data-driven</div></div>
  <div class="ir-card"><div class="ir-card-title">Reports</div><div class="ir-card-value">Exportable</div></div>
</div>
"""))

## 1. Config kernel

OUTPUT_DIR = OUTPUT_ROOT
TABLES_OUTPUT_DIR = TABLES_DIR
FIGURES_OUTPUT_DIR = FIGURES_DIR
LOGS_OUTPUT_DIR = LOGS_DIR

log_file = LOGS_DIR / "investment_research_platform_pro.log"
logger = logging.getLogger("investment_research_platform_pro")
logger.setLevel(logging.INFO)
logger.handlers = []
_file_handler = logging.FileHandler(log_file)
_stream_handler = logging.StreamHandler()
_formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
_file_handler.setFormatter(_formatter)
_stream_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)
logger.addHandler(_stream_handler)
logger.info("Portfolio research platform paths and logger initialized.")


In [ ]:
# 1.1 Paths and project database integration

def _find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and ((candidate / ".git").exists() or (candidate / "src").exists()):
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PORTFOLIO_SRC_DIR = PROJECT_ROOT / "portfolio_analysis" / "src"
if PORTFOLIO_SRC_DIR.exists() and str(PORTFOLIO_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(PORTFOLIO_SRC_DIR))

if IN_COLAB:
    DB_BASE = Path(os.getenv("DB_BASE", "/content/drive/MyDrive/Database Finanziario"))
else:
    DB_BASE = Path(os.getenv(
        "DB_BASE",
        "/Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/Database Finanziario"
    ))

DATA_PATH = Path(os.getenv("DATA_PATH", str(DB_BASE)))
PROJECT_DATA_DIR = PROJECT_ROOT / "data"
PROJECT_CATALOG_DIR = PROJECT_DATA_DIR / "catalog"
PROJECT_MARKET_CACHE_DIRS = [
    PROJECT_DATA_DIR / "europe_stoxx_companies",
    PROJECT_DATA_DIR / "open_market_data",
    PROJECT_DATA_DIR / "alpha_factor_research",
    PROJECT_DATA_DIR / "tree_models",
]

OUTPUT_ROOT = DB_BASE / "investment_research_platform_pro"
DATA_DIR = OUTPUT_ROOT / "data"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
REPORTS_DIR = OUTPUT_ROOT / "reports"
CONFIG_DIR = OUTPUT_ROOT / "config"
SNAPSHOT_DIR = CONFIG_DIR / "snapshots"
DASHBOARD_DIR = OUTPUT_ROOT / "dashboard"
LOGS_DIR = OUTPUT_ROOT / "logs"

for path in [OUTPUT_ROOT, DATA_DIR, TABLES_DIR, FIGURES_DIR, REPORTS_DIR, CONFIG_DIR, SNAPSHOT_DIR, DASHBOARD_DIR, LOGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

try:
    from data.data_catalog import build_data_inventory, domains_to_dataframe
except Exception:
    build_data_inventory = None
    domains_to_dataframe = None

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_BASE:", DB_BASE)
print("DATA_PATH:", DATA_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


OUTPUT_DIR = OUTPUT_ROOT
TABLES_OUTPUT_DIR = TABLES_DIR
FIGURES_OUTPUT_DIR = FIGURES_DIR
LOGS_OUTPUT_DIR = LOGS_DIR

log_file = LOGS_DIR / "investment_research_platform_pro.log"
logger = logging.getLogger("investment_research_platform_pro")
logger.setLevel(logging.INFO)
logger.handlers = []
_file_handler = logging.FileHandler(log_file)
_stream_handler = logging.StreamHandler()
_formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
_file_handler.setFormatter(_formatter)
_stream_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)
logger.addHandler(_stream_handler)
logger.info("Portfolio research platform paths and logger initialized.")


In [ ]:
# 1.2 MASTERREQUEST and canonical configuration blocks

MASTERREQUEST = None

WATCHLISTS = {
    "US Mega Cap": ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "AVGO"],
    "US Semiconductors": ["NVDA", "AMD", "AVGO", "INTC", "QCOM", "TSM", "ASML", "MU", "TXN"],
    "US Quality Compounders": ["AAPL", "MSFT", "COST", "V", "MA", "ADBE", "ORCL", "UNH"],
    "US Value / Cash Flow": ["BRK-B", "JPM", "XOM", "CVX", "PFE", "CSCO", "INTC"],
    "US Healthcare": ["LLY", "UNH", "JNJ", "MRK", "ABBV", "PFE", "TMO", "ABT"],
    "Italy Banks": ["ISP.MI", "UCG.MI", "BAMI.MI", "BPE.MI", "MB.MI"],
    "Europe Quality": ["ASML.AS", "MC.PA", "NESN.SW", "NOVO-B.CO", "SAP.DE", "OR.PA"],
    "Global ETF": ["SPY", "QQQ", "ACWI", "VEA", "VWO", "TLT", "GLD"],
}

SECTOR_PEERS = {
    "technology": ["AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN", "AVGO", "ORCL", "ADBE"],
    "semiconductors": ["NVDA", "AMD", "AVGO", "INTC", "QCOM", "TSM", "ASML", "MU", "TXN"],
    "banks": ["JPM", "BAC", "MS", "GS", "C", "WFC", "ISP.MI", "UCG.MI"],
    "consumer": ["AMZN", "COST", "WMT", "HD", "MCD", "NKE"],
    "healthcare": ["LLY", "UNH", "JNJ", "MRK", "ABBV", "PFE", "TMO", "ABT"],
}

PROFILE_PRESETS = {
    "Balanced": {"objective": "Balanced risk-adjusted return", "benchmark": "SPY", "risk_tolerance": "medium", "style": ["quality", "value", "momentum"], "max_drawdown_target": -0.18, "cost_bps": 12.0, "turnover_limit": 0.30, "tax_rate": 0.26, "primary_model": "elasticnet"},
    "Quality": {"objective": "High-quality compounders", "benchmark": "QQQ", "risk_tolerance": "medium", "style": ["quality", "profitability", "low leverage"], "max_drawdown_target": -0.20, "cost_bps": 12.0, "turnover_limit": 0.25, "tax_rate": 0.26, "primary_model": "random_forest"},
    "Value": {"objective": "Margin of safety and valuation discount", "benchmark": "SPY", "risk_tolerance": "medium_high", "style": ["value", "cash flow", "mean reversion"], "max_drawdown_target": -0.25, "cost_bps": 15.0, "turnover_limit": 0.40, "tax_rate": 0.26, "primary_model": "gradient_boosting"},
    "Growth": {"objective": "Growth, momentum and innovation", "benchmark": "QQQ", "risk_tolerance": "high", "style": ["growth", "momentum", "innovation"], "max_drawdown_target": -0.30, "cost_bps": 18.0, "turnover_limit": 0.50, "tax_rate": 0.26, "primary_model": "random_forest"},
    "Minimum Volatility": {"objective": "Lower volatility and smoother drawdowns", "benchmark": "USMV", "risk_tolerance": "low", "style": ["low volatility", "quality", "defensive"], "max_drawdown_target": -0.12, "cost_bps": 8.0, "turnover_limit": 0.20, "tax_rate": 0.26, "primary_model": "ridge"},
}


MODEL_DEPTH_PRESETS = {
    "quick": {
        "description": "Fast smoke-test run for Colab and demos.",
        "selected_models": ["ridge", "random_forest"],
        "cv_folds": 3,
        "max_features": 12,
        "run_explainability": False,
        "run_backtest": True,
        "model_params": {
            "random_forest": {"n_estimators": 100, "max_depth": 4, "min_samples_leaf": 3},
            "extra_trees": {"n_estimators": 100, "max_depth": 4, "min_samples_leaf": 3},
            "gradient_boosting": {"n_estimators": 100, "max_depth": 2, "learning_rate": 0.05},
        },
    },
    "standard": {
        "description": "Balanced professional research run.",
        "selected_models": ["ridge", "random_forest", "extra_trees"],
        "cv_folds": 5,
        "max_features": 20,
        "run_explainability": True,
        "run_backtest": True,
        "model_params": {
            "random_forest": {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 2},
            "extra_trees": {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 2},
            "gradient_boosting": {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.04},
        },
    },
    "institutional": {
        "description": "Institutional-grade model comparison and governance.",
        "selected_models": ["ridge", "lasso", "random_forest", "extra_trees", "gradient_boosting"],
        "cv_folds": 5,
        "max_features": 30,
        "run_explainability": True,
        "run_backtest": True,
        "model_params": {
            "random_forest": {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 2},
            "extra_trees": {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 2},
            "gradient_boosting": {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.035},
        },
    },
    "research_deep_dive": {
        "description": "Deep research mode for fuller diagnostics and charting.",
        "selected_models": ["ridge", "lasso", "elasticnet", "random_forest", "extra_trees", "gradient_boosting"],
        "cv_folds": 7,
        "max_features": 50,
        "run_explainability": True,
        "run_backtest": True,
        "model_params": {
            "random_forest": {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 1},
            "extra_trees": {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 1},
            "gradient_boosting": {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.03},
        },
    },
}


def apply_model_depth_preset(model_depth):
    preset = MODEL_DEPTH_PRESETS.get(model_depth, MODEL_DEPTH_PRESETS["standard"])
    requested_models = list(ML_CONFIG.get("selected_models", []))
    depth_models = list(preset["selected_models"])
    selected = [m for m in requested_models if m in depth_models] or depth_models
    base_params = dict(ML_CONFIG.get("model_params", {}))
    for name, params in preset["model_params"].items():
        merged = dict(base_params.get(name, {}))
        merged.update(params)
        base_params[name] = merged
    base_params.setdefault("ridge", {"alpha": 1.0})
    base_params.setdefault("lasso", {"alpha": 0.01, "random_state": 42})
    base_params.setdefault("elasticnet", {"alpha": 0.01, "l1_ratio": 0.5, "random_state": 42})
    ML_CONFIG["model_depth"] = model_depth
    ML_CONFIG["depth_description"] = preset["description"]
    ML_CONFIG["selected_models"] = selected
    ML_CONFIG["cv_folds"] = int(preset["cv_folds"])
    ML_CONFIG["max_features"] = int(preset["max_features"])
    ML_CONFIG["run_explainability"] = bool(preset["run_explainability"])
    ML_CONFIG["run_backtest"] = bool(preset["run_backtest"])
    ML_CONFIG["model_params"] = base_params
    return ML_CONFIG


EXPERIMENT = {}
UNIVERSE = {}
USER_SELECTION = {}
PORTFOLIO_CONFIG = {}
VALUATION_CONFIG = {}
ML_CONFIG = {}
RISK_CONFIG = {}
GOVERNANCE_CONFIG = {}
REPORTING_CONFIG = {}
PROJECT_DATABASE_CONFIG = {}
PORTFOLIO_ENGINE_CONFIG = {}
TIME_SERIES_ENGINE_CONFIG = {}
DEEP_STOCK_ENGINE_CONFIG = {}
CRYPTO_ENGINE_CONFIG = {}


DETAIL_LEVELS = {
    "executive": {"label": "Executive", "description": "KPI, warnings and decision-ready summary only.", "tables": 8, "charts": "core", "formula_depth": "plain language"},
    "professional": {"label": "Professional", "description": "Core formulas, diagnostics, tables and model leaderboard.", "tables": 30, "charts": "standard", "formula_depth": "formulas + intuition"},
    "institutional": {"label": "Institutional", "description": "Governance, robustness, factor decomposition and audit trail.", "tables": 75, "charts": "full", "formula_depth": "full formulas + assumptions"},
    "research_deep_dive": {"label": "Research Deep Dive", "description": "Maximum transparency, project chapter links, advanced model labs and data provenance.", "tables": 150, "charts": "full + diagnostics", "formula_depth": "derivations + caveats"},
}

RESEARCH_SECTION_CATALOG = {
    "data_foundation": {"title": "Data Foundation", "chapter": "02_market_and_fundamental_data", "formula": "Coverage_i = valid_rows_i / expected_rows_i", "outputs": ["data_quality", "source_mix", "coverage_audit"]},
    "linear_models": {"title": "Linear Model Lab", "chapter": "07_linear_models", "formula": "r_{t+h} = alpha + beta'X_t + epsilon_{t+h}", "outputs": ["model_comparison", "model_leaderboard"]},
    "time_series": {"title": "Time Series Lab", "chapter": "19_recurrent_neural_nets", "formula": "sigma_t = std(r_{t-w:t-1}) * sqrt(252)", "outputs": ["rolling_volatility", "rolling_sharpe"]},
    "factor_research": {"title": "Factor Research", "chapter": "24_alpha_factor_library", "formula": "Score = w_v Value + w_q Quality + w_m Momentum + w_r Risk + w_g Growth", "outputs": ["factor_signals", "factor_exposure"]},
    "gradient_boosting": {"title": "Gradient Boosting", "chapter": "11_gradient_boosting_machines", "formula": "F_m(x)=F_{m-1}(x)+nu h_m(x)", "outputs": ["model_comparison", "feature_importance"]},
    "unsupervised_learning": {"title": "Unsupervised Learning", "chapter": "13_unsupervised_learning", "formula": "cluster = argmin_k ||x_i - mu_k||^2", "outputs": ["peer_similarity", "score_heatmap"]},
    "deep_learning": {"title": "Deep Learning / Autoencoders", "chapter": "20_autoencoders_for_conditional_risk_factors", "formula": "z=f_theta(X), X_hat=g_phi(z), L=||X-X_hat||^2", "outputs": ["factor_exposure", "risk_dashboard"]},
    "recurrent_nets": {"title": "Recurrent Nets / Sequence Models", "chapter": "19_recurrent_neural_nets", "formula": "h_t=f(W_x x_t + W_h h_{t-1})", "outputs": ["backtest_curve", "rolling_sharpe"]},
    "alternative_data": {"title": "Alternative Data", "chapter": "03_alternative_data", "formula": "Signal_t = normalize(news + filings + flows + sentiment)", "outputs": ["project_database_overview", "data_source_summary"]},
    "strategy_evaluation": {"title": "Backtest / Strategy Evaluation", "chapter": "05_strategy_evaluation", "formula": "NAV_t = NAV_{t-1}(1 + r_t - costs_t)", "outputs": ["backtest_performance", "backtest_curve"]},
}

DATA_INTEGRATION_LAYERS = {
    "project_db": {"label": "Project database", "path": str(PROJECT_DATA_DIR), "priority": 1},
    "database_finanziario": {"label": "Database Finanziario / Drive", "path": str(DB_BASE), "priority": 2},
    "local_cache": {"label": "Local market cache", "path": str(PROJECT_DATA_DIR), "priority": 3},
    "yfinance": {"label": "yfinance API", "path": "remote_api", "priority": 4},
    "api_registry": {"label": "Project API provider registry", "path": "02_market_and_fundamental_data/03_data_providers", "priority": 5},
    "synthetic_fallback": {"label": "Synthetic fallback", "path": "generated_with_warning", "priority": 99},
}


RISK_FACTOR_LIBRARY = {
    "market": {"ticker": "SPY", "description": "US equity market beta proxy", "family": "fama_french_style"},
    "size_smb": {"ticker": "IWM", "benchmark": "SPY", "description": "Small-minus-large proxy using Russell 2000 minus S&P 500", "family": "fama_french_style"},
    "value_hml": {"ticker": "IWD", "benchmark": "IWF", "description": "Value-minus-growth proxy", "family": "fama_french_style"},
    "profitability_rmw": {"ticker": "QUAL", "benchmark": "SPY", "description": "Quality/profitability-minus-market proxy", "family": "fama_french_5_factor"},
    "investment_cma": {"ticker": "USMV", "benchmark": "SPY", "description": "Conservative investment / low-volatility proxy", "family": "fama_french_5_factor"},
    "momentum": {"ticker": "MTUM", "benchmark": "SPY", "description": "Momentum factor proxy", "family": "carhart_alpha_factor"},
    "rates_duration": {"ticker": "TLT", "description": "Long-duration US Treasury rate sensitivity proxy", "family": "macro_rates"},
    "usd_index": {"ticker": "UUP", "description": "US dollar index ETF proxy", "family": "currency"},
    "eurusd_fx": {"ticker": "EURUSD=X", "description": "EUR/USD exchange-rate proxy", "family": "fx_rate"},
    "broad_commodities": {"ticker": "DBC", "description": "Broad commodities proxy", "family": "commodities"},
    "gold": {"ticker": "GLD", "description": "Gold commodity proxy", "family": "commodities"},
    "oil": {"ticker": "USO", "description": "Crude-oil proxy", "family": "commodities"},
}

MODEL_FAMILY_CATALOG = {
    "linear": ["ols", "ridge", "lasso", "elasticnet"],
    "machine_learning": ["random_forest", "extra_trees"],
    "gradient_boosting": ["gradient_boosting"],
    "time_series": ["rolling_volatility", "rolling_sharpe", "walk_forward"],
    "factor": ["value", "quality", "momentum", "risk", "growth"],
    "unsupervised": ["peer_clustering", "similarity_distance", "pca_factor_map"],
    "deep_learning": ["autoencoder_factor_proxy", "neural_net_placeholder"],
    "recurrent_net": ["lstm_sequence_proxy", "recurrent_risk_state"],
}

PROJECT_CHAPTER_LINKS = [
    {"chapter": "02_market_and_fundamental_data", "topic": "market/fundamental data", "path": str(PROJECT_ROOT / "02_market_and_fundamental_data")},
    {"chapter": "03_alternative_data", "topic": "alternative data", "path": str(PROJECT_ROOT / "03_alternative_data")},
    {"chapter": "04_alpha_factor_research", "topic": "alpha factor research", "path": str(PROJECT_ROOT / "04_alpha_factor_research")},
    {"chapter": "05_strategy_evaluation", "topic": "backtesting and portfolio evaluation", "path": str(PROJECT_ROOT / "05_strategy_evaluation")},
    {"chapter": "07_linear_models", "topic": "linear models", "path": str(PROJECT_ROOT / "07_linear_models")},
    {"chapter": "11_gradient_boosting_machines", "topic": "gradient boosting", "path": str(PROJECT_ROOT / "11_gradient_boosting_machines")},
    {"chapter": "13_unsupervised_learning", "topic": "unsupervised learning", "path": str(PROJECT_ROOT / "13_unsupervised_learning")},
    {"chapter": "19_recurrent_neural_nets", "topic": "recurrent neural nets", "path": str(PROJECT_ROOT / "19_recurrent_neural_nets")},
    {"chapter": "20_autoencoders_for_conditional_risk_factors", "topic": "deep learning / autoencoders", "path": str(PROJECT_ROOT / "20_autoencoders_for_conditional_risk_factors")},
    {"chapter": "24_alpha_factor_library", "topic": "factor library", "path": str(PROJECT_ROOT / "24_alpha_factor_library")},
]

def parse_tickers(text):
    if text is None:
        return []
    items = str(text).replace(";", ",").replace("\n", ",").split(",")
    out = []
    for x in items:
        t = x.strip().upper()
        if t and t not in out:
            out.append(t)
    return out

def infer_sector(main_ticker):
    t = main_ticker.upper()
    for sector, names in SECTOR_PEERS.items():
        if t in names:
            return sector
    return "technology"

def build_master_request(payload):
    if payload is None:
        raise ValueError("MASTERREQUEST is missing. Use the UI and click Apply Selection.")
    required = ["profile", "main_ticker", "watchlist", "benchmark", "start_date", "end_date"]
    missing = [k for k in required if k not in payload or payload[k] in ["", None, []]]
    if missing:
        raise ValueError(f"MASTERREQUEST is incomplete. Missing: {missing}")
    return payload

def sync_all_configs_from_user_selection():
    global MASTERREQUEST, EXPERIMENT, UNIVERSE, USER_SELECTION, PORTFOLIO_CONFIG
    global VALUATION_CONFIG, ML_CONFIG, RISK_CONFIG, GOVERNANCE_CONFIG, REPORTING_CONFIG, PROJECT_DATABASE_CONFIG, PORTFOLIO_ENGINE_CONFIG
    global TIME_SERIES_ENGINE_CONFIG, DEEP_STOCK_ENGINE_CONFIG, CRYPTO_ENGINE_CONFIG

    mr = build_master_request(MASTERREQUEST)
    preset = PROFILE_PRESETS[mr["profile"]]
    inferred_sector = infer_sector(mr["main_ticker"])

    if mr["peer_mode"] == "manual":
        peers = mr.get("peers", [])
    elif mr["peer_mode"] == "sector":
        peers = [x for x in SECTOR_PEERS[inferred_sector] if x != mr["main_ticker"]]
    elif mr["peer_mode"] == "watchlist-based":
        peers = [x for x in mr["watchlist"] if x not in [mr["main_ticker"], mr["benchmark"]]]
    elif mr["peer_mode"] in ["clustered", "factor-similar", "custom-screened"]:
        peers = [x for x in (SECTOR_PEERS[inferred_sector] + mr["watchlist"]) if x not in [mr["main_ticker"], mr["benchmark"]]]
    else:
        peers = [mr["benchmark"]]

    all_tickers = []
    for t in [mr["main_ticker"], mr["benchmark"]] + mr["watchlist"] + peers:
        if t and t not in all_tickers:
            all_tickers.append(t)

    EXPERIMENT = {
        "name": "investment_research_platform_pro",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "start_date": str(mr["start_date"]),
        "end_date": str(mr["end_date"]),
        "test_start": str(mr["test_start"]),
        "analysis_horizon_months": int(mr["horizon_months"]),
        "random_state": 42,
        "run_backtest": bool(mr["run_backtest"]),
        "run_explainability": bool(mr["run_explainability"]),
        "model_depth": mr.get("model_depth", "standard"),
        "data_source": mr.get("data_source", "auto"),
        "run_scenarios": bool(mr["run_scenarios"]),
        "run_sensitivity": bool(mr["run_sensitivity"]),
        "detail_level": mr.get("detail_level", "institutional"),
        "enabled_sections": mr.get("enabled_sections", list(RESEARCH_SECTION_CATALOG.keys())),
        "enabled_data_layers": mr.get("data_layers", ["project_db", "database_finanziario", "yfinance"]),
        "weekly_refresh_missing": bool(mr.get("weekly_refresh_missing", True)),
        "max_data_staleness_days": int(mr.get("max_data_staleness_days", 7)),
    }

    UNIVERSE = {
        "main_ticker": mr["main_ticker"],
        "benchmark": mr["benchmark"],
        "watchlist": mr["watchlist"],
        "peer_mode": mr["peer_mode"],
        "peers": peers,
        "all_tickers": all_tickers,
        "inferred_sector": inferred_sector,
        "data_source": mr.get("data_source", "auto"),
        "market": mr["market"],
        "reporting_currency": mr["currency"],
    }

    USER_SELECTION = {"profile": mr["profile"], "objective": preset["objective"], "risk_tolerance": preset["risk_tolerance"], "style": preset["style"]}
    PORTFOLIO_CONFIG = {"max_drawdown_target": preset["max_drawdown_target"], "transaction_cost_bps": float(mr["cost_bps"]), "slippage_bps": float(mr["slippage_bps"]), "tax_rate": float(mr["tax_rate"]), "turnover_limit": float(mr["turnover_limit"]), "benchmark": mr["benchmark"]}

    PORTFOLIO_ENGINE_CONFIG = {
        "enabled": bool(mr.get("portfolio_engine_enabled", False)),
        "engine": mr.get("portfolio_engine", "none"),
        "objective": mr.get("portfolio_objective", "mean_variance"),
        "risk_model": mr.get("portfolio_risk_model", "sample_cov"),
        "expected_returns": mr.get("portfolio_expected_returns", "historical_mean"),
        "risk_free_rate": float(mr.get("risk_free_rate", 0.02)),
        "constraints": {
            "min_weight": float(mr.get("portfolio_min_weight", 0.0)),
            "max_weight": float(mr.get("portfolio_max_weight", 0.30)),
            "leverage": float(mr.get("portfolio_leverage", 1.0)),
            "short": bool(mr.get("portfolio_short", False)),
            "turnover_limit": float(mr.get("turnover_limit", preset["turnover_limit"])),
            "tracking_error_target": mr.get("tracking_error_target", None),
        },
        "frontier_points": int(mr.get("frontier_points", 25)),
        "cvar_alpha": float(mr.get("cvar_alpha", 0.95)),
        "backtest_horizon": int(mr.get("backtest_horizon", 252)),
        "rebalance_freq": mr.get("rebalance_freq", "M"),
        "fallback_engine": "internal_mean_variance",
    }

    TIME_SERIES_ENGINE_CONFIG = {
        "enabled": bool(mr.get("time_series_enabled", False)),
        "model": mr.get("time_series_model", "ridge_light"),
        "task": "forecasting",
        "lookback": int(mr.get("time_series_lookback", 60)),
        "horizon": int(mr.get("time_series_horizon", 5)),
        "min_history": int(mr.get("time_series_min_history", 120)),
        "test_size": int(mr.get("time_series_test_size", 40)),
        "anomaly_z": float(mr.get("time_series_anomaly_z", 2.5)),
        "regime_window": int(mr.get("time_series_regime_window", 63)),
        "random_state": int(mr.get("random_state", 42)),
        "external_patterns": ["thuml/Time-Series-Library", "grimmlab/ForeTiS"],
        "fallback_policy": "light_sklearn_or_keras_if_available",
    }

    DEEP_STOCK_ENGINE_CONFIG = {
        "enabled": bool(mr.get("deep_stock_enabled", False)),
        "model": mr.get("deep_stock_model", "lstm_basic"),
        "lookback": int(mr.get("deep_stock_lookback", 60)),
        "horizon": int(mr.get("deep_stock_horizon", mr.get("time_series_horizon", 5))),
        "epochs": int(mr.get("deep_stock_epochs", 8)),
        "batch_size": int(mr.get("deep_stock_batch_size", 32)),
        "hidden_units": int(mr.get("deep_stock_hidden_units", 32)),
        "dropout": float(mr.get("deep_stock_dropout", 0.10)),
        "min_history": int(mr.get("deep_stock_min_history", 160)),
        "fallback_model": mr.get("deep_stock_fallback_model", "gradient_boosting_light"),
        "random_state": int(mr.get("random_state", 42)),
        "external_patterns": ["hacess/stock-price-prediction-lstm", "034adarsh/Stock-Price-Prediction-Using-LSTM"],
    }

    CRYPTO_ENGINE_CONFIG = {
        "enabled": bool(mr.get("crypto_ml_enabled", False)),
        "model": mr.get("crypto_model", "gradient_boosting_light"),
        "lookback": int(mr.get("crypto_lookback", 48)),
        "horizon": int(mr.get("crypto_horizon", 3)),
        "buy_threshold": float(mr.get("crypto_buy_threshold", 0.02)),
        "sell_threshold": float(mr.get("crypto_sell_threshold", -0.02)),
        "min_history": int(mr.get("crypto_min_history", 120)),
        "random_state": int(mr.get("random_state", 42)),
        "external_patterns": ["josericodata/CryptoPredictor", "dtoyoda10/crypto-forcast"],
    }


    VALUATION_CONFIG = {
        "methods": ["multiples", "quality_adjusted_score", "scenario_targets"],
        "multiple_metrics": ["trailing_pe", "forward_pe", "price_to_book", "price_to_sales", "ev_to_ebitda_proxy"],
        "scenario_names": ["bear", "base", "bull"],
        "sensitivity_grid": {"discount_rate": [-0.02, -0.01, 0.00, 0.01, 0.02], "terminal_growth": [-0.01, -0.005, 0.00, 0.005, 0.01], "multiple_re_rating": [-0.20, -0.10, 0.00, 0.10, 0.20]},
        "default_cost_of_equity": 0.09,
        "market_risk_premium": 0.055,
    }

    ML_CONFIG = {
        "selected_models": mr["models"],
        "requested_models": mr["models"],
        "model_depth": mr.get("model_depth", "standard"),
        "primary_model": preset["primary_model"],
        "validation": "time_series_safe",
        "random_state": 42,
        "model_params": {
            "ridge": {"alpha": 1.0},
            "lasso": {"alpha": 0.01, "random_state": 42},
            "elasticnet": {"alpha": 0.01, "l1_ratio": 0.5, "random_state": 42},
            "random_forest": {"n_estimators": 300, "max_depth": 6, "random_state": 42},
            "extra_trees": {"n_estimators": 300, "max_depth": 6, "random_state": 42},
            "gradient_boosting": {"n_estimators": 200, "max_depth": 3, "random_state": 42},
        },
        "linear_alpha": 1.0,
        "elastic_l1_ratio": 0.5,
        "model_families": mr.get("model_families", ["linear", "machine_learning", "gradient_boosting", "time_series", "factor"]),
        "available_advanced_families": MODEL_FAMILY_CATALOG,
    }
    apply_model_depth_preset(mr.get("model_depth", "standard"))

    RISK_CONFIG = {
        "market_risk": ["beta", "volatility", "downside_volatility", "drawdown", "tracking_error", "rolling_sharpe"],
        "factor_risk": ["value", "momentum", "quality", "size", "low_volatility", "leverage", "profitability"],
        "risk_factor_library": RISK_FACTOR_LIBRARY,
        "risk_factor_model": "fama_french_5_plus_macro_fx_commodities",
        "fundamental_valuation_risk": ["leverage", "interest_coverage", "margin_fragility", "valuation_dispersion", "upside_downside_asymmetry", "terminal_value_sensitivity"],
        "accounting_forensic_risk": ["accrual_ratio", "earnings_quality_flags", "working_capital_stress"],
        "governance_structural_risk": ["ownership_concentration", "controversy_flags", "esg_placeholder"],
        "data_model_risk": ["missingness", "stale_data", "failed_tickers", "unmatched_merges", "ranking_instability"],
    }

    GOVERNANCE_CONFIG = {
        "required_price_columns": ["date", "ticker", "price"],
        "required_fundamental_columns": ["ticker", "market_cap", "revenue", "profit_margin"],
        "key_columns": ["date", "ticker"],
        "no_lookahead": True,
        "fail_on_missing_masterrequest": True,
        "export_config_snapshot": True,
        "weekly_refresh_missing": bool(mr.get("weekly_refresh_missing", True)),
        "max_data_staleness_days": int(mr.get("max_data_staleness_days", 7)),
        "weekly_update_script": str(PROJECT_ROOT / "scripts" / "weekly_update.py"),
    }

    PROJECT_DATABASE_CONFIG = {
        "project_root": str(PROJECT_ROOT),
        "project_data_dir": str(PROJECT_DATA_DIR),
        "project_catalog_dir": str(PROJECT_CATALOG_DIR),
        "db_base": str(DB_BASE),
        "data_path": str(DATA_PATH),
        "market_cache_dirs": [str(x) for x in PROJECT_MARKET_CACHE_DIRS],
        "source_preference": mr.get("data_source", "auto"),
        "enabled_data_layers": mr.get("data_layers", ["project_db", "database_finanziario", "yfinance"]),
        "weekly_update_script": str(PROJECT_ROOT / "scripts" / "weekly_update.py"),
        "project_chapters": PROJECT_CHAPTER_LINKS,
        "research_sections": RESEARCH_SECTION_CATALOG,
    }

    REPORTING_CONFIG = {
        "output_root": str(OUTPUT_ROOT),
        "data_dir": str(DATA_DIR),
        "tables_dir": str(TABLES_DIR),
        "figures_dir": str(FIGURES_DIR),
        "reports_dir": str(REPORTS_DIR),
        "dashboard_dir": str(DASHBOARD_DIR),
        "snapshot_dir": str(SNAPSHOT_DIR),
        "export_csv": True,
        "export_html": True,
        "export_markdown": True,
    }
    return True


In [ ]:
# 1.3 Model factory and governance

try:
    from portfolio_engine import (
        DEFAULT_PORTFOLIO_ENGINE_CONFIG,
        build_returns_matrix_from_universe,
        merge_portfolio_engine_config,
        run_cvxportfolio_engine,
        run_pyportfolioopt_engine,
        run_riskfolio_engine,
        unified_run_portfolio_engine,
    )
except Exception as exc:
    DEFAULT_PORTFOLIO_ENGINE_CONFIG = {"enabled": False, "engine": "none", "constraints": {"min_weight": 0.0, "max_weight": 0.30, "leverage": 1.0, "short": False}}
    def build_returns_matrix_from_universe(price_df, universe, experiment=None):
        if price_df is None or price_df.empty:
            return pd.DataFrame()
        df = price_df.copy()
        price_col = "price" if "price" in df.columns else "adj_close" if "adj_close" in df.columns else "close"
        wide = df.pivot_table(index="date", columns="ticker", values=price_col, aggfunc="last").sort_index().ffill()
        return wide.pct_change(fill_method=None).dropna(how="all")
    def merge_portfolio_engine_config(engine_config=None, portfolio_config=None, risk_config=None):
        cfg = dict(DEFAULT_PORTFOLIO_ENGINE_CONFIG)
        cfg.update(engine_config or {})
        return cfg
    def unified_run_portfolio_engine(price_df, returns_df, engine_config, universe=None, experiment=None, portfolio_config=None, risk_config=None):
        return {"weights": pd.DataFrame(), "frontier": pd.DataFrame(), "backtest": pd.DataFrame(), "metrics": {}, "diagnostics": {"engine_status": "module_import_failed", "error": str(exc)}}
    run_pyportfolioopt_engine = run_riskfolio_engine = run_cvxportfolio_engine = None

try:
    from portfolio_research_screener import (
        PORTFOLIO_FILTER_SCHEMA,
        PORTFOLIO_PRESETS,
        run_portfolio_selection_layer,
        schema_frame as portfolio_selection_schema_frame,
        presets_frame as portfolio_selection_presets_frame,
    )
except Exception as exc:
    PORTFOLIO_FILTER_SCHEMA = {}
    PORTFOLIO_PRESETS = {}
    def run_portfolio_selection_layer(namespace, config=None):
        namespace["portfolio_selection_import_error"] = str(exc)
        return None
    def portfolio_selection_schema_frame():
        return pd.DataFrame()
    def portfolio_selection_presets_frame():
        return pd.DataFrame()

try:
    from ml_time_series_engine import (
        DEFAULT_CRYPTO_ENGINE_CONFIG,
        DEFAULT_DEEP_STOCK_ENGINE_CONFIG,
        DEFAULT_TIME_SERIES_ENGINE_CONFIG,
        build_anomaly_flags,
        build_price_panel_from_long,
        merge_engine_config,
        run_crypto_signal_engine,
        run_deep_stock_model,
        run_time_series_lab,
        unified_run_ml_time_series_engine,
    )
except Exception as exc:
    DEFAULT_TIME_SERIES_ENGINE_CONFIG = {"enabled": False, "model": "ridge_light"}
    DEFAULT_DEEP_STOCK_ENGINE_CONFIG = {"enabled": False, "model": "lstm_basic"}
    DEFAULT_CRYPTO_ENGINE_CONFIG = {"enabled": False, "model": "gradient_boosting_light"}
    def merge_engine_config(defaults, override=None):
        cfg = dict(defaults or {})
        cfg.update(override or {})
        return cfg
    def build_anomaly_flags(returns_df, z_threshold=2.5):
        return pd.DataFrame()
    def build_price_panel_from_long(price_df, universe=None):
        if price_df is None or price_df.empty:
            return pd.DataFrame()
        price_col = "price" if "price" in price_df.columns else "adj_close" if "adj_close" in price_df.columns else "close"
        return price_df.pivot_table(index="date", columns="ticker", values=price_col, aggfunc="last").sort_index().ffill()
    def run_time_series_lab(returns_df, config):
        return {"forecast": pd.DataFrame(), "diagnostics": pd.DataFrame([{"engine_status": "module_import_failed", "error": str(exc)}]), "anomalies": pd.DataFrame(), "regimes": pd.DataFrame()}
    def run_deep_stock_model(price_panel, config):
        return {"signals": pd.DataFrame(), "forecast": pd.DataFrame(), "diagnostics": pd.DataFrame([{"engine_status": "module_import_failed", "error": str(exc)}])}
    def run_crypto_signal_engine(crypto_price_df, config):
        return {"signals": pd.DataFrame(), "forecast": pd.DataFrame(), "diagnostics": pd.DataFrame([{"engine_status": "module_import_failed", "error": str(exc)}])}
    def unified_run_ml_time_series_engine(price_df, returns_df, universe, experiment, time_series_config=None, deep_stock_config=None, crypto_config=None, crypto_price_df=None):
        return {"time_series": run_time_series_lab(returns_df, time_series_config or {}), "deep_stock": run_deep_stock_model(build_price_panel_from_long(price_df, universe), deep_stock_config or {}), "crypto": run_crypto_signal_engine(crypto_price_df, crypto_config or {})}


def build_model(model_name, experiment=None):
    experiment = experiment or EXPERIMENT
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

    params = ML_CONFIG.get("model_params", {})
    seed = int(experiment.get("random_state", experiment.get("random_seed", 42)))
    name = str(model_name).lower().replace("-", "_")
    linear_alpha = float(ML_CONFIG.get("linear_alpha", params.get("ridge", {}).get("alpha", 1.0)))
    elastic_l1_ratio = float(ML_CONFIG.get("elastic_l1_ratio", params.get("elasticnet", {}).get("l1_ratio", 0.50)))

    if name in ["ols", "linear", "linear_regression"]:
        return Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
    if name == "ridge":
        p = {"alpha": linear_alpha}
        p.update(params.get("ridge", {}))
        return Pipeline([("scaler", StandardScaler()), ("model", Ridge(**p))])
    if name == "lasso":
        p = {"alpha": 0.01, "random_state": seed, "max_iter": 10000}
        p.update(params.get("lasso", {}))
        return Pipeline([("scaler", StandardScaler()), ("model", Lasso(**p))])
    if name in ["elasticnet", "elastic_net"]:
        p = {"alpha": 0.01, "l1_ratio": elastic_l1_ratio, "random_state": seed, "max_iter": 10000}
        p.update(params.get("elasticnet", params.get("elastic_net", {})))
        return Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(**p))])
    if name == "random_forest":
        p = {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 2, "random_state": seed, "n_jobs": -1}
        p.update(params.get("random_forest", {}))
        return RandomForestRegressor(**p)
    if name == "extra_trees":
        p = {"n_estimators": 300, "max_depth": 6, "min_samples_leaf": 2, "random_state": seed, "n_jobs": -1}
        p.update(params.get("extra_trees", {}))
        return ExtraTreesRegressor(**p)
    if name == "gradient_boosting":
        p = {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.04, "random_state": seed}
        p.update(params.get("gradient_boosting", {}))
        return GradientBoostingRegressor(**p)
    raise ValueError(f"Unsupported model: {model_name}")


def _model_feature_importance(model, feature_cols):
    estimator = model.named_steps.get("model") if hasattr(model, "named_steps") else model
    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
    elif hasattr(estimator, "coef_"):
        values = np.abs(np.ravel(estimator.coef_))
    else:
        return pd.DataFrame()
    return pd.DataFrame({"feature": feature_cols, "importance": values}).sort_values("importance", ascending=False)


def validate_required_columns(df, required):
    missing = [c for c in required if c not in df.columns]
    return {"status": "PASS" if not missing else "FAIL", "missing": missing}


def check_duplicate_keys(df, keys):
    if not all(k in df.columns for k in keys):
        return {"status": "WARN", "duplicates": None, "message": "key columns not complete"}
    dups = int(df.duplicated(keys).sum())
    return {"status": "PASS" if dups == 0 else "WARN", "duplicates": dups}


def check_configuration_consistency():
    if MASTERREQUEST is None:
        raise ValueError("MASTERREQUEST missing. Click Apply Selection first.")
    rows = [
        {"check": "MASTERREQUEST", "status": "PASS", "detail": "applied"},
        {"check": "Main ticker", "status": "PASS" if UNIVERSE.get("main_ticker") else "FAIL", "detail": UNIVERSE.get("main_ticker")},
        {"check": "Peer engine", "status": "PASS" if UNIVERSE.get("peer_mode") else "FAIL", "detail": UNIVERSE.get("peer_mode")},
        {"check": "Peer group", "status": "PASS" if len(UNIVERSE.get("peers", [])) >= 1 else "WARN", "detail": ", ".join(UNIVERSE.get("peers", []))},
        {"check": "Coverage universe", "status": "PASS" if len(UNIVERSE.get("all_tickers", [])) >= 3 else "WARN", "detail": f"{len(UNIVERSE.get('all_tickers', []))} tickers"},
        {"check": "Model depth", "status": "PASS" if ML_CONFIG.get("model_depth") in MODEL_DEPTH_PRESETS else "WARN", "detail": ML_CONFIG.get("model_depth")},
        {"check": "Models", "status": "PASS" if ML_CONFIG.get("selected_models") else "FAIL", "detail": ", ".join(ML_CONFIG.get("selected_models", []))},
        {"check": "Project database", "status": "PASS" if PROJECT_DATA_DIR.exists() else "WARN", "detail": str(PROJECT_DATA_DIR)},
        {"check": "External DB_BASE", "status": "PASS" if DB_BASE.exists() else "WARN", "detail": str(DB_BASE)},
        {"check": "Outputs", "status": "PASS" if OUTPUT_ROOT.exists() else "FAIL", "detail": str(OUTPUT_ROOT)},
    ]
    return pd.DataFrame(rows)


def run_enterprise_diagnostics():
    rows = []
    def add(check, status, severity, detail, fix_hint="", owner="research"):
        rows.append({"check": check, "status": status, "severity": severity, "detail": detail, "fix_hint": fix_hint, "owner": owner})
    add("MASTERREQUEST applied", "PASS" if MASTERREQUEST is not None else "FAIL", "critical", "MASTERREQUEST exists" if MASTERREQUEST is not None else "MASTERREQUEST is missing", "Use Control Center -> Apply configuration.", "configuration")
    add("Universe size", "PASS" if len(UNIVERSE.get("all_tickers", [])) >= 3 else "WARN", "high", f"{len(UNIVERSE.get('all_tickers', []))} tickers selected", "Add peers or watchlist tickers for meaningful cross-sectional ranking.", "data")
    add("Model depth preset", "PASS" if ML_CONFIG.get("model_depth") in MODEL_DEPTH_PRESETS else "WARN", "medium", str(ML_CONFIG.get("model_depth")), "Use quick, standard, institutional, or research_deep_dive.", "modeling")
    add("Selected models", "PASS" if ML_CONFIG.get("selected_models") else "FAIL", "critical", ", ".join(ML_CONFIG.get("selected_models", [])), "Select at least one model.", "modeling")
    add("Project data catalog", "PASS" if PROJECT_CATALOG_DIR.exists() else "WARN", "medium", str(PROJECT_CATALOG_DIR), "Run data catalog export or sync project data.", "data")
    add("Governance export", "PASS" if GOVERNANCE_CONFIG.get("export_config_snapshot", False) else "WARN", "medium", f"export_config_snapshot={GOVERNANCE_CONFIG.get('export_config_snapshot')}", "Enable config snapshots for reproducibility.", "governance")
    diag = pd.DataFrame(rows)
    diag["is_blocker"] = diag["status"].eq("FAIL") & diag["severity"].isin(["critical", "high"])
    return diag


def display_diagnostics(diag_df):
    if diag_df is None or diag_df.empty:
        display(HTML("<div class='ir-status-warn'>No diagnostics available.</div>"))
        return
    html_rows = []
    for _, r in diag_df.fillna("").iterrows():
        cls = "ir-status-pass" if r["status"] == "PASS" else "ir-status-warn" if r["status"] == "WARN" else "ir-status-fail"
        sev = r.get("severity", "")
        fix = r.get("fix_hint", "")
        owner = r.get("owner", "")
        html_rows.append(f"<tr><td>{r['check']}</td><td><span class='{cls}'>{r['status']}</span></td><td>{sev}</td><td>{r['detail']}</td><td>{fix}</td><td>{owner}</td></tr>")
    display(HTML(f"<table><thead><tr><th>Check</th><th>Status</th><th>Severity</th><th>Detail</th><th>Fix hint</th><th>Owner</th></tr></thead><tbody>{''.join(html_rows)}</tbody></table>"))


def assert_ready_to_run():
    if MASTERREQUEST is None:
        raise ValueError("Click Apply Selection before running.")
    diag = pd.concat([check_configuration_consistency(), run_enterprise_diagnostics()], ignore_index=True, sort=False)
    if (diag["status"] == "FAIL").any():
        display_diagnostics(diag)
        raise ValueError("Configuration contains FAIL diagnostics.")
    return diag


## 0.1.5 Control Center - Interactive Company Selection / Valuation Setup

This is the main operating surface of the notebook. It is intentionally central: use it to configure input data, company selection, benchmark, peers, portfolio universe, model depth, data layers, advanced model labs and output rules.

| Area | What to do | Result |
|---|---|---|
| Input Setup | Choose main ticker, benchmark, universe and peers | Draft configuration |
| Apply configuration | Validate and sync one canonical `MASTERREQUEST` | Config blocks updated |
| Diagnostics | Read configuration and data readiness checks | Warnings before running |
| Results Dashboard | Explore KPI, tables and interactive Plotly charts | Professional visual analysis |
| Saved Outputs | Find CSV, HTML, Markdown, snapshots and static dashboard | Reproducible exports |

The UI below is not auxiliary: it is the notebook entry point.


In [ ]:
# 0.1.5 Interactive Company Selection / Valuation Setup: advanced input UX

if not WIDGETS_AVAILABLE:
    raise ImportError("ipywidgets is required for the interactive Research Platform UI.")

STYLE = {"description_width": "128px"}
W220 = widgets.Layout(width="220px")
W260 = widgets.Layout(width="260px")
W320 = widgets.Layout(width="320px")
W420 = widgets.Layout(width="420px")
FULL = widgets.Layout(width="100%")
PANEL = widgets.Layout(border="1px solid #e5e1d8", padding="16px", margin="8px 0 12px 0", width="100%")

BENCHMARK_GROUPS = {
    "US Equity": ["SPY", "QQQ", "DIA", "IWM", "VTI", "RSP", "USMV"],
    "Global Equity": ["ACWI", "URTH", "EFA", "EEM", "VEA", "VWO"],
    "Europe / Italy": ["SX5E.DE", "FEZ", "EZU", "FTSEMIB.MI", "EWI"],
    "Rates / Credit": ["TLT", "IEF", "SHY", "HYG", "LQD", "BIL"],
    "Commodities / Macro": ["GLD", "SLV", "USO", "UUP", "DBC"],
}

TICKER_META = {
    "AAPL": ("Apple", "technology", "US"), "MSFT": ("Microsoft", "technology", "US"),
    "NVDA": ("NVIDIA", "semiconductors", "US"), "GOOGL": ("Alphabet", "technology", "US"),
    "AMZN": ("Amazon", "consumer", "US"), "META": ("Meta Platforms", "technology", "US"),
    "AVGO": ("Broadcom", "semiconductors", "US"), "AMD": ("Advanced Micro Devices", "semiconductors", "US"),
    "INTC": ("Intel", "semiconductors", "US"), "QCOM": ("Qualcomm", "semiconductors", "US"),
    "TSM": ("Taiwan Semiconductor", "semiconductors", "Global"), "ASML": ("ASML ADR", "semiconductors", "Global"),
    "ASML.AS": ("ASML", "semiconductors", "Europe"), "JPM": ("JPMorgan Chase", "banks", "US"),
    "BAC": ("Bank of America", "banks", "US"), "MS": ("Morgan Stanley", "banks", "US"),
    "GS": ("Goldman Sachs", "banks", "US"), "ISP.MI": ("Intesa Sanpaolo", "banks", "Italy"),
    "UCG.MI": ("UniCredit", "banks", "Italy"), "LLY": ("Eli Lilly", "healthcare", "US"),
    "UNH": ("UnitedHealth", "healthcare", "US"), "JNJ": ("Johnson & Johnson", "healthcare", "US"),
    "COST": ("Costco", "consumer", "US"), "WMT": ("Walmart", "consumer", "US"),
}

for _watch_name, _tickers in WATCHLISTS.items():
    for _ticker in _tickers:
        TICKER_META.setdefault(_ticker, (_ticker, infer_sector(_ticker), "Watchlist"))
for _sector, _tickers in SECTOR_PEERS.items():
    for _ticker in _tickers:
        TICKER_META.setdefault(_ticker, (_ticker, _sector, "Sector peer"))
for _group, _tickers in BENCHMARK_GROUPS.items():
    for _ticker in _tickers:
        TICKER_META.setdefault(_ticker, (_ticker, "benchmark", _group))

TICKER_DIRECTORY = pd.DataFrame([
    {"ticker": ticker, "name": meta[0], "sector": meta[1], "market": meta[2]}
    for ticker, meta in sorted(TICKER_META.items())
])

CONFIG_PRESET_DIR = CONFIG_DIR / "research_platform_presets"
CONFIG_PRESET_DIR.mkdir(parents=True, exist_ok=True)
CURRENT_SELECTED_PEERS = []
CURRENT_UNIVERSE = []
LAST_PREVIEW_DF = pd.DataFrame()
CONFIG_APPLIED_AT = None


def _ticker_clean(value):
    return str(value or "").strip().upper()


def _ticker_status(ticker, role="ticker"):
    ticker = _ticker_clean(ticker)
    if not ticker:
        return {"status": "FAIL", "message": f"{role} is required."}
    if not any(ch.isalnum() for ch in ticker):
        return {"status": "FAIL", "message": f"{ticker} is not a valid ticker format."}
    if ticker in TICKER_DIRECTORY["ticker"].tolist():
        row = TICKER_DIRECTORY[TICKER_DIRECTORY["ticker"] == ticker].iloc[0]
        return {"status": "PASS", "message": f"{ticker} · {row['name']} · {row['sector']} · {row['market']}"}
    if "." in ticker or "-" in ticker or len(ticker) <= 6:
        return {"status": "WARN", "message": f"{ticker} format looks plausible, but it is not in the local directory. It will be validated by the data loader."}
    return {"status": "WARN", "message": f"{ticker} is unknown locally. Check symbol spelling before running."}


def _status_html(status):
    cls = "ir-status-pass" if status["status"] == "PASS" else "ir-status-warn" if status["status"] == "WARN" else "ir-status-fail"
    return f"<span class='{cls}'>{status['status']}</span> {status['message']}"


def _candidate_table(query="", sector=None, market=None, limit=40):
    df = TICKER_DIRECTORY.copy()
    q = str(query or "").strip().lower()
    if q:
        df = df[df["ticker"].str.lower().str.contains(q) | df["name"].str.lower().str.contains(q) | df["sector"].str.lower().str.contains(q)]
    if sector and sector != "All":
        df = df[df["sector"].eq(sector)]
    if market and market != "All":
        df = df[df["market"].eq(market)]
    return df.head(limit).reset_index(drop=True)


def _universe_components(main, benchmark, watchlist, peers):
    rows = []
    def add(ticker, role):
        ticker = _ticker_clean(ticker)
        if ticker:
            meta = TICKER_META.get(ticker, (ticker, infer_sector(ticker), "custom"))
            rows.append({"ticker": ticker, "role": role, "name": meta[0], "sector": meta[1], "market": meta[2], "validation": _ticker_status(ticker)["status"]})
    add(main, "main ticker")
    add(benchmark, "benchmark")
    for ticker in watchlist:
        add(ticker, "watchlist")
    for ticker in peers:
        add(ticker, "peer")
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    role_priority = {"main ticker": 0, "benchmark": 1, "peer": 2, "watchlist": 3}
    df["role_priority"] = df["role"].map(role_priority).fillna(9)
    df = df.sort_values(["ticker", "role_priority"]).drop_duplicates("ticker", keep="first").sort_values(["role_priority", "ticker"])
    return df.drop(columns="role_priority").reset_index(drop=True)


def _saved_presets():
    files = sorted(CONFIG_PRESET_DIR.glob("*.json"), reverse=True)
    return ["<none>"] + [f.stem for f in files]

# Base controls
profile_w = widgets.Dropdown(options=list(PROFILE_PRESETS.keys()), value="Balanced", description="Profile", style=STYLE, layout=W260)
main_ticker_w = widgets.Combobox(options=sorted(TICKER_DIRECTORY["ticker"].tolist()), value="AAPL", description="Main ticker", placeholder="Search ticker", ensure_option=False, style=STYLE, layout=W260)
benchmark_group_w = widgets.Dropdown(options=list(BENCHMARK_GROUPS.keys()), value="US Equity", description="Bench group", style=STYLE, layout=W260)
benchmark_w = widgets.Combobox(options=BENCHMARK_GROUPS["US Equity"], value="SPY", description="Benchmark", ensure_option=False, style=STYLE, layout=W260)

# Search and browser
search_w = widgets.Text(value="", description="Search", placeholder="ticker, company, sector", style=STYLE, layout=W320)
sector_filter_w = widgets.Dropdown(options=["All"] + sorted(TICKER_DIRECTORY["sector"].dropna().unique().tolist()), value="All", description="Sector", style=STYLE, layout=W260)
market_filter_w = widgets.Dropdown(options=["All"] + sorted(TICKER_DIRECTORY["market"].dropna().unique().tolist()), value="All", description="Market", style=STYLE, layout=W260)
candidate_w = widgets.SelectMultiple(options=[], description="Candidates", style=STYLE, layout=widgets.Layout(width="520px", height="160px"))
set_main_btn = widgets.Button(description="Set main", icon="star", button_style="info", layout=widgets.Layout(width="126px"))
add_peers_btn = widgets.Button(description="Add peers", icon="plus", button_style="success", layout=widgets.Layout(width="126px"))
remove_peers_btn = widgets.Button(description="Remove peers", icon="minus", button_style="warning", layout=widgets.Layout(width="126px"))

# Watchlist and peer builder
watchlist_w = widgets.Dropdown(options=list(WATCHLISTS.keys()) + ["Manual"], value="US Mega Cap", description="Watchlist", style=STYLE, layout=W260)
load_watchlist_btn = widgets.Button(description="Load watchlist", icon="download", layout=widgets.Layout(width="150px"))
universe_text_w = widgets.Textarea(value=",".join(WATCHLISTS["US Mega Cap"]), description="Universe", placeholder="Comma-separated tickers", style=STYLE, layout=widgets.Layout(width="680px", height="78px"))
peer_mode_w = widgets.Dropdown(options=["clustered", "factor-similar", "custom-screened", "sector", "watchlist-based", "manual", "benchmark"], value="clustered", description="Peer mode", style=STYLE, layout=W260)
peer_sector_w = widgets.Dropdown(options=sorted(SECTOR_PEERS.keys()), value="technology", description="Peer sector", style=STYLE, layout=W260)
selected_peers_w = widgets.SelectMultiple(options=[], description="Selected peers", style=STYLE, layout=widgets.Layout(width="520px", height="138px"))
peers_text_w = widgets.Textarea(value="", description="Manual peers", placeholder="Optional comma-separated peers", style=STYLE, layout=widgets.Layout(width="680px", height="64px"))

# Dates/models/advanced
start_date_w = widgets.DatePicker(description="Start date", value=date(2020, 1, 1), style=STYLE, layout=W260)
end_date_w = widgets.DatePicker(description="End date", value=date.today(), style=STYLE, layout=W260)
test_start_w = widgets.DatePicker(description="Test start", value=date(2023, 1, 1), style=STYLE, layout=W260)
horizon_w = widgets.IntSlider(value=12, min=3, max=36, step=3, description="Horizon", style=STYLE, layout=W420)
models_w = widgets.SelectMultiple(options=["ols", "ridge", "lasso", "elasticnet", "random_forest", "extra_trees", "gradient_boosting"], value=("ridge", "random_forest", "gradient_boosting"), description="Models", style=STYLE, layout=widgets.Layout(width="420px", height="140px"))
model_depth_w = widgets.Dropdown(options=["quick", "standard", "institutional", "research_deep_dive"], value="institutional", description="Depth", style=STYLE, layout=W260)
data_source_w = widgets.Dropdown(options=["auto", "database_finanziario", "local_cache", "yfinance", "synthetic_fallback"], value="auto", description="Data source", style=STYLE, layout=W260)
market_w = widgets.Dropdown(options=["US", "Italy", "Europe", "Global"], value="US", description="Market", style=STYLE, layout=W260)
currency_w = widgets.Dropdown(options=["USD", "EUR", "GBP", "CHF"], value="USD", description="Currency", style=STYLE, layout=W260)
cost_w = widgets.FloatSlider(value=12.0, min=0, max=100, step=1, description="Cost bps", style=STYLE, layout=W420)
slippage_w = widgets.FloatSlider(value=5.0, min=0, max=50, step=1, description="Slippage", style=STYLE, layout=W420)
tax_w = widgets.FloatSlider(value=0.26, min=0, max=0.50, step=0.01, description="Tax", style=STYLE, layout=W420)
turnover_w = widgets.FloatSlider(value=0.30, min=0.05, max=1.0, step=0.05, description="Turnover", style=STYLE, layout=W420)
run_backtest_w = widgets.Checkbox(value=True, description="Backtest Lab", indent=False)
run_explainability_w = widgets.Checkbox(value=True, description="Explainability", indent=False)
scenario_w = widgets.Checkbox(value=True, description="Scenario analysis", indent=False)
sensitivity_w = widgets.Checkbox(value=True, description="Sensitivity analysis", indent=False)
portfolio_engine_w = widgets.Dropdown(options=[("None", "none"), ("PyPortfolioOpt", "pyportfolioopt"), ("Riskfolio-Lib", "riskfolio"), ("cvxportfolio", "cvxportfolio")], value="none", description="Portfolio Engine", style=STYLE, layout=W320)
portfolio_objective_w = widgets.Dropdown(options=["mean_variance", "min_volatility", "cvar", "hrp", "risk_parity", "black_litterman", "multi_period"], value="mean_variance", description="Objective", style=STYLE, layout=W320)
portfolio_risk_model_w = widgets.Dropdown(options=["sample_cov", "shrunk_cov", "factor_model"], value="sample_cov", description="Risk model", style=STYLE, layout=W260)
portfolio_max_weight_w = widgets.FloatSlider(value=0.30, min=0.05, max=1.0, step=0.05, description="Max weight", style=STYLE, layout=W420)
portfolio_min_weight_w = widgets.FloatSlider(value=0.00, min=0.0, max=0.25, step=0.01, description="Min weight", style=STYLE, layout=W420)
portfolio_leverage_w = widgets.FloatSlider(value=1.0, min=1.0, max=2.0, step=0.05, description="Leverage", style=STYLE, layout=W420)
portfolio_short_w = widgets.Checkbox(value=False, description="Allow short", indent=False)
cvar_alpha_w = widgets.FloatSlider(value=0.95, min=0.80, max=0.99, step=0.01, description="CVaR alpha", style=STYLE, layout=W420)
rebalance_freq_w = widgets.Dropdown(options=["W", "M", "Q"], value="M", description="Rebalance", style=STYLE, layout=W260)
portfolio_engine_out = widgets.Output()

time_series_enabled_w = widgets.Checkbox(value=False, description="Enable Time Series Lab", indent=False)
time_series_model_w = widgets.Dropdown(options=["ridge_light", "gradient_boosting_light", "lstm_basic", "tslib_transformer_pattern", "foretis_benchmark_pattern"], value="ridge_light", description="TS model", style=STYLE, layout=W320)
time_series_lookback_w = widgets.IntSlider(value=60, min=20, max=252, step=5, description="TS lookback", style=STYLE, layout=W420)
time_series_horizon_w = widgets.IntSlider(value=5, min=1, max=30, step=1, description="TS horizon", style=STYLE, layout=W420)
deep_stock_enabled_w = widgets.Checkbox(value=False, description="Enable Deep Stock", indent=False)
deep_stock_model_w = widgets.Dropdown(options=["lstm_basic", "transformer_light", "gradient_boosting_light"], value="lstm_basic", description="Deep model", style=STYLE, layout=W320)
deep_stock_lookback_w = widgets.IntSlider(value=60, min=20, max=252, step=5, description="Deep lookback", style=STYLE, layout=W420)
deep_stock_epochs_w = widgets.IntSlider(value=8, min=1, max=50, step=1, description="Epochs", style=STYLE, layout=W420)
crypto_ml_enabled_w = widgets.Checkbox(value=False, description="Enable Crypto ML", indent=False)
crypto_model_w = widgets.Dropdown(options=["gradient_boosting_light", "lstm_basic"], value="gradient_boosting_light", description="Crypto model", style=STYLE, layout=W320)
crypto_lookback_w = widgets.IntSlider(value=48, min=12, max=240, step=4, description="Crypto lookback", style=STYLE, layout=W420)
crypto_horizon_w = widgets.IntSlider(value=3, min=1, max=30, step=1, description="Crypto horizon", style=STYLE, layout=W420)
ml_time_series_out = widgets.Output()

detail_level_w = widgets.Dropdown(options=list(DETAIL_LEVELS.keys()), value="institutional", description="Detail", style=STYLE, layout=W260)
section_focus_w = widgets.SelectMultiple(options=list(RESEARCH_SECTION_CATALOG.keys()), value=tuple(RESEARCH_SECTION_CATALOG.keys()), description="Sections", style=STYLE, layout=widgets.Layout(width="520px", height="170px"))
data_layers_w = widgets.SelectMultiple(options=list(DATA_INTEGRATION_LAYERS.keys()), value=("project_db", "database_finanziario", "local_cache", "yfinance", "api_registry"), description="Data layers", style=STYLE, layout=widgets.Layout(width="520px", height="130px"))
model_families_w = widgets.SelectMultiple(options=list(MODEL_FAMILY_CATALOG.keys()), value=("linear", "machine_learning", "gradient_boosting", "time_series", "factor", "unsupervised", "deep_learning", "recurrent_net"), description="Model labs", style=STYLE, layout=widgets.Layout(width="520px", height="150px"))
weekly_refresh_w = widgets.Checkbox(value=True, description="Weekly refresh missing/stale data", indent=False)
max_staleness_w = widgets.IntSlider(value=7, min=1, max=30, step=1, description="Max stale days", style=STYLE, layout=W420)

# Presets/actions
preset_name_w = widgets.Text(value="balanced_quality_aapl", description="Preset name", style=STYLE, layout=W320)
saved_preset_w = widgets.Dropdown(options=_saved_presets(), value="<none>", description="Load preset", style=STYLE, layout=W320)
save_preset_btn = widgets.Button(description="Save config", icon="save", button_style="info", layout=widgets.Layout(width="140px"))
load_preset_btn = widgets.Button(description="Load config", icon="folder-open", layout=widgets.Layout(width="140px"))
apply_btn = widgets.Button(description="Apply configuration", button_style="success", icon="check", layout=widgets.Layout(width="190px", height="42px"))
run_btn = widgets.Button(description="Run research", button_style="primary", icon="play", layout=widgets.Layout(width="170px", height="42px"))

# Outputs
validation_out = widgets.Output()
preview_out = widgets.Output()
watchlist_out = widgets.Output()
peer_out = widgets.Output()
status_out = widgets.Output()
diagnostics_out = widgets.Output()
results_out = widgets.Output()
config_summary_out = widgets.Output()
saved_outputs_out = widgets.Output()



def _refresh_config_summary(change=None):
    with config_summary_out:
        clear_output(wait=True)
        watchlist = parse_tickers(universe_text_w.value)
        peers = _auto_peers()
        universe_count = len(_universe_components(main_ticker_w.value, benchmark_w.value, watchlist, peers))
        main_status = _ticker_status(main_ticker_w.value, "Main ticker")
        bench_status = _ticker_status(benchmark_w.value, "Benchmark")
        status_label = "Applied" if CONFIG_APPLIED_AT else "Draft"
        status_class = "ir-status-pass" if CONFIG_APPLIED_AT else "ir-status-warn"
        applied_copy = f"Last applied: {CONFIG_APPLIED_AT}" if CONFIG_APPLIED_AT else "Apply configuration before running research."
        db_status = "ready" if DB_BASE.exists() else "missing"
        project_status = "ready" if PROJECT_DATA_DIR.exists() else "missing"
        depth_copy = MODEL_DEPTH_PRESETS.get(model_depth_w.value, {}).get("description", "Custom model depth")
        display(HTML(f"""
        <div class="ir-grid-4">
          <div class="ir-card"><div class="ir-card-title">Config status</div><div class="ir-card-value">{status_label}</div><div class="{status_class}" style="margin-top:8px">{applied_copy}</div></div>
          <div class="ir-card"><div class="ir-card-title">Main ticker</div><div class="ir-card-value">{_ticker_clean(main_ticker_w.value)}</div><div style="color:{COLORS['muted']};font-size:12px">{main_status['message']}</div></div>
          <div class="ir-card"><div class="ir-card-title">Benchmark</div><div class="ir-card-value">{_ticker_clean(benchmark_w.value)}</div><div style="color:{COLORS['muted']};font-size:12px">{bench_status['message']}</div></div>
          <div class="ir-card"><div class="ir-card-title">Universe</div><div class="ir-card-value">{universe_count}</div><div style="color:{COLORS['muted']};font-size:12px">{len(peers)} peers · {profile_w.value} profile</div></div>
          <div class="ir-card"><div class="ir-card-title">Model depth</div><div class="ir-card-value">{model_depth_w.value}</div><div style="color:{COLORS['muted']};font-size:12px">{depth_copy}</div></div>
          <div class="ir-card"><div class="ir-card-title">Data source</div><div class="ir-card-value">{data_source_w.value}</div><div style="color:{COLORS['muted']};font-size:12px">Project DB {project_status} · DB_BASE {db_status}</div></div>
          <div class="ir-card"><div class="ir-card-title">Selected models</div><div class="ir-card-value">{len(models_w.value)}</div><div style="color:{COLORS['muted']};font-size:12px">{', '.join(models_w.value)[:64]}</div></div>
          <div class="ir-card"><div class="ir-card-title">Output root</div><div class="ir-card-value">Drive</div><div style="color:{COLORS['muted']};font-size:12px">{OUTPUT_ROOT}</div></div>
        </div>
        """))

def _refresh_candidates(change=None):
    df = _candidate_table(search_w.value, sector_filter_w.value, market_filter_w.value)
    candidate_w.options = [f"{r.ticker} · {r.name} · {r.sector} · {r.market}" for r in df.itertuples()]
    with watchlist_out:
        clear_output(wait=True)
        display(HTML("<div class='ir-card-title'>Ticker browser</div>"))
        display(df)


def _selected_candidate_tickers():
    out = []
    for label in candidate_w.value:
        ticker = label.split(" · ", 1)[0].strip().upper()
        if ticker and ticker not in out:
            out.append(ticker)
    return out


def _set_main(_=None):
    selected = _selected_candidate_tickers()
    if selected:
        main_ticker_w.value = selected[0]


def _sync_selected_peers_widget():
    selected_peers_w.options = CURRENT_SELECTED_PEERS
    selected_peers_w.value = tuple(CURRENT_SELECTED_PEERS[: min(len(CURRENT_SELECTED_PEERS), 6)])
    peers_text_w.value = ",".join(CURRENT_SELECTED_PEERS)


def _add_peers(_=None):
    global CURRENT_SELECTED_PEERS
    for ticker in _selected_candidate_tickers():
        if ticker not in CURRENT_SELECTED_PEERS and ticker not in [main_ticker_w.value.strip().upper(), benchmark_w.value.strip().upper()]:
            CURRENT_SELECTED_PEERS.append(ticker)
    peer_mode_w.value = "manual"
    _sync_selected_peers_widget()
    _refresh_preview()


def _remove_peers(_=None):
    global CURRENT_SELECTED_PEERS
    remove = set(selected_peers_w.value)
    CURRENT_SELECTED_PEERS = [ticker for ticker in CURRENT_SELECTED_PEERS if ticker not in remove]
    _sync_selected_peers_widget()
    _refresh_preview()


def _load_watchlist(_=None):
    if watchlist_w.value != "Manual":
        universe_text_w.value = ",".join(WATCHLISTS[watchlist_w.value])
    _refresh_preview()


def _benchmark_group_changed(change=None):
    benchmark_w.options = BENCHMARK_GROUPS[benchmark_group_w.value]
    if benchmark_w.value not in benchmark_w.options:
        benchmark_w.value = BENCHMARK_GROUPS[benchmark_group_w.value][0]


def _profile_changed(change=None):
    preset = PROFILE_PRESETS[profile_w.value]
    benchmark = preset.get("benchmark")
    if benchmark:
        for group, tickers in BENCHMARK_GROUPS.items():
            if benchmark in tickers:
                benchmark_group_w.value = group
                benchmark_w.options = tickers
                benchmark_w.value = benchmark
                break
    cost_w.value = float(preset.get("cost_bps", cost_w.value))
    tax_w.value = float(preset.get("tax_rate", tax_w.value))
    turnover_w.value = float(preset.get("turnover_limit", turnover_w.value))


def _auto_peers():
    main = main_ticker_w.value.strip().upper()
    watchlist = parse_tickers(universe_text_w.value)
    if peer_mode_w.value == "manual":
        return parse_tickers(peers_text_w.value)
    if peer_mode_w.value == "sector":
        return [x for x in SECTOR_PEERS.get(peer_sector_w.value, []) if x != main]
    if peer_mode_w.value == "watchlist-based":
        return [x for x in watchlist if x not in [main, benchmark_w.value]]
    if peer_mode_w.value in ["clustered", "factor-similar", "custom-screened"]:
        sector = infer_sector(main)
        return [x for x in (SECTOR_PEERS.get(sector, []) + watchlist) if x not in [main, benchmark_w.value]][:12]
    return [benchmark_w.value]


def _refresh_validation(change=None):
    with validation_out:
        clear_output(wait=True)
        main_status = _ticker_status(main_ticker_w.value, "Main ticker")
        bench_status = _ticker_status(benchmark_w.value, "Benchmark")
        watchlist = parse_tickers(universe_text_w.value)
        warnings_list = []
        if _ticker_clean(main_ticker_w.value) == _ticker_clean(benchmark_w.value):
            warnings_list.append("Main ticker and benchmark are identical.")
        if _ticker_clean(main_ticker_w.value) not in watchlist:
            warnings_list.append("Main ticker is not in the watchlist; it will still be included in coverage.")
        display(HTML("<div class='ir-card-title'>Live validation</div>" + _status_html(main_status) + "<br>" + _status_html(bench_status)))
        if warnings_list:
            display(HTML("".join([f"<div class='ir-status-warn'>{w}</div>" for w in warnings_list])))


def _refresh_peer_panel(change=None):
    peers = _auto_peers()
    with peer_out:
        clear_output(wait=True)
        display(HTML(f"<div class='ir-card-title'>Peer builder · {peer_mode_w.value}</div>"))
        if peers:
            df = _universe_components(main_ticker_w.value, benchmark_w.value, [], peers)
            display(df[df["role"].eq("peer")])
        else:
            display(HTML("<div class='ir-status-warn'>No peers selected yet.</div>"))


def _refresh_preview(change=None):
    global LAST_PREVIEW_DF
    watchlist = parse_tickers(universe_text_w.value)
    peers = _auto_peers()
    LAST_PREVIEW_DF = _universe_components(main_ticker_w.value, benchmark_w.value, watchlist, peers)
    with preview_out:
        clear_output(wait=True)
        role_counts = LAST_PREVIEW_DF["role"].value_counts().to_dict() if not LAST_PREVIEW_DF.empty else {}
        display(HTML(f"""
        <div class="ir-grid-4">
          <div class="ir-card"><div class="ir-card-title">Main ticker</div><div class="ir-card-value">{_ticker_clean(main_ticker_w.value)}</div></div>
          <div class="ir-card"><div class="ir-card-title">Benchmark</div><div class="ir-card-value">{_ticker_clean(benchmark_w.value)}</div></div>
          <div class="ir-card"><div class="ir-card-title">Peers</div><div class="ir-card-value">{len(peers)}</div></div>
          <div class="ir-card"><div class="ir-card-title">Coverage universe</div><div class="ir-card-value">{len(LAST_PREVIEW_DF)}</div></div>
        </div>
        """))
        display(HTML("<div class='ir-card-title'>Universe preview</div>"))
        display(LAST_PREVIEW_DF)
    _refresh_validation()
    _refresh_peer_panel()
    _refresh_config_summary()


def _make_request():
    peers = _auto_peers()
    return {
        "profile": profile_w.value,
        "main_ticker": _ticker_clean(main_ticker_w.value),
        "watchlist": parse_tickers(universe_text_w.value),
        "benchmark": _ticker_clean(benchmark_w.value),
        "benchmark_group": benchmark_group_w.value,
        "peers": peers,
        "peer_mode": peer_mode_w.value,
        "peer_sector": peer_sector_w.value,
        "start_date": start_date_w.value,
        "end_date": end_date_w.value,
        "test_start": test_start_w.value,
        "market": market_w.value,
        "currency": currency_w.value,
        "horizon_months": horizon_w.value,
        "models": list(models_w.value),
        "model_depth": model_depth_w.value,
        "data_source": data_source_w.value,
        "cost_bps": cost_w.value,
        "slippage_bps": slippage_w.value,
        "tax_rate": tax_w.value,
        "turnover_limit": turnover_w.value,
        "run_backtest": run_backtest_w.value,
        "run_explainability": run_explainability_w.value,
        "run_scenarios": scenario_w.value,
        "run_sensitivity": sensitivity_w.value,
        "portfolio_engine_enabled": portfolio_engine_w.value != "none",
        "portfolio_engine": portfolio_engine_w.value,
        "portfolio_objective": portfolio_objective_w.value,
        "portfolio_risk_model": portfolio_risk_model_w.value,
        "portfolio_max_weight": portfolio_max_weight_w.value,
        "portfolio_min_weight": portfolio_min_weight_w.value,
        "portfolio_leverage": portfolio_leverage_w.value,
        "portfolio_short": portfolio_short_w.value,
        "cvar_alpha": cvar_alpha_w.value,
        "rebalance_freq": rebalance_freq_w.value,
        "time_series_enabled": time_series_enabled_w.value,
        "time_series_model": time_series_model_w.value,
        "time_series_lookback": time_series_lookback_w.value,
        "time_series_horizon": time_series_horizon_w.value,
        "deep_stock_enabled": deep_stock_enabled_w.value,
        "deep_stock_model": deep_stock_model_w.value,
        "deep_stock_lookback": deep_stock_lookback_w.value,
        "deep_stock_epochs": deep_stock_epochs_w.value,
        "crypto_ml_enabled": crypto_ml_enabled_w.value,
        "crypto_model": crypto_model_w.value,
        "crypto_lookback": crypto_lookback_w.value,
        "crypto_horizon": crypto_horizon_w.value,
        "detail_level": detail_level_w.value,
        "enabled_sections": list(section_focus_w.value),
        "data_layers": list(data_layers_w.value),
        "model_families": list(model_families_w.value),
        "weekly_refresh_missing": weekly_refresh_w.value,
        "max_data_staleness_days": max_staleness_w.value,
    }


def _apply_payload(payload):
    global MASTERREQUEST, CONFIG_APPLIED_AT
    MASTERREQUEST = payload
    sync_all_configs_from_user_selection()
    CONFIG_APPLIED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    CONFIG_DIR.joinpath(f"masterrequest_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json").write_text(json.dumps(MASTERREQUEST, indent=2, default=str), encoding="utf-8")


def _apply(_=None):
    with status_out:
        clear_output(wait=True)
        try:
            payload = _make_request()
            main_status = _ticker_status(payload["main_ticker"], "Main ticker")
            if main_status["status"] == "FAIL":
                raise ValueError(main_status["message"])
            _apply_payload(payload)
            display(HTML(f"""
            <div class='ir-card'>
              <div class='ir-card-title'>Configuration applied</div>
              <div><b>{UNIVERSE['main_ticker']}</b> vs <b>{UNIVERSE['benchmark']}</b> · {UNIVERSE['peer_mode']} peers · {len(UNIVERSE['all_tickers'])} tickers · {payload['model_depth']} depth · {payload['data_source']} data</div>
            </div>
            """))
            with diagnostics_out:
                clear_output(wait=True)
                display(HTML("<div class='ir-section'>Configuration diagnostics</div>"))
                display_diagnostics(check_configuration_consistency())
            _refresh_preview()
            _refresh_config_summary()
        except Exception as exc:
            display(HTML(f"<div class='ir-status-fail'>Configuration error: {exc}</div>"))


def _save_preset(_=None):
    with status_out:
        try:
            name = preset_name_w.value.strip().replace(" ", "_") or f"preset_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            path = CONFIG_PRESET_DIR / f"{name}.json"
            path.write_text(json.dumps(_make_request(), indent=2, default=str), encoding="utf-8")
            saved_preset_w.options = _saved_presets()
            saved_preset_w.value = name
            display(HTML(f"<div class='ir-status-pass'>Saved preset: {path.name}</div>"))
        except Exception as exc:
            display(HTML(f"<div class='ir-status-fail'>Save failed: {exc}</div>"))


def _set_widget_date(widget, value):
    if value is None:
        return
    widget.value = pd.Timestamp(value).date()


def _load_preset(_=None):
    if saved_preset_w.value == "<none>":
        return
    with status_out:
        try:
            payload = json.loads((CONFIG_PRESET_DIR / f"{saved_preset_w.value}.json").read_text(encoding="utf-8"))
            profile_w.value = payload.get("profile", profile_w.value)
            main_ticker_w.value = payload.get("main_ticker", main_ticker_w.value)
            if payload.get("benchmark_group") in BENCHMARK_GROUPS:
                benchmark_group_w.value = payload["benchmark_group"]
            benchmark_w.value = payload.get("benchmark", benchmark_w.value)
            watchlist_w.value = "Manual"
            universe_text_w.value = ",".join(payload.get("watchlist", []))
            peer_mode_w.value = payload.get("peer_mode", peer_mode_w.value)
            peer_sector_w.value = payload.get("peer_sector", peer_sector_w.value)
            peers_text_w.value = ",".join(payload.get("peers", []))
            _set_widget_date(start_date_w, payload.get("start_date"))
            _set_widget_date(end_date_w, payload.get("end_date"))
            _set_widget_date(test_start_w, payload.get("test_start"))
            models_w.value = tuple([m for m in payload.get("models", models_w.value) if m in models_w.options])
            model_depth_w.value = payload.get("model_depth", model_depth_w.value)
            data_source_w.value = payload.get("data_source", data_source_w.value)
            cost_w.value = float(payload.get("cost_bps", cost_w.value))
            slippage_w.value = float(payload.get("slippage_bps", slippage_w.value))
            tax_w.value = float(payload.get("tax_rate", tax_w.value))
            turnover_w.value = float(payload.get("turnover_limit", turnover_w.value))
            run_backtest_w.value = bool(payload.get("run_backtest", run_backtest_w.value))
            run_explainability_w.value = bool(payload.get("run_explainability", run_explainability_w.value))
            scenario_w.value = bool(payload.get("run_scenarios", scenario_w.value))
            sensitivity_w.value = bool(payload.get("run_sensitivity", sensitivity_w.value))
            portfolio_engine_w.value = payload.get("portfolio_engine", portfolio_engine_w.value)
            time_series_enabled_w.value = payload.get("time_series_enabled", time_series_enabled_w.value)
            time_series_model_w.value = payload.get("time_series_model", time_series_model_w.value)
            time_series_lookback_w.value = payload.get("time_series_lookback", time_series_lookback_w.value)
            time_series_horizon_w.value = payload.get("time_series_horizon", time_series_horizon_w.value)
            deep_stock_enabled_w.value = payload.get("deep_stock_enabled", deep_stock_enabled_w.value)
            deep_stock_model_w.value = payload.get("deep_stock_model", deep_stock_model_w.value)
            deep_stock_lookback_w.value = payload.get("deep_stock_lookback", deep_stock_lookback_w.value)
            deep_stock_epochs_w.value = payload.get("deep_stock_epochs", deep_stock_epochs_w.value)
            crypto_ml_enabled_w.value = payload.get("crypto_ml_enabled", crypto_ml_enabled_w.value)
            crypto_model_w.value = payload.get("crypto_model", crypto_model_w.value)
            crypto_lookback_w.value = payload.get("crypto_lookback", crypto_lookback_w.value)
            crypto_horizon_w.value = payload.get("crypto_horizon", crypto_horizon_w.value)
            portfolio_objective_w.value = payload.get("portfolio_objective", portfolio_objective_w.value)
            portfolio_risk_model_w.value = payload.get("portfolio_risk_model", portfolio_risk_model_w.value)
            portfolio_max_weight_w.value = float(payload.get("portfolio_max_weight", portfolio_max_weight_w.value))
            portfolio_min_weight_w.value = float(payload.get("portfolio_min_weight", portfolio_min_weight_w.value))
            portfolio_leverage_w.value = float(payload.get("portfolio_leverage", portfolio_leverage_w.value))
            portfolio_short_w.value = bool(payload.get("portfolio_short", portfolio_short_w.value))
            cvar_alpha_w.value = float(payload.get("cvar_alpha", cvar_alpha_w.value))
            rebalance_freq_w.value = payload.get("rebalance_freq", rebalance_freq_w.value)
            detail_level_w.value = payload.get("detail_level", detail_level_w.value)
            section_focus_w.value = tuple([x for x in payload.get("enabled_sections", section_focus_w.value) if x in section_focus_w.options]) or section_focus_w.value
            data_layers_w.value = tuple([x for x in payload.get("data_layers", data_layers_w.value) if x in data_layers_w.options]) or data_layers_w.value
            model_families_w.value = tuple([x for x in payload.get("model_families", model_families_w.value) if x in model_families_w.options]) or model_families_w.value
            weekly_refresh_w.value = bool(payload.get("weekly_refresh_missing", weekly_refresh_w.value))
            max_staleness_w.value = int(payload.get("max_data_staleness_days", max_staleness_w.value))
            _refresh_preview()
            _refresh_config_summary()
            display(HTML(f"<div class='ir-status-pass'>Loaded preset: {saved_preset_w.value}</div>"))
        except Exception as exc:
            display(HTML(f"<div class='ir-status-fail'>Load failed: {exc}</div>"))

# Wire interactions
for widget in [search_w, sector_filter_w, market_filter_w]:
    widget.observe(_refresh_candidates, names="value")
for widget in [main_ticker_w, benchmark_w, watchlist_w, universe_text_w, peer_mode_w, peer_sector_w, peers_text_w, start_date_w, end_date_w, test_start_w, data_source_w, models_w, detail_level_w, section_focus_w, data_layers_w, model_families_w, weekly_refresh_w, max_staleness_w, portfolio_engine_w, portfolio_objective_w, portfolio_risk_model_w, portfolio_max_weight_w, portfolio_min_weight_w, portfolio_leverage_w, portfolio_short_w, cvar_alpha_w, rebalance_freq_w, time_series_enabled_w, time_series_model_w, time_series_lookback_w, time_series_horizon_w, deep_stock_enabled_w, deep_stock_model_w, deep_stock_lookback_w, deep_stock_epochs_w, crypto_ml_enabled_w, crypto_model_w, crypto_lookback_w, crypto_horizon_w]:
    widget.observe(_refresh_preview, names="value")
try:
    model_depth_w.observe(_model_depth_changed, names="value")
except NameError:
    def _model_depth_changed(change=None):
        try:
            _refresh_preview()
        except Exception:
            return
    model_depth_w.observe(_model_depth_changed, names="value")
benchmark_group_w.observe(_benchmark_group_changed, names="value")
profile_w.observe(_profile_changed, names="value")
set_main_btn.on_click(_set_main)
add_peers_btn.on_click(_add_peers)
remove_peers_btn.on_click(_remove_peers)
load_watchlist_btn.on_click(_load_watchlist)
apply_btn.on_click(_apply)
save_preset_btn.on_click(_save_preset)
load_preset_btn.on_click(_load_preset)

# Layout: progressive disclosure with professional panels
base_panel = widgets.VBox([
    widgets.HTML("<h3>1 · Core selection</h3><p>Start with the company, benchmark, and professional preset. Everything else updates from these choices.</p>"),
    widgets.HBox([profile_w, main_ticker_w, benchmark_group_w, benchmark_w]),
    validation_out,
], layout=PANEL)

ticker_browser_panel = widgets.VBox([
    widgets.HTML("<h3>2 · Assisted ticker search</h3><p>Search by ticker, company, sector or market. Select one or more candidates, then set the main ticker or add peers.</p>"),
    widgets.HBox([search_w, sector_filter_w, market_filter_w]),
    widgets.HBox([candidate_w, widgets.VBox([set_main_btn, add_peers_btn, remove_peers_btn])]),
    watchlist_out,
], layout=PANEL)

universe_panel = widgets.VBox([
    widgets.HTML("<h3>3 · Universe and peer builder</h3><p>Browse watchlists, build a peer set visually, and preview coverage before running.</p>"),
    widgets.HBox([watchlist_w, load_watchlist_btn, peer_mode_w, peer_sector_w]),
    universe_text_w,
    peers_text_w,
    widgets.HBox([selected_peers_w, peer_out]),
    preview_out,
], layout=PANEL)

model_panel = widgets.VBox([
    widgets.HTML("<h3>Model lab setup</h3><p>Choose model family and research depth. The model factory remains canonical.</p>"),
    widgets.HBox([model_depth_w, data_source_w]),
    models_w,
    widgets.HBox([horizon_w, cost_w]),
], layout=PANEL)


section_config_panel = widgets.VBox([
    widgets.HTML("<h3>4 · Section depth and project interconnections</h3><p>Choose how much detail each research section should expose. The notebook links portfolio analysis to the project chapters instead of treating it as an isolated workflow.</p>"),
    widgets.HBox([detail_level_w, max_staleness_w]),
    section_focus_w,
    widgets.HTML("<div class='ir-big-button-note'><b>Progressive disclosure:</b> executive keeps the page compact; institutional and research_deep_dive expose formulas, governance, robustness, factor diagnostics and chapter links.</div>"),
], layout=PANEL)

data_refresh_panel = widgets.VBox([
    widgets.HTML("<h3>5 · Data integrations and weekly refresh</h3><p>Priority is project DB and Database Finanziario, then local cache/API registry/yfinance, with synthetic fallback only when logged as a warning.</p>"),
    data_layers_w,
    widgets.HBox([weekly_refresh_w]),
    widgets.HTML(f"<div class='ir-card'><div class='ir-card-title'>Weekly update command</div><code>python scripts/weekly_update.py</code><br><small>Use this to refresh missing/stale datasets under DB_BASE without saving heavy files in git.</small></div>"),
], layout=PANEL)


portfolio_engine_panel = widgets.VBox([
    widgets.HTML("<h3>Portfolio Optimization & Allocation Lab</h3><p>Choose an optimizer layer above the existing analytics: PyPortfolioOpt, Riskfolio-Lib, or cvxportfolio-style convex allocation. The engine reads canonical config and existing returns.</p>"),
    widgets.HBox([portfolio_engine_w, portfolio_objective_w, portfolio_risk_model_w]),
    widgets.HBox([portfolio_min_weight_w, portfolio_max_weight_w, portfolio_leverage_w]),
    widgets.HBox([portfolio_short_w, cvar_alpha_w, rebalance_freq_w]),
    portfolio_engine_out,
], layout=PANEL)

ml_time_series_panel = widgets.VBox([
    widgets.HTML("<h3>ML & Time Series Engine</h3><p>Optional governed layer for forecasting, anomaly flags, regime detection, stock-picking deep models and crypto signals. Heavy external frameworks are treated as patterns; the notebook uses light, reusable pandas/sklearn/Keras-compatible functions by default.</p>"),
    widgets.HBox([time_series_enabled_w, time_series_model_w]),
    widgets.HBox([time_series_lookback_w, time_series_horizon_w]),
    widgets.HBox([deep_stock_enabled_w, deep_stock_model_w]),
    widgets.HBox([deep_stock_lookback_w, deep_stock_epochs_w]),
    widgets.HBox([crypto_ml_enabled_w, crypto_model_w]),
    widgets.HBox([crypto_lookback_w, crypto_horizon_w]),
    widgets.HTML("<div class='ir-big-button-note'><b>Safe integration:</b> TSLib, ForeTiS, LSTM stock and crypto repos are used as architecture references. No external repo code is copied into this notebook.</div>"),
    ml_time_series_out,
], layout=PANEL)

advanced_model_panel = widgets.VBox([
    widgets.HTML("<h3>6 · Advanced model families</h3><p>Enable the research lenses to surface in the model lab: linear, ML, time series, factor, gradient boosting, unsupervised, deep learning and recurrent nets. Advanced/deep families are exposed as governed project integrations unless the required data/model artifacts exist.</p>"),
    model_families_w,
], layout=PANEL)

advanced_panel = widgets.VBox([
    widgets.HBox([start_date_w, end_date_w, test_start_w]),
    widgets.HBox([market_w, currency_w]),
    widgets.HBox([slippage_w, tax_w, turnover_w]),
    widgets.HBox([run_backtest_w, run_explainability_w, scenario_w, sensitivity_w]),
], layout=PANEL)

preset_panel = widgets.VBox([
    widgets.HTML("<h3>Configurations</h3><p>Save and reload professional presets for repeatable research workflows.</p>"),
    widgets.HBox([preset_name_w, save_preset_btn]),
    widgets.HBox([saved_preset_w, load_preset_btn]),
], layout=PANEL)

advanced_accordion = widgets.Accordion(children=[advanced_panel, preset_panel])
advanced_accordion.set_title(0, "Advanced assumptions")
advanced_accordion.set_title(1, "Saved configurations")

setup_tabs = widgets.Tab(children=[widgets.VBox([base_panel, universe_panel, status_out]), ticker_browser_panel, model_panel, portfolio_engine_panel, ml_time_series_panel, section_config_panel, data_refresh_panel, advanced_model_panel, advanced_accordion])
setup_tabs.set_title(0, "Input Setup")
setup_tabs.set_title(1, "Ticker Browser")
setup_tabs.set_title(2, "Model Lab")
setup_tabs.set_title(3, "Portfolio Engine")
setup_tabs.set_title(4, "ML / TS Engine")
setup_tabs.set_title(5, "Section Depth")
setup_tabs.set_title(6, "Data Refresh")
setup_tabs.set_title(7, "Advanced Labs")
setup_tabs.set_title(8, "Advanced Settings")

control_header = widgets.HTML(f"""
<div class="ir-control-center-title">
  <h1>CONTROL CENTER</h1>
  <p><b>Interactive Company Selection / Valuation Setup.</b> Configure the target company, benchmark, peers, universe, data layers, model depth, section detail and project integrations here. This panel writes one canonical MASTERREQUEST and keeps the enterprise backend intact.</p>
</div>
<div class="ir-control-mini"><strong>Start here:</strong> open Input Setup, review the mini status cards, click Apply configuration, read Diagnostics, then Run research and explore Results Dashboard.</div>
<div class="ir-step-ribbon">
  <div class="ir-step-card"><span>1</span><b>Input Setup</b><br><small>Target ticker, benchmark, peer builder and watchlist.</small></div>
  <div class="ir-step-card"><span>2</span><b>Apply</b><br><small>Validate inputs and sync MASTERREQUEST/config blocks.</small></div>
  <div class="ir-step-card"><span>3</span><b>Run Research</b><br><small>Generate valuation, risk, model and backtest results.</small></div>
  <div class="ir-step-card"><span>4</span><b>Dashboard</b><br><small>Explore interactive charts, tables, diagnostics and exports.</small></div>
</div>
""")
action_bar = widgets.VBox([
    widgets.HTML("<div class='ir-action-bar'><b>Primary actions:</b> first click <span style='color:#01696f'>Apply configuration</span>, then click <span style='color:#01696f'>Run research</span>. The dashboard appears in the Results Dashboard tab.</div>"),
    widgets.HBox([apply_btn, run_btn]),
    widgets.HTML("<div class='ir-big-button-note'>Colab tip: if widgets do not render, run the setup cell once and refresh the notebook output area. This notebook enables the Colab custom widget manager automatically.</div>"),
])

with diagnostics_out:
    clear_output(wait=True)
    display(HTML("<div class='ir-status-warn'>Diagnostics will appear here after you click <b>Apply configuration</b>.</div>"))
with results_out:
    clear_output(wait=True)
    display(HTML("""
    <div class='ir-dashboard-empty'>
      <h3>Interactive Dashboard Preview</h3>
      <p>The full visual dashboard appears here after <b>Run research</b>. It is organized into eight navigable result views: Executive Summary, Portfolio, Peers, Valuation, Risk, Models, Backtest and Outputs. Data quality and methodology are embedded in the relevant result views instead of appearing as raw output blocks.</p>
      <div class='ir-grid-4'>
        <div class='ir-card'><div class='ir-card-title'>Valuation</div><div class='ir-card-value'>Bear/Base/Bull</div></div>
        <div class='ir-card'><div class='ir-card-title'>Risk</div><div class='ir-card-value'>Vol/Drawdown</div></div>
        <div class='ir-card'><div class='ir-card-title'>Models</div><div class='ir-card-value'>Leaderboard</div></div>
        <div class='ir-card'><div class='ir-card-title'>Exports</div><div class='ir-card-value'>CSV/HTML</div></div>
      </div>
    </div>
    """))
with saved_outputs_out:
    clear_output(wait=True)
    display(HTML(f"""
    <div class='ir-card'>
      <div class='ir-card-title'>Saved outputs and data roots</div>
      <p><b>Preset folder:</b><br>{CONFIG_PRESET_DIR}</p>
      <p><b>Research output root:</b><br>{OUTPUT_ROOT}</p>
      <p><b>Project data:</b><br>{PROJECT_DATA_DIR}</p>
      <p><b>External database:</b><br>{DB_BASE}</p>
      <p>CSV, HTML charts, Markdown report, static dashboard and MASTERREQUEST snapshots are written under the external output area after each run.</p>
    </div>
    """))
    display(pd.DataFrame({"available_presets": _saved_presets()}))
    display(pd.DataFrame([
        {"resource": "PROJECT_DATA_DIR", "path": str(PROJECT_DATA_DIR), "exists": PROJECT_DATA_DIR.exists()},
        {"resource": "PROJECT_CATALOG_DIR", "path": str(PROJECT_CATALOG_DIR), "exists": PROJECT_CATALOG_DIR.exists()},
        {"resource": "DB_BASE", "path": str(DB_BASE), "exists": DB_BASE.exists()},
        {"resource": "DATA_PATH", "path": str(DATA_PATH), "exists": DATA_PATH.exists()},
    ]))

control_center_panel = widgets.VBox([control_header, config_summary_out, action_bar, widgets.HTML("<div class='ir-widget-frame'>"), setup_tabs, widgets.HTML("</div>")])
main_nav = widgets.Tab(children=[
    control_center_panel,
    widgets.VBox([widgets.HTML("<div class='ir-section'>Diagnostics</div>"), diagnostics_out]),
    widgets.VBox([widgets.HTML("<div class='ir-section'>Results Dashboard - Interactive Visual Analytics</div>"), results_out]),
    widgets.VBox([widgets.HTML("<div class='ir-section'>Saved Outputs</div>"), saved_outputs_out]),
])
main_nav.set_title(0, "Input Setup")
main_nav.set_title(1, "Diagnostics")
main_nav.set_title(2, "Results Dashboard")
main_nav.set_title(3, "Saved Outputs")

display(main_nav)
_refresh_candidates()
_refresh_preview()
_refresh_config_summary()


## 0.1.5b Fintech Currency, FX and API Console


In [ ]:
# 0.1.5b Fintech UX Extension - Currency, FX, APIs and Universe Presets
import os
from IPython.display import display, clear_output, HTML
try:
    import pandas as pd
except Exception:
    pd = None
try:
    import ipywidgets as widgets
    _WIDGETS_OK = True
except Exception:
    widgets = None
    _WIDGETS_OK = False

for _name in ["MASTERREQUEST", "MASTER_REQUEST", "EXPERIMENT", "USER_SELECTION", "PORTFOLIO_CONFIG", "UNIVERSE"]:
    if _name not in globals() or globals()[_name] is None:
        globals()[_name] = {}

API_KEY_FIELDS = {
    "FMP_API_KEY": "Financial Modeling Prep", "FINNHUB_API_KEY": "Finnhub", "ALPHA_VANTAGE_API_KEY": "Alpha Vantage",
    "EODHD_API_KEY": "EODHD", "FRED_API_KEY": "FRED", "POLYGON_API_KEY": "Polygon",
}
CURRENCY_PRESETS = {
    "USD": {"benchmark": "SPY", "fx": "DX-Y.NYB", "hedge": ["UUP", "TLT", "GLD", "DBC"]},
    "EUR": {"benchmark": "SX5E.DE", "fx": "EURUSD=X", "hedge": ["EURUSD=X", "FEZ", "EWI", "GLD"]},
    "GBP": {"benchmark": "ISF.L", "fx": "GBPUSD=X", "hedge": ["GBPUSD=X", "EWU", "GLD"]},
    "CHF": {"benchmark": "EWL", "fx": "CHF=X", "hedge": ["CHF=X", "EWL", "GLD"]},
    "JPY": {"benchmark": "EWJ", "fx": "JPY=X", "hedge": ["JPY=X", "EWJ", "GLD"]},
}

def _parse_keys(text):
    out = {}
    for line in str(text or "").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        sep = "=" if "=" in line else ":" if ":" in line else None
        if sep:
            k, v = line.split(sep, 1)
            k = k.strip().upper(); v = v.strip().strip('"').strip("'")
            if k in API_KEY_FIELDS and v:
                out[k] = v
    return out

def _api_badges():
    return " ".join([f"<span style='display:inline-block;margin:3px;padding:5px 8px;border-radius:999px;background:{'#e8f5e9' if os.environ.get(k) else '#fff4df'};color:{'#1b5e20' if os.environ.get(k) else '#8a4b00'}'>{label}: {'OK' if os.environ.get(k) else 'missing'}</span>" for k,label in API_KEY_FIELDS.items()])

if not _WIDGETS_OK:
    display(HTML("<div style='background:#fff7ed;border-left:5px solid #da7101;padding:12px;border-radius:8px'>ipywidgets unavailable. FX/API extension skipped; set MASTERREQUEST and os.environ manually.</div>"))
else:
    STYLE = {"description_width": "125px"}
    W320 = widgets.Layout(width="320px")
    currency_w = widgets.Dropdown(options=list(CURRENCY_PRESETS.keys()), value=str(MASTERREQUEST.get("currency", "USD")), description="Currency", style=STYLE, layout=W320)
    fx_w = widgets.Combobox(options=[v["fx"] for v in CURRENCY_PRESETS.values()] + ["EURUSD=X", "GBPUSD=X", "JPY=X", "CHF=X", "UUP", "DX-Y.NYB"], value=str(MASTERREQUEST.get("fx_pair", CURRENCY_PRESETS["USD"]["fx"])), description="FX proxy", ensure_option=False, style=STYLE, layout=W320)
    benchmark_w = widgets.Combobox(options=[v["benchmark"] for v in CURRENCY_PRESETS.values()] + ["SPY", "QQQ", "ACWI", "SX5E.DE", "FEZ"], value=str(MASTERREQUEST.get("benchmark", MASTERREQUEST.get("benchmark_ticker", "SPY"))), description="Benchmark", ensure_option=False, style=STYLE, layout=W320)
    hedge_w = widgets.SelectMultiple(options=sorted(set(sum([v["hedge"] for v in CURRENCY_PRESETS.values()], []))), value=tuple(CURRENCY_PRESETS["USD"]["hedge"]), description="Macro/FX", style=STYLE, layout=widgets.Layout(width="480px", height="110px"))
    api_text = widgets.Textarea(value="", placeholder="FRED_API_KEY=...\nFMP_API_KEY=...", description="API keys", style=STYLE, layout=widgets.Layout(width="700px", height="90px"))
    refresh_w = widgets.Checkbox(value=True, description="Drive-first, cache-second, API-last refresh when stale", indent=False)
    universe_preset_w = widgets.Dropdown(options=list(WATCHLISTS.keys()) if "WATCHLISTS" in globals() else ["Manual"], value=(list(WATCHLISTS.keys())[0] if "WATCHLISTS" in globals() and WATCHLISTS else "Manual"), description="Universe", style=STYLE, layout=W320)
    apply_btn = widgets.Button(description="Apply FX / API extension", icon="check", button_style="success", layout=widgets.Layout(width="220px", height="40px"))
    out = widgets.Output()

    def _currency_changed(change=None):
        p = CURRENCY_PRESETS.get(currency_w.value, CURRENCY_PRESETS["USD"])
        fx_w.value = p["fx"]; benchmark_w.value = p["benchmark"]; hedge_w.value = tuple(p["hedge"])

    def _apply(_=None):
        with out:
            clear_output(wait=True)
            parsed = _parse_keys(api_text.value)
            for k, v in parsed.items(): os.environ[k] = v
            api_text.value = ""
            preset_tickers = WATCHLISTS.get(universe_preset_w.value, []) if "WATCHLISTS" in globals() else []
            MASTERREQUEST.update({"currency": currency_w.value, "reporting_currency": currency_w.value, "fx_pair": fx_w.value.strip(), "benchmark": benchmark_w.value.strip().upper(), "benchmark_ticker": benchmark_w.value.strip().upper(), "macro_fx_tickers": list(hedge_w.value), "refresh_cache": bool(refresh_w.value)})
            MASTER_REQUEST.update(MASTERREQUEST)
            EXPERIMENT.update({"currency": currency_w.value, "reporting_currency": currency_w.value, "fx_pair": fx_w.value.strip(), "benchmark_ticker": benchmark_w.value.strip().upper(), "macro_fx_tickers": list(hedge_w.value), "refresh_cache": bool(refresh_w.value)})
            PORTFOLIO_CONFIG.update({"base_currency": currency_w.value, "fx_hedge_proxy": fx_w.value.strip(), "macro_fx_tickers": list(hedge_w.value)})
            if preset_tickers:
                UNIVERSE["preset_tickers"] = preset_tickers
            display(HTML(f"<div style='background:#f6f8fb;border-left:5px solid #01696f;border-radius:8px;padding:12px'><b>Applied.</b> Currency {currency_w.value}, FX {fx_w.value}, benchmark {benchmark_w.value}. Keys updated: {len(parsed)}.<br>{_api_badges()}</div>"))
            if pd is not None:
                display(pd.DataFrame([{"currency": currency_w.value, "fx_pair": fx_w.value, "benchmark": benchmark_w.value, "macro_fx_tickers": ', '.join(hedge_w.value), "universe_preset": universe_preset_w.value}]))

    currency_w.observe(_currency_changed, names="value")
    apply_btn.on_click(_apply)
    display(HTML("""
    <div style='background:#f6f8fb;border:1px solid #d9e2ec;border-left:5px solid #01696f;border-radius:10px;padding:14px;margin:10px 0'>
      <h3 style='margin:0;color:#01696f'>Portfolio Currency, FX and API Console</h3>
      <p style='margin:6px 0 0;color:#344054'>Adds a clean overlay for reporting currency, FX proxy, macro hedge tickers and API keys without changing the portfolio engine.</p>
    </div>
    """))
    display(widgets.VBox([widgets.HBox([currency_w, fx_w, benchmark_w]), hedge_w, widgets.HBox([universe_preset_w, refresh_w]), api_text, apply_btn, out]))
    _currency_changed(); _apply()


## 2.2 Methodology & Formulas

This section explains the quantitative workflow before the engine runs. The notebook keeps a single source of truth in `MASTERREQUEST`, then derives canonical config blocks through the sync function. The formulas below describe the calculations used by the existing backend, so the user can audit the logic without reading implementation details.

| Layer | What it does | Main output |
|---|---|---|
| Price layer | Loads daily prices, computes returns and normalized curves | `price_data`, `price_metrics`, `pivot` |
| Fundamental layer | Loads company facts, multiples and analyst targets when available | `fundamentals` |
| Scoring layer | Converts heterogeneous metrics into comparable percentile scores | `valuation_ranking` |
| Peer layer | Compares selected companies through fundamentals, risk and factor scores | `peer_comparison`, `peer_similarity` |
| Valuation layer | Builds bear/base/bull target proxies and sensitivity tables | `scenario_framework`, `sensitivity` |
| Risk/backtest layer | Computes drawdown, volatility, rolling Sharpe and net strategy curve | `risk_dashboard`, `backtest_curve` |

Core return definitions:

$$
r_t = \frac{P_t}{P_{t-1}} - 1
$$

$$
R_{ann} = (1 + R_{total})^{252/N} - 1
$$

$$
\sigma_{ann} = \operatorname{std}(r_t) \sqrt{252}
$$

$$
DD_t = \frac{C_t}{\max_{s \le t} C_s} - 1
$$

Economic intuition: the platform ranks companies cross-sectionally, not in isolation. A metric is valuable only relative to the selected universe, peers, benchmark and data coverage context.

## 2.3 How Scores Are Built

All factor scores are cross-sectional percentile ranks in the active universe. Let `PR(x)` be the percentile rank of metric `x`, where higher is better after sign adjustment.

| Score | Formula | Intuition |
|---|---|---|
| Value | $$0.30PR(-PE) + 0.25PR(-P/S) + 0.20PR(-P/B) + 0.25PR(FCF\ Yield)$$ | Cheap multiples and higher free-cash-flow yield rank better. |
| Quality | $$0.25PR(Net\ Margin) + 0.25PR(Op\ Margin) + 0.25PR(ROE) + 0.25PR(-NetDebt/EBITDA)$$ | Profitable, capital-efficient, less leveraged companies rank better. |
| Momentum | $$PR(R_{ann})$$ | Stronger annualized price performance ranks better. |
| Risk | $$0.50(1-PR(\sigma_{ann})) + 0.50(1-PR(|MaxDD|))$$ | Lower volatility and smaller drawdowns rank better. |
| Growth | $$PR(GrowthProxy)$$ | Current implementation uses annual return as a growth proxy when richer estimates are unavailable. |

Composite score:

$$
Composite = 0.22Value + 0.28Quality + 0.22Momentum + 0.18Risk + 0.10Growth
$$

The score is not a buy/sell signal by itself. It is a structured research ranking that combines value, quality, momentum, risk and growth in one interpretable cross-sectional measure.

## 2.4 How Valuation Is Computed

The valuation engine uses pragmatic target proxies that work with public data and degrade gracefully when analyst targets are missing. It is a research valuation framework, not a full statutory DCF model.

| Measure | Formula | Notes |
|---|---|---|
| EV/EBITDA proxy | $$EV/EBITDA = \frac{EnterpriseValue}{EBITDA}$$ | Missing or invalid EBITDA becomes unavailable, not zero-filled. |
| FCF yield | $$FCF\ Yield = \frac{FreeCashFlow}{MarketCap}$$ | Higher yield contributes positively to value score. |
| Net debt | $$NetDebt = TotalDebt - TotalCash$$ | Used for leverage and quality diagnostics. |
| Net debt / EBITDA | $$\frac{NetDebt}{EBITDA}$$ | Lower leverage improves quality score. |
| Base target | Analyst mean target, else $$P_0(1 + clip(R_{ann}, -25\%, 25\%))$$ | Uses analyst target when available. |
| Bear target | Analyst low target, else $$P_0(1 - clip(\sigma_{ann}, 5\%, 60\%))$$ | Penalizes high volatility in downside case. |
| Bull target | Analyst high target, else $$P_0(1 + clip(\sigma_{ann}, 5\%, 60\%))$$ | Uses volatility as a rough upside range. |

Scenario upside/downside:

$$
Upside_{scenario} = \frac{Target_{scenario}}{P_0} - 1
$$

Sensitivity proxy:

$$
Target^{*} = BaseTarget \times (1 - 3\Delta r) \times (1 + 4\Delta g) \times (1 + ReRating)
$$

Economic intuition: the valuation view is designed to reveal relative asymmetry and sensitivity, especially when data quality varies across the peer set.

## 2.5 How Risk Metrics Are Computed

Risk metrics are computed from daily returns and displayed both as summary tables and as interactive Plotly charts.

| Metric | Formula | Output view |
|---|---|---|
| Annual volatility | $$\sigma_{ann} = std(r_t)\sqrt{252}$$ | Risk dashboard, risk-return scatter |
| Downside volatility | $$\sigma_{down} = std(r_t \mid r_t < 0)\sqrt{252}$$ | Price metrics |
| Max drawdown | $$\min_t \left(\frac{C_t}{\max_{s\le t} C_s} - 1\right)$$ | Drawdown chart, risk table |
| Rolling volatility | $$std(r_{t-62:t})\sqrt{252}$$ | Visual analytics, backtest |
| Rolling Sharpe | $$\frac{mean(r_{t-62:t})252}{std(r_{t-62:t})\sqrt{252}}$$ | Visual analytics, backtest |
| Net return | $$r^{net}_t = r_t - \frac{CostBps + SlippageBps}{10000 \times 252}$$ | Backtest curve |

The platform separates market risk, factor risk, valuation risk, accounting/governance placeholders and data/model risk. Missing data is surfaced through diagnostics instead of hidden behind silent defaults.

## 3. Analytics engine

In [ ]:
# 3.1 Data foundation and diagnostics

def _safe_float(x):
    try:
        if x is None or pd.isna(x): return np.nan
        return float(x)
    except Exception:
        return np.nan

def _synthetic_price_data(tickers, start, end):
    dates = pd.bdate_range(start=start, end=end)
    rows = []
    for i, ticker in enumerate(tickers):
        drift = 0.00018 + 0.00004 * (i % 5)
        vol = 0.012 + 0.002 * (i % 4)
        returns = np.random.normal(drift, vol, len(dates))
        prices = 100 * np.exp(np.cumsum(returns))
        for d, p, r in zip(dates, prices, returns):
            rows.append({"date": d, "ticker": ticker, "price": float(p), "return": float(r), "source": "synthetic_fallback"})
    return pd.DataFrame(rows)


def _ticker_cache_stems(ticker):
    base = str(ticker).strip().lower()
    normalized = base.replace(".", "_").replace("-", "_")
    return [base, normalized, f"{normalized}_1d", f"{base}_1d"]


def _candidate_price_paths(ticker):
    stems = _ticker_cache_stems(ticker)
    paths = []
    for directory in PROJECT_MARKET_CACHE_DIRS:
        for stem in stems:
            paths.extend([directory / f"{stem}.parquet", directory / f"{stem}.csv"])
    for directory in [DATA_DIR, DATA_PATH / "market_data", DB_BASE / "market_data", DB_BASE / "prices", DB_BASE / "daily_prices"]:
        for stem in stems:
            paths.extend([directory / f"{stem}.parquet", directory / f"{stem}.csv"])
    return paths


def _read_price_file(path, ticker):
    try:
        if path.suffix.lower() == ".parquet":
            raw = pd.read_parquet(path)
        elif path.suffix.lower() == ".csv":
            raw = pd.read_csv(path)
        else:
            return pd.DataFrame()
        df = raw.reset_index()
        date_col = next((c for c in ["date", "Date", "date_time", "datetime", "timestamp"] if c in df.columns), None)
        price_col = next((c for c in ["adjclose", "adjusted", "Adj Close", "Close", "close", "price"] if c in df.columns), None)
        ticker_col = next((c for c in ["ticker", "symbol", "Symbol"] if c in df.columns), None)
        if date_col is None or price_col is None:
            return pd.DataFrame()
        out = df[[date_col, price_col] + ([ticker_col] if ticker_col else [])].copy()
        out.columns = ["date", "price"] + (["ticker"] if ticker_col else [])
        out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.tz_localize(None)
        out["price"] = pd.to_numeric(out["price"], errors="coerce")
        if "ticker" not in out.columns:
            out["ticker"] = ticker
        else:
            out["ticker"] = out["ticker"].fillna(ticker).astype(str).str.upper()
            out = out[out["ticker"].eq(str(ticker).upper()) | out["ticker"].eq(str(ticker))]
            if out.empty:
                out["ticker"] = ticker
        out["ticker"] = ticker
        out = out.dropna(subset=["date", "price"]).sort_values("date")
        out["return"] = out["price"].pct_change(fill_method=None)
        out["source"] = f"project_cache:{path.relative_to(PROJECT_ROOT) if path.is_relative_to(PROJECT_ROOT) else path.name}"
        return out.dropna(subset=["return"])
    except Exception as exc:
        print(f"WARNING: failed reading local price cache {path}: {exc}")
        return pd.DataFrame()


def _load_project_price_cache(tickers, start, end):
    frames, loaded = [], []
    start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
    for ticker in tickers:
        for path in _candidate_price_paths(ticker):
            if not path.exists():
                continue
            df = _read_price_file(path, ticker)
            if df.empty:
                continue
            df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)]
            if len(df) >= 5:
                frames.append(df)
                loaded.append(ticker)
                break
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(), loaded)


def _load_project_ticker_metadata():
    frames = []
    meta_path = PROJECT_DATA_DIR / "us_equities_meta_data.csv"
    if meta_path.exists():
        try:
            meta = pd.read_csv(meta_path)
            cols = {"ticker": "ticker", "name": "company_name", "sector": "sector", "industry": "industry", "marketcap": "market_cap"}
            keep = [c for c in cols if c in meta.columns]
            meta = meta[keep].rename(columns=cols)
            meta["country"] = "United States"
            meta["currency"] = "USD"
            frames.append(meta)
        except Exception as exc:
            print(f"WARNING: failed reading project metadata {meta_path}: {exc}")
    for directory in PROJECT_MARKET_CACHE_DIRS:
        if not directory.exists():
            continue
        for path in sorted(directory.glob("*.parquet"))[:200]:
            try:
                raw = pd.read_parquet(path, columns=None).head(1).reset_index()
                if "symbol" in raw.columns:
                    frames.append(pd.DataFrame([{
                        "ticker": str(raw.get("symbol", pd.Series([path.stem])).iloc[0]).upper(),
                        "company_name": raw.get("name", pd.Series([path.stem])).iloc[0] if "name" in raw.columns else path.stem,
                        "sector": raw.get("sector", pd.Series(["Unknown"])).iloc[0] if "sector" in raw.columns else "Unknown",
                        "country": raw.get("country", pd.Series(["Unknown"])).iloc[0] if "country" in raw.columns else "Unknown",
                        "currency": raw.get("currency", pd.Series(["USD"])).iloc[0] if "currency" in raw.columns else "USD",
                    }]))
            except Exception:
                continue
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates("ticker")


def build_project_database_overview():
    rows = []
    for name, path in {
        "DB_BASE": DB_BASE,
        "DATA_PATH": DATA_PATH,
        "PROJECT_DATA_DIR": PROJECT_DATA_DIR,
        "PROJECT_CATALOG_DIR": PROJECT_CATALOG_DIR,
        "OUTPUT_ROOT": OUTPUT_ROOT,
    }.items():
        rows.append({"resource": name, "path": str(path), "exists": path.exists(), "files_visible": len(list(path.glob('*'))) if path.exists() and path.is_dir() else 0})
    catalog_path = PROJECT_CATALOG_DIR / "data_inventory.csv"
    if catalog_path.exists():
        try:
            inv = pd.read_csv(catalog_path)
            rows.append({"resource": "catalog_inventory", "path": str(catalog_path), "exists": True, "files_visible": int(inv.get("files", pd.Series(dtype=float)).sum())})
        except Exception:
            pass
    return pd.DataFrame(rows)

def load_price_layer(tickers, start, end):
    loaded_frames, loaded, failed = [], [], []
    source_preference = UNIVERSE.get("data_source", PROJECT_DATABASE_CONFIG.get("source_preference", "auto"))

    if source_preference in ["auto", "database_finanziario", "local_cache"]:
        cache_df, cache_loaded = _load_project_price_cache(tickers, start, end)
        if not cache_df.empty:
            loaded_frames.append(cache_df)
            loaded.extend(cache_loaded)

    remaining = [ticker for ticker in tickers if ticker not in loaded]
    if source_preference == "synthetic_fallback":
        failed = remaining
    elif remaining and yf is not None and source_preference in ["auto", "yfinance", "database_finanziario", "local_cache"]:
        for ticker in remaining:
            try:
                raw = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
                if raw is None or raw.empty:
                    failed.append(ticker)
                    continue
                price_col = "Close" if "Close" in raw.columns else raw.columns[0]
                df = raw[[price_col]].reset_index()
                df.columns = ["date", "price"]
                df["ticker"] = ticker
                df["return"] = df["price"].pct_change(fill_method=None)
                df["source"] = "yfinance"
                loaded_frames.append(df.dropna())
                loaded.append(ticker)
            except Exception as exc:
                print(f"WARNING: yfinance failed for {ticker}: {exc}")
                failed.append(ticker)
    else:
        failed = remaining

    still_missing = [ticker for ticker in tickers if ticker not in loaded]
    if still_missing:
        loaded_frames.append(_synthetic_price_data(still_missing, start, end))
        failed = sorted(set(failed + still_missing))
    if not loaded_frames:
        return _synthetic_price_data(tickers, start, end), [], tickers
    return pd.concat(loaded_frames, ignore_index=True), loaded, failed


def load_risk_factor_layer(start, end):
    """Load institutional risk-factor proxies using project cache/API fallback.

    The factor map translates project alpha-factor/Fama-French ideas into liquid
    ETF/FX/commodity proxies so the notebook works in Colab without licensed
    factor feeds. If a proxy is unavailable, load_price_layer creates a logged
    synthetic fallback and diagnostics surface the source.
    """
    factor_defs = RISK_CONFIG.get("risk_factor_library", RISK_FACTOR_LIBRARY)
    proxy_tickers = []
    for meta in factor_defs.values():
        for key in ["ticker", "benchmark"]:
            ticker = meta.get(key)
            if ticker and ticker not in proxy_tickers:
                proxy_tickers.append(ticker)
    factor_prices, loaded, failed = load_price_layer(proxy_tickers, start, end)
    if factor_prices.empty:
        return pd.DataFrame(), pd.DataFrame()
    wide = factor_prices.pivot_table(index="date", columns="ticker", values="price", aggfunc="last").sort_index().ffill()
    returns = wide.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    factors = pd.DataFrame(index=returns.index)
    metadata = []
    for name, meta in factor_defs.items():
        ticker = meta.get("ticker")
        benchmark = meta.get("benchmark")
        if ticker not in returns.columns:
            series = pd.Series(np.nan, index=returns.index)
        elif benchmark and benchmark in returns.columns:
            series = returns[ticker] - returns[benchmark]
        else:
            series = returns[ticker]
        factors[name] = series
        metadata.append({
            "factor": name,
            "proxy_ticker": ticker,
            "benchmark_proxy": benchmark or "",
            "family": meta.get("family", "custom"),
            "description": meta.get("description", ""),
            "source_status": "loaded" if ticker in loaded else "fallback_or_missing" if ticker in failed else "derived",
        })
    factors = factors.reset_index().rename(columns={"index": "date"})
    return factors, pd.DataFrame(metadata)

def _ticker_info(ticker):
    if yf is None: return {}
    try: return yf.Ticker(ticker).info or {}
    except Exception: return {}

def load_fundamental_layer(tickers):
    project_meta = _load_project_ticker_metadata()
    meta_lookup = project_meta.set_index("ticker").to_dict("index") if not project_meta.empty and "ticker" in project_meta.columns else {}
    rows = []
    for ticker in tickers:
        info = _ticker_info(ticker)
        local = meta_lookup.get(str(ticker).upper(), {})
        rows.append({
            "ticker": ticker,
            "company_name": info.get("longName") or local.get("company_name") or ticker,
            "sector": info.get("sector") or local.get("sector") or "Unknown",
            "industry": info.get("industry") or local.get("industry") or "Unknown",
            "country": info.get("country") or local.get("country") or "Unknown",
            "currency": info.get("currency") or local.get("currency") or UNIVERSE.get("reporting_currency", "USD"),
            "market_cap": _safe_float(info.get("marketCap", local.get("market_cap"))),
            "enterprise_value": _safe_float(info.get("enterpriseValue")),
            "revenue": _safe_float(info.get("totalRevenue")),
            "ebitda": _safe_float(info.get("ebitda")),
            "gross_margin": _safe_float(info.get("grossMargins")),
            "operating_margin": _safe_float(info.get("operatingMargins")),
            "profit_margin": _safe_float(info.get("profitMargins")),
            "return_on_equity": _safe_float(info.get("returnOnEquity")),
            "return_on_assets": _safe_float(info.get("returnOnAssets")),
            "total_debt": _safe_float(info.get("totalDebt")),
            "total_cash": _safe_float(info.get("totalCash")),
            "current_ratio": _safe_float(info.get("currentRatio")),
            "debt_to_equity": _safe_float(info.get("debtToEquity")),
            "free_cashflow": _safe_float(info.get("freeCashflow")),
            "operating_cashflow": _safe_float(info.get("operatingCashflow")),
            "dividend_yield": _safe_float(info.get("dividendYield")),
            "payout_ratio": _safe_float(info.get("payoutRatio")),
            "beta": _safe_float(info.get("beta")),
            "trailing_pe": _safe_float(info.get("trailingPE")),
            "forward_pe": _safe_float(info.get("forwardPE")),
            "price_to_book": _safe_float(info.get("priceToBook")),
            "price_to_sales": _safe_float(info.get("priceToSalesTrailing12Months")),
            "recommendation": info.get("recommendationKey", "n/a"),
            "target_mean_price": _safe_float(info.get("targetMeanPrice")),
            "target_high_price": _safe_float(info.get("targetHighPrice")),
            "target_low_price": _safe_float(info.get("targetLowPrice")),
            "analyst_count": _safe_float(info.get("numberOfAnalystOpinions")),
            "fundamental_source": "yfinance+project_metadata" if info and local else "yfinance" if info else "project_metadata" if local else "empty",
        })
    return pd.DataFrame(rows)

def build_price_metrics(price_df):
    pivot = price_df.pivot_table(index="date", columns="ticker", values="price").sort_index()
    returns = pivot.pct_change(fill_method=None).dropna()
    rows = []
    for ticker in pivot.columns:
        series = pivot[ticker].dropna()
        r = returns[ticker].dropna() if ticker in returns.columns else pd.Series(dtype=float)
        if len(series) < 5: continue
        total_return = series.iloc[-1] / series.iloc[0] - 1
        ann_return = (1 + total_return) ** (252 / max(len(r), 1)) - 1 if len(r) > 0 else 0.0
        vol = r.std() * np.sqrt(252) if len(r) > 1 else 0.0
        curve = (1 + r).cumprod() if len(r) > 0 else pd.Series([1.0])
        drawdown = curve / curve.cummax() - 1
        rows.append({
            "ticker": ticker,
            "last_price": series.iloc[-1],
            "total_return": total_return,
            "annual_return": ann_return,
            "volatility": vol,
            "downside_volatility": r[r < 0].std() * np.sqrt(252) if len(r[r < 0]) > 1 else 0.0,
            "max_drawdown": drawdown.min() if len(drawdown) else 0.0,
            "sharpe_proxy": ann_return / vol if vol and vol > 0 else np.nan,
            "rolling_vol_63d": r.rolling(63).std().iloc[-1] * np.sqrt(252) if len(r) >= 63 else vol,
            "rolling_sharpe_63d": (r.rolling(63).mean().iloc[-1] * 252) / (r.rolling(63).std().iloc[-1] * np.sqrt(252)) if len(r) >= 63 and r.rolling(63).std().iloc[-1] else np.nan,
        })
    return pd.DataFrame(rows), pivot, returns

def build_data_quality_layer(price_df, fund_df, loaded, failed):
    price_coverage = price_df.groupby("ticker").agg(
        price_rows=("price", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        missing_price=("price", lambda s: int(s.isna().sum())),
        source=("source", lambda s: ",".join(sorted(set(s.astype(str)))))
    ).reset_index() if not price_df.empty else pd.DataFrame()
    fund_missing = fund_df.set_index("ticker").isna().mean(axis=1).reset_index(name="fundamental_missing_ratio") if not fund_df.empty else pd.DataFrame()
    coverage = price_coverage.merge(fund_missing, on="ticker", how="outer")
    coverage["data_status"] = np.where(coverage.get("fundamental_missing_ratio", 1).fillna(1) < 0.50, "usable", "limited")
    source_mix = price_df.groupby("source")["ticker"].nunique().reset_index(name="ticker_count") if not price_df.empty else pd.DataFrame()
    audit = pd.DataFrame({
        "metric": ["price_rows", "fundamental_rows", "loaded_tickers", "fallback_or_failed_tickers", "source_count"],
        "value": [len(price_df), len(fund_df), len(loaded), len(failed), len(source_mix)],
    })
    return coverage, source_mix, audit


def _pct_rank(series, ascending=True, default=0.5):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if values.notna().sum() == 0:
        return pd.Series(default, index=series.index, dtype="float64")
    return values.rank(pct=True, ascending=ascending).fillna(default)


def table_sws_scorecard(df):
    """Simply-Wall-St-inspired 0-6 scorecard for portfolio candidates."""
    if df is None or df.empty:
        return pd.DataFrame()
    out = df[["ticker"]].copy()
    value = 0.45 * _pct_rank(df.get("trailing_pe", pd.Series(index=df.index)), ascending=False) + 0.35 * _pct_rank(df.get("price_to_sales", pd.Series(index=df.index)), ascending=False) + 0.20 * _pct_rank(df.get("upside_base", pd.Series(index=df.index)), ascending=True)
    future = 0.55 * _pct_rank(df.get("growth_score", pd.Series(index=df.index)), ascending=True) + 0.45 * _pct_rank(df.get("upside_bull", pd.Series(index=df.index)), ascending=True)
    past = 0.50 * _pct_rank(df.get("annual_return", pd.Series(index=df.index)), ascending=True) + 0.50 * _pct_rank(df.get("momentum_score", pd.Series(index=df.index)), ascending=True)
    health = 0.50 * _pct_rank(df.get("net_debt_to_ebitda", pd.Series(index=df.index)), ascending=False) + 0.50 * _pct_rank(df.get("risk_score", pd.Series(index=df.index)), ascending=True)
    income = 0.65 * _pct_rank(df.get("dividend_yield", pd.Series(index=df.index)), ascending=True) + 0.35 * _pct_rank(df.get("fcf_yield", pd.Series(index=df.index)), ascending=True)
    out["value_axis"] = (value * 6).clip(0, 6)
    out["future_axis"] = (future * 6).clip(0, 6)
    out["past_axis"] = (past * 6).clip(0, 6)
    out["health_axis"] = (health * 6).clip(0, 6)
    out["income_axis"] = (income * 6).clip(0, 6)
    out["sws_total_score"] = out[["value_axis", "future_axis", "past_axis", "health_axis", "income_axis"]].mean(axis=1)
    out["sws_quality_label"] = pd.cut(out["sws_total_score"], bins=[-0.01, 2.0, 4.0, 6.01], labels=["weak", "balanced", "strong"])
    return out.sort_values("sws_total_score", ascending=False).reset_index(drop=True)


def table_feature_missingness(df):
    if df is None or df.empty:
        return pd.DataFrame()
    feature_cols = [c for c in ["value_score", "quality_score", "momentum_score", "risk_score", "growth_score", "size_score", "trailing_pe", "price_to_sales", "volatility", "annual_return", "upside_base"] if c in df.columns]
    rows = []
    for col in feature_cols:
        rows.append({"feature": col, "missing_pct": float(df[col].isna().mean()), "non_missing_n": int(df[col].notna().sum()), "block": col.split("_")[0]})
    return pd.DataFrame(rows).sort_values(["missing_pct", "feature"], ascending=[False, True])


def table_data_source_summary(price_df, fund_df, loaded, failed, source_mix):
    rows = [
        {"series": "price_data", "source": "mixed_project_yfinance_synthetic", "rows": len(price_df), "real_or_synthetic": "mixed" if failed else "real/cache"},
        {"series": "fundamentals", "source": "yfinance_project_metadata", "rows": len(fund_df), "real_or_synthetic": "mixed"},
        {"series": "loaded_tickers", "source": ", ".join(loaded) if loaded else "none", "rows": len(loaded), "real_or_synthetic": "real/cache"},
        {"series": "fallback_tickers", "source": ", ".join(failed) if failed else "none", "rows": len(failed), "real_or_synthetic": "synthetic" if failed else "none"},
    ]
    if source_mix is not None and not source_mix.empty:
        for _, r in source_mix.iterrows():
            rows.append({"series": "source_mix", "source": r.get("source"), "rows": r.get("ticker_count"), "real_or_synthetic": "synthetic" if "synthetic" in str(r.get("source")) else "real/cache"})
    return pd.DataFrame(rows)


def table_robustness_checks(valuation_df, data_quality, model_leaderboard, risk_table):
    rows = []
    rows.append({"check": "Minimum universe size", "status": "PASS" if valuation_df is not None and len(valuation_df) >= 3 else "WARN", "detail": f"n={0 if valuation_df is None else len(valuation_df)}"})
    missing = data_quality.get("fundamental_missing_ratio", pd.Series(dtype=float)).mean() if data_quality is not None and not data_quality.empty else np.nan
    rows.append({"check": "Fundamental coverage", "status": "PASS" if pd.notna(missing) and missing < 0.60 else "WARN", "detail": f"avg missing={missing:.2%}" if pd.notna(missing) else "n/a"})
    valid_models = model_leaderboard[model_leaderboard.get("governance_status", pd.Series(dtype=str)).eq("PASS")] if model_leaderboard is not None and not model_leaderboard.empty else pd.DataFrame()
    rows.append({"check": "Model leaderboard", "status": "PASS" if len(valid_models) >= 1 else "WARN", "detail": f"valid models={len(valid_models)}"})
    warn_count = int((risk_table.get("status", pd.Series(dtype=str)) == "WARN").sum()) if risk_table is not None and not risk_table.empty else 0
    rows.append({"check": "Risk warning count", "status": "PASS" if warn_count <= 4 else "WARN", "detail": f"WARN={warn_count}"})
    if valuation_df is not None and not valuation_df.empty and "composite_score" in valuation_df.columns:
        dispersion = valuation_df["composite_score"].std()
        rows.append({"check": "Ranking dispersion", "status": "PASS" if pd.notna(dispersion) and dispersion > 0.01 else "WARN", "detail": f"std={dispersion:.4f}" if pd.notna(dispersion) else "n/a"})
    return pd.DataFrame(rows)


In [ ]:
# 3.2 Research calculations

def build_valuation_metrics(fund_df, price_metrics):
    df = fund_df.merge(price_metrics, on="ticker", how="left")
    df["ev_to_ebitda_proxy"] = np.where(df["ebitda"].abs() > 0, df["enterprise_value"] / df["ebitda"], np.nan)
    df["fcf_yield"] = np.where(df["market_cap"].abs() > 0, df["free_cashflow"] / df["market_cap"], np.nan)
    df["net_debt"] = df["total_debt"].fillna(0) - df["total_cash"].fillna(0)
    df["net_debt_to_ebitda"] = np.where(df["ebitda"].abs() > 0, df["net_debt"] / df["ebitda"], np.nan)
    df["sales_multiple_proxy"] = np.where(df["revenue"].abs() > 0, df["market_cap"] / df["revenue"], np.nan)
    df["growth_proxy"] = df["annual_return"].fillna(0)
    df["size_score"] = df["market_cap"].rank(pct=True).fillna(0.5)

    df["value_score"] = ((-df["trailing_pe"].replace([np.inf, -np.inf], np.nan)).rank(pct=True).fillna(0.5) * 0.30 + (-df["price_to_sales"].replace([np.inf, -np.inf], np.nan)).rank(pct=True).fillna(0.5) * 0.25 + (-df["price_to_book"].replace([np.inf, -np.inf], np.nan)).rank(pct=True).fillna(0.5) * 0.20 + df["fcf_yield"].replace([np.inf, -np.inf], np.nan).rank(pct=True).fillna(0.5) * 0.25)
    df["quality_score"] = (df["profit_margin"].rank(pct=True).fillna(0.5) * 0.25 + df["operating_margin"].rank(pct=True).fillna(0.5) * 0.25 + df["return_on_equity"].rank(pct=True).fillna(0.5) * 0.25 + (-df["net_debt_to_ebitda"].replace([np.inf, -np.inf], np.nan)).rank(pct=True).fillna(0.5) * 0.25)
    df["momentum_score"] = df["annual_return"].rank(pct=True).fillna(0.5)
    df["risk_score"] = (1 - df["volatility"].rank(pct=True).fillna(0.5)) * 0.50 + (1 - (-df["max_drawdown"]).rank(pct=True).fillna(0.5)) * 0.50
    df["growth_score"] = df["growth_proxy"].rank(pct=True).fillna(0.5)

    df["composite_score"] = 0.22 * df["value_score"] + 0.28 * df["quality_score"] + 0.22 * df["momentum_score"] + 0.18 * df["risk_score"] + 0.10 * df["growth_score"]
    df["rank"] = df["composite_score"].rank(ascending=False, method="dense").astype(int)
    df["base_target_proxy"] = np.where(df["target_mean_price"].fillna(0) > 0, df["target_mean_price"], df["last_price"] * (1 + df["annual_return"].fillna(0).clip(-0.25, 0.25)))
    df["bear_target_proxy"] = np.where(df["target_low_price"].fillna(0) > 0, df["target_low_price"], df["last_price"] * (1 - df["volatility"].fillna(0.20).clip(0.05, 0.60)))
    df["bull_target_proxy"] = np.where(df["target_high_price"].fillna(0) > 0, df["target_high_price"], df["last_price"] * (1 + df["volatility"].fillna(0.20).clip(0.05, 0.60)))
    df["upside_base"] = df["base_target_proxy"] / df["last_price"] - 1
    df["upside_bull"] = df["bull_target_proxy"] / df["last_price"] - 1
    df["downside_bear"] = df["bear_target_proxy"] / df["last_price"] - 1
    return df.sort_values(["rank", "ticker"]).reset_index(drop=True)

def refine_peers_with_similarity(valuation_df):
    mode = UNIVERSE["peer_mode"]
    main = UNIVERSE["main_ticker"]
    if mode not in ["clustered", "factor-similar", "custom-screened"]:
        return UNIVERSE["peers"], pd.DataFrame()
    feature_cols = ["market_cap", "revenue", "profit_margin", "return_on_equity", "trailing_pe", "price_to_sales", "annual_return", "volatility", "quality_score", "value_score", "momentum_score"]
    available = [c for c in feature_cols if c in valuation_df.columns]
    df = valuation_df.copy()
    if main not in df["ticker"].tolist() or len(df) < 3 or StandardScaler is None:
        return UNIVERSE["peers"], pd.DataFrame()
    X = df[available].replace([np.inf, -np.inf], np.nan).fillna(df[available].median(numeric_only=True)).fillna(0)
    Xs = StandardScaler().fit_transform(X)
    main_idx = df.index[df["ticker"] == main][0]
    distances = np.sqrt(((Xs - Xs[main_idx]) ** 2).sum(axis=1))
    df["similarity_distance"] = distances
    df["similarity_score"] = 1 / (1 + distances)
    if KMeans is not None and mode == "clustered":
        n_clusters = min(3, max(1, len(df)//3))
        labels = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit_predict(Xs)
        df["peer_cluster"] = labels
        main_cluster = df.loc[df["ticker"] == main, "peer_cluster"].iloc[0]
        selected = df[(df["peer_cluster"] == main_cluster) & (df["ticker"] != main)].sort_values("similarity_distance")["ticker"].head(8).tolist()
    elif mode == "custom-screened":
        selected = df[(df["composite_score"] >= df["composite_score"].median()) & (df["ticker"] != main)].sort_values("similarity_distance")["ticker"].head(8).tolist()
    else:
        selected = df[df["ticker"] != main].sort_values("similarity_distance")["ticker"].head(8).tolist()
    return selected, df[[c for c in ["ticker", "similarity_distance", "similarity_score", "peer_cluster"] if c in df.columns]].sort_values("similarity_distance")

def table_company_overview(df):
    cols = ["ticker", "company_name", "sector", "industry", "country", "market_cap", "revenue", "gross_margin", "operating_margin", "profit_margin", "return_on_equity", "trailing_pe", "forward_pe", "price_to_sales", "last_price", "base_target_proxy", "upside_base", "composite_score"]
    main = df[df["ticker"] == UNIVERSE["main_ticker"]]
    return main[[c for c in cols if c in main.columns]].copy()

def table_peer_comparison(df):
    tickers = [UNIVERSE["main_ticker"]] + UNIVERSE["peers"]
    cols = ["ticker", "company_name", "market_cap", "revenue", "profit_margin", "return_on_equity", "trailing_pe", "forward_pe", "price_to_sales", "price_to_book", "ev_to_ebitda_proxy", "annual_return", "volatility", "max_drawdown", "value_score", "quality_score", "momentum_score", "risk_score", "growth_score", "composite_score", "rank"]
    return df[df["ticker"].isin(tickers)][[c for c in cols if c in df.columns]].sort_values("rank")

def table_scenarios(df):
    rows = []
    for _, r in df.iterrows():
        last = r.get("last_price", np.nan)
        if pd.isna(last) or last == 0: continue
        bear, base, bull = r.get("bear_target_proxy", last * 0.85), r.get("base_target_proxy", last), r.get("bull_target_proxy", last * 1.15)
        rows.append({"ticker": r["ticker"], "current_price": last, "bear_target": bear, "base_target": base, "bull_target": bull, "bear_return": bear/last-1, "base_return": base/last-1, "bull_return": bull/last-1, "valuation_risk": abs(bear/last-1), "asymmetry": (bull/last-1)/abs(bear/last-1) if abs(bear/last-1)>0 else np.nan})
    return pd.DataFrame(rows)

def table_sensitivity(df):
    main = UNIVERSE["main_ticker"]
    row = df[df["ticker"] == main]
    if row.empty: return pd.DataFrame()
    row = row.iloc[0]
    base_target = row.get("base_target_proxy", row.get("last_price", 100))
    rows = []
    for dr in VALUATION_CONFIG["sensitivity_grid"]["discount_rate"]:
        for rerating in VALUATION_CONFIG["sensitivity_grid"]["multiple_re_rating"]:
            for tg in VALUATION_CONFIG["sensitivity_grid"]["terminal_growth"]:
                rows.append({"ticker": main, "discount_rate_delta": dr, "terminal_growth_delta": tg, "multiple_rerating": rerating, "target_proxy": base_target * (1 - dr * 3) * (1 + tg * 4) * (1 + rerating)})
    return pd.DataFrame(rows)

def table_factor_signals(df):
    cols = ["ticker", "value_score", "quality_score", "momentum_score", "risk_score", "growth_score", "size_score", "composite_score", "rank"]
    return df[[c for c in cols if c in df.columns]].sort_values("rank")


def table_portfolio_allocation(df):
    if df is None or df.empty:
        return pd.DataFrame()
    alloc = df.copy()
    alloc["score_for_weight"] = pd.to_numeric(alloc.get("composite_score", 0.0), errors="coerce").clip(lower=0).fillna(0)
    if alloc["score_for_weight"].sum() <= 0:
        alloc["target_weight"] = 1.0 / len(alloc)
    else:
        alloc["target_weight"] = alloc["score_for_weight"] / alloc["score_for_weight"].sum()
    max_weight = min(0.35, max(0.08, float(PORTFOLIO_CONFIG.get("turnover_limit", 0.30))))
    alloc["target_weight_capped"] = alloc["target_weight"].clip(upper=max_weight)
    alloc["target_weight_capped"] = alloc["target_weight_capped"] / alloc["target_weight_capped"].sum()
    alloc["current_weight_proxy"] = 1.0 / len(alloc)
    alloc["active_weight"] = alloc["target_weight_capped"] - alloc["current_weight_proxy"]
    cols = ["ticker", "rank", "target_weight_capped", "current_weight_proxy", "active_weight", "composite_score", "annual_return", "volatility", "max_drawdown", "upside_base"]
    return alloc[[c for c in cols if c in alloc.columns]].sort_values("target_weight_capped", ascending=False).reset_index(drop=True)


def table_portfolio_factor_exposure(allocation_df, factor_df):
    if allocation_df is None or allocation_df.empty or factor_df is None or factor_df.empty:
        return pd.DataFrame()
    factors = ["value_score", "quality_score", "momentum_score", "risk_score", "growth_score", "size_score"]
    merged = allocation_df[["ticker", "target_weight_capped"]].merge(factor_df[[c for c in ["ticker"] + factors if c in factor_df.columns]], on="ticker", how="left")
    rows = []
    for factor in [c for c in factors if c in merged.columns]:
        values = pd.to_numeric(merged[factor], errors="coerce")
        rows.append({"factor": factor, "weighted_exposure": (values * merged["target_weight_capped"]).sum(), "min_security_score": values.min(), "max_security_score": values.max()})
    return pd.DataFrame(rows)


def table_benchmark_comparison(backtest_perf, backtest_curve):
    rows = []
    if backtest_perf is not None and not backtest_perf.empty:
        row = backtest_perf.iloc[0].to_dict()
        rows.extend([
            {"metric": "strategy_annual_return_net", "value": row.get("annual_return_net"), "source": "backtest_performance"},
            {"metric": "strategy_annual_vol_net", "value": row.get("annual_vol_net"), "source": "backtest_performance"},
            {"metric": "strategy_max_drawdown_net", "value": row.get("max_drawdown_net"), "source": "backtest_performance"},
            {"metric": "strategy_rolling_sharpe_latest", "value": row.get("rolling_sharpe_latest"), "source": "backtest_performance"},
        ])
    if backtest_curve is not None and not backtest_curve.empty and {"strategy_net", "benchmark"}.issubset(backtest_curve.columns):
        strat_total = backtest_curve["strategy_net"].iloc[-1] - 1
        bench_total = backtest_curve["benchmark"].iloc[-1] - 1
        rows.extend([
            {"metric": "strategy_total_return", "value": strat_total, "source": "backtest_curve"},
            {"metric": "benchmark_total_return", "value": bench_total, "source": "backtest_curve"},
            {"metric": "active_total_return", "value": strat_total - bench_total, "source": "backtest_curve"},
        ])
    return pd.DataFrame(rows)


def table_optimization_summary(df, allocation_df):
    if df is None or df.empty:
        return pd.DataFrame()
    opt = allocation_df.copy() if allocation_df is not None and not allocation_df.empty else table_portfolio_allocation(df)
    if opt.empty:
        return pd.DataFrame()
    ret = pd.to_numeric(opt.get("annual_return", 0.0), errors="coerce").fillna(0.0)
    vol = pd.to_numeric(opt.get("volatility", 0.0), errors="coerce").replace(0, np.nan)
    score = pd.to_numeric(opt.get("composite_score", 0.0), errors="coerce").fillna(0.0)
    opt["risk_adjusted_score"] = (ret / vol).replace([np.inf, -np.inf], np.nan).fillna(0.0) * 0.45 + score * 0.55
    opt["optimized_weight"] = opt["risk_adjusted_score"].clip(lower=0)
    if opt["optimized_weight"].sum() <= 0:
        opt["optimized_weight"] = opt["target_weight_capped"]
    else:
        opt["optimized_weight"] = opt["optimized_weight"] / opt["optimized_weight"].sum()
    opt["weight_change_vs_target"] = opt["optimized_weight"] - opt["target_weight_capped"]
    cols = ["ticker", "optimized_weight", "target_weight_capped", "weight_change_vs_target", "risk_adjusted_score", "annual_return", "volatility", "composite_score"]
    return opt[[c for c in cols if c in opt.columns]].sort_values("optimized_weight", ascending=False).reset_index(drop=True)


def table_portfolio_scenarios(allocation_df, scenarios):
    if allocation_df is None or allocation_df.empty or scenarios is None or scenarios.empty:
        return pd.DataFrame()
    merged = allocation_df[["ticker", "target_weight_capped"]].merge(scenarios, on="ticker", how="left")
    rows = []
    for scenario, col in [("bear", "bear_return"), ("base", "base_return"), ("bull", "bull_return")]:
        values = pd.to_numeric(merged[col], errors="coerce") if col in merged.columns else pd.Series(dtype=float)
        rows.append({"scenario": scenario, "portfolio_return": (values.fillna(0) * merged["target_weight_capped"]).sum(), "worst_security_return": values.min() if not values.empty else np.nan, "best_security_return": values.max() if not values.empty else np.nan, "coverage_weight": merged.loc[values.notna(), "target_weight_capped"].sum() if not values.empty else 0.0})
    return pd.DataFrame(rows)

def table_model_lab(df):
    feature_cols = ["value_score", "quality_score", "momentum_score", "risk_score", "growth_score", "size_score", "volatility", "max_drawdown"]
    clean = df.dropna(subset=["composite_score"]).copy()
    rows, contribution = [], pd.DataFrame()
    if len(clean) < 3:
        return pd.DataFrame([{"model": "insufficient_data", "mae": np.nan, "r2": np.nan}]), contribution
    X = clean[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = clean["composite_score"]
    for model_name in ML_CONFIG["selected_models"]:
        try:
            model = build_model(model_name)
            model.fit(X, y)
            pred = model.predict(X)
            rows.append({"model": model_name, "mae": float(np.mean(np.abs(pred-y))), "r2": float(1 - ((pred-y)**2).sum()/((y-y.mean())**2).sum()) if ((y-y.mean())**2).sum() else np.nan})
            candidate_importance = _model_feature_importance(model, feature_cols)
            if not candidate_importance.empty:
                contribution = candidate_importance
        except Exception as exc:
            rows.append({"model": model_name, "mae": np.nan, "r2": np.nan, "error": str(exc)})
    return pd.DataFrame(rows).sort_values("mae"), contribution

def table_backtest(df, pivot):
    if pivot.empty or df.empty: return pd.DataFrame(), pd.DataFrame()
    top = df.sort_values("composite_score", ascending=False)["ticker"].head(min(5, len(df))).tolist()
    top = [t for t in top if t in pivot.columns]
    bench = UNIVERSE["benchmark"] if UNIVERSE["benchmark"] in pivot.columns else None
    if not top: return pd.DataFrame(), pd.DataFrame()
    returns = pivot[top].pct_change(fill_method=None).dropna().mean(axis=1)
    gross_curve = (1 + returns).cumprod()
    cost_drag = (PORTFOLIO_CONFIG["transaction_cost_bps"] + PORTFOLIO_CONFIG["slippage_bps"]) / 10000 / 12
    net_returns = returns - cost_drag / 21
    net_curve = (1 + net_returns).cumprod()
    curve = pd.DataFrame({"date": returns.index, "strategy_gross": gross_curve.values, "strategy_net": net_curve.values})
    if bench:
        b = pivot[bench].pct_change(fill_method=None).dropna()
        curve = curve.merge(pd.DataFrame({"date": b.index, "benchmark": (1+b).cumprod().values}), on="date", how="left")
    perf = pd.DataFrame([{"portfolio": "top_ranked_strategy", "tickers": ", ".join(top), "annual_return_net": net_returns.mean()*252, "annual_vol_net": net_returns.std()*np.sqrt(252), "max_drawdown_net": (net_curve/net_curve.cummax()-1).min(), "rolling_vol_latest": net_returns.rolling(63).std().iloc[-1]*np.sqrt(252) if len(net_returns)>=63 else np.nan, "rolling_sharpe_latest": (net_returns.rolling(63).mean().iloc[-1]*252)/(net_returns.rolling(63).std().iloc[-1]*np.sqrt(252)) if len(net_returns)>=63 and net_returns.rolling(63).std().iloc[-1] else np.nan, "estimated_monthly_cost_drag": cost_drag}])
    return curve, perf


def table_risk_factor_exposures(allocation_df, pivot, risk_factors):
    """Estimate portfolio beta/exposure to Fama-French-style, FX and commodity proxies."""
    if allocation_df is None or allocation_df.empty or pivot is None or pivot.empty or risk_factors is None or risk_factors.empty:
        return pd.DataFrame()
    weights = allocation_df.set_index("ticker")["target_weight_capped"]
    available = [ticker for ticker in weights.index if ticker in pivot.columns]
    if not available:
        return pd.DataFrame()
    asset_returns = pivot[available].pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    weights = weights.reindex(available).fillna(0)
    weights = weights / weights.sum() if weights.sum() else weights
    portfolio_returns = asset_returns.mul(weights, axis=1).sum(axis=1).rename("portfolio_return")
    factors = risk_factors.copy()
    factors["date"] = pd.to_datetime(factors["date"])
    factors = factors.set_index("date").sort_index()
    data = pd.concat([portfolio_returns, factors], axis=1).dropna(how="any")
    factor_cols = [c for c in factors.columns if c in data.columns]
    rows = []
    if len(data) < 30 or not factor_cols:
        return pd.DataFrame([{"factor": "insufficient_factor_history", "beta": np.nan, "correlation": np.nan, "t_stat_proxy": np.nan, "family": "diagnostic", "description": "Need at least 30 aligned observations."}])
    y = data["portfolio_return"]
    y_std = y.std()
    for factor in factor_cols:
        x = data[factor]
        beta = x.cov(y) / x.var() if x.var() and not pd.isna(x.var()) else np.nan
        corr = x.corr(y)
        t_proxy = corr * np.sqrt((len(data) - 2) / max(1e-9, 1 - corr**2)) if pd.notna(corr) and abs(corr) < 1 else np.nan
        meta = RISK_CONFIG.get("risk_factor_library", RISK_FACTOR_LIBRARY).get(factor, {})
        contribution = beta * x.std() / y_std if pd.notna(beta) and y_std and pd.notna(x.std()) else np.nan
        rows.append({
            "factor": factor,
            "family": meta.get("family", "custom"),
            "proxy_ticker": meta.get("ticker", ""),
            "description": meta.get("description", ""),
            "beta": beta,
            "correlation": corr,
            "t_stat_proxy": t_proxy,
            "risk_contribution_proxy": contribution,
            "observations": len(data),
        })
    return pd.DataFrame(rows).sort_values("risk_contribution_proxy", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)


def table_factor_risk_summary(risk_factor_exposures):
    if risk_factor_exposures is None or risk_factor_exposures.empty:
        return pd.DataFrame()
    df = risk_factor_exposures.copy()
    df["abs_contribution"] = pd.to_numeric(df.get("risk_contribution_proxy"), errors="coerce").abs()
    return df.groupby("family", dropna=False).agg(
        n_factors=("factor", "count"),
        avg_abs_beta=("beta", lambda s: pd.to_numeric(s, errors="coerce").abs().mean()),
        max_abs_risk_contribution=("abs_contribution", "max"),
    ).reset_index().sort_values("max_abs_risk_contribution", ascending=False)

def table_risk_dashboard(df, failed, factor_risk_summary=None):
    factor_score = df[["value_score", "quality_score", "momentum_score", "risk_score"]].std().mean()
    factor_driver = "factor dispersion"
    if factor_risk_summary is not None and not factor_risk_summary.empty and "max_abs_risk_contribution" in factor_risk_summary.columns:
        factor_score = pd.to_numeric(factor_risk_summary["max_abs_risk_contribution"], errors="coerce").max()
        top_family = factor_risk_summary.iloc[0].get("family", "factor")
        factor_driver = f"factor model exposure: {top_family}"
    rows = [
        {"risk_family": "Market Risk", "score": df["volatility"].mean(), "status": "WARN" if df["volatility"].mean() > 0.35 else "PASS", "drivers": "volatility, drawdown, beta proxy"},
        {"risk_family": "Factor Risk", "score": factor_score, "status": "WARN", "drivers": factor_driver},
        {"risk_family": "Fundamental / Valuation Risk", "score": df["net_debt_to_ebitda"].replace([np.inf, -np.inf], np.nan).abs().median(), "status": "WARN", "drivers": "leverage, multiples, asymmetry"},
        {"risk_family": "Accounting / Forensic Risk", "score": np.nan, "status": "WARN", "drivers": "limited statement granularity"},
        {"risk_family": "Governance / Structural Risk", "score": np.nan, "status": "WARN", "drivers": "ESG/governance provider needed"},
        {"risk_family": "Data / Model Risk", "score": len(failed), "status": "PASS" if len(failed)==0 else "WARN", "drivers": f"fallback tickers: {', '.join(failed) if failed else 'none'}"},
    ]
    return pd.DataFrame(rows)


def build_model_leaderboard(model_cmp, feat_imp=None):
    if model_cmp is None or model_cmp.empty:
        return pd.DataFrame([{
            "rank": 1,
            "model": "no_model_result",
            "mae": np.nan,
            "r2": np.nan,
            "leaderboard_score": np.nan,
            "governance_status": "WARN",
            "comment": "No model comparison results were produced.",
        }])
    lb = model_cmp.copy()
    for col in ["mae", "r2"]:
        if col not in lb.columns:
            lb[col] = np.nan
        lb[col] = pd.to_numeric(lb[col], errors="coerce")
    lb["mae_rank"] = lb["mae"].rank(ascending=True, method="dense")
    lb["r2_rank"] = lb["r2"].rank(ascending=False, method="dense")
    lb["leaderboard_score"] = 0.65 * (1 / lb["mae_rank"].replace(0, np.nan)) + 0.35 * (1 / lb["r2_rank"].replace(0, np.nan))
    lb["rank"] = lb["leaderboard_score"].rank(ascending=False, method="dense").astype("Int64")
    lb["governance_status"] = np.where(lb["mae"].notna() & lb["r2"].notna(), "PASS", "WARN")
    lb["comment"] = np.where(lb["governance_status"].eq("PASS"), "Model produced valid comparison metrics.", "Model failed or returned incomplete metrics.")
    cols = ["rank", "model", "mae", "r2", "leaderboard_score", "governance_status", "comment"]
    return lb[cols].sort_values(["rank", "model"], na_position="last").reset_index(drop=True)


def build_sws_portfolio_snapshot(valuation_df, backtest_perf, data_quality):
    rows = []
    if valuation_df is not None and not valuation_df.empty:
        rows.extend([
            {"metric": "universe_count", "value": len(valuation_df), "source": "valuation_ranking"},
            {"metric": "median_composite_score", "value": valuation_df.get("composite_score", pd.Series(dtype=float)).median(), "source": "valuation_ranking"},
            {"metric": "median_base_upside", "value": valuation_df.get("upside_base", pd.Series(dtype=float)).median(), "source": "scenario_framework"},
            {"metric": "average_volatility", "value": valuation_df.get("volatility", pd.Series(dtype=float)).mean(), "source": "price_metrics"},
        ])
    if backtest_perf is not None and not backtest_perf.empty:
        for col in ["annual_return_net", "annual_vol_net", "max_drawdown_net", "rolling_sharpe_latest"]:
            if col in backtest_perf.columns:
                rows.append({"metric": col, "value": backtest_perf[col].iloc[0], "source": "backtest_performance"})
    if data_quality is not None and not data_quality.empty:
        rows.append({"metric": "avg_fundamental_missing_ratio", "value": data_quality.get("fundamental_missing_ratio", pd.Series(dtype=float)).mean(), "source": "data_quality"})
    return pd.DataFrame(rows)


In [ ]:
# 3.3 Charts, outputs and interactive dashboard rendering

def _safe_numeric_column(frame, column, default=0.0):
    """Return a finite numeric series for Plotly fields that cannot receive NaN/inf."""
    if column in frame.columns:
        values = pd.to_numeric(frame[column], errors="coerce")
    else:
        values = pd.Series(default, index=frame.index, dtype="float64")
    values = values.replace([np.inf, -np.inf], np.nan)
    if values.notna().any():
        fallback = values.median()
        if pd.isna(fallback):
            fallback = default
        return values.fillna(fallback)
    return pd.Series(default, index=frame.index, dtype="float64")


def _clean_plotly_marker_size(series, default=24.0, min_size=12.0, max_size=72.0):
    """Scale marker sizes while removing NaN, inf and non-positive values."""
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    positive = values.where(values > 0)
    fallback = positive.median()
    if pd.isna(fallback) or fallback <= 0:
        fallback = 1.0
    filled = positive.fillna(fallback)
    if filled.nunique(dropna=True) <= 1:
        return pd.Series(default, index=values.index, dtype="float64")
    span = filled.max() - filled.min()
    scaled = min_size + ((filled - filled.min()) / span) * (max_size - min_size)
    return scaled.clip(lower=min_size, upper=max_size).astype("float64")


def _ticker_roles(tickers):
    roles = {}
    for ticker in tickers:
        if ticker == UNIVERSE["main_ticker"]:
            roles[ticker] = "main ticker"
        elif ticker == UNIVERSE["benchmark"]:
            roles[ticker] = "benchmark"
        elif ticker in UNIVERSE["peers"]:
            roles[ticker] = "peer"
        else:
            roles[ticker] = "watchlist"
    return roles


def _style_figure(fig, percent_y=False, percent_x=False, height=440):
    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(color=COLORS["text"]),
        title_font=dict(size=18, color=COLORS["primary"]),
        legend_title_text="",
        hovermode="x unified",
        height=height,
        margin=dict(l=48, r=30, t=72, b=48),
    )
    fig.update_xaxes(showgrid=True, gridcolor=COLORS["border"], zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor=COLORS["border"], zerolinecolor=COLORS["neutral"])
    if percent_y:
        fig.update_yaxes(tickformat=".1%")
    if percent_x:
        fig.update_xaxes(tickformat=".1%")
    return fig


def _build_rolling_charts(backtest_curve):
    charts = {}
    if backtest_curve is None or backtest_curve.empty or "date" not in backtest_curve.columns:
        return charts
    bt = backtest_curve.copy()
    bt["date"] = pd.to_datetime(bt["date"])
    bt = bt.sort_values("date").set_index("date")
    curve_cols = [c for c in bt.columns if pd.api.types.is_numeric_dtype(bt[c])]
    if not curve_cols:
        return charts
    returns = bt[curve_cols].pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    rolling_vol = returns.rolling(63, min_periods=20).std() * np.sqrt(252)
    rolling_sharpe = (returns.rolling(63, min_periods=20).mean() * 252) / (returns.rolling(63, min_periods=20).std() * np.sqrt(252))
    rolling_sharpe = rolling_sharpe.replace([np.inf, -np.inf], np.nan)
    if rolling_vol.notna().any().any():
        charts["rolling_volatility"] = _style_figure(px.line(rolling_vol.reset_index(), x="date", y=curve_cols, title="Rolling Volatility - 63 Trading Days", template=PLOTLY_TEMPLATE), percent_y=True)
    if rolling_sharpe.notna().any().any():
        charts["rolling_sharpe"] = _style_figure(px.line(rolling_sharpe.reset_index(), x="date", y=curve_cols, title="Rolling Sharpe - 63 Trading Days", template=PLOTLY_TEMPLATE))
    return charts


def build_charts(val, pivot, scenarios, sensitivity, backtest_curve, model_cmp, feat_imp, risk_table, data_quality=None, allocation=None, optimization=None, portfolio_scenarios=None, portfolio_factor_exposure=None, benchmark_comparison=None, risk_factor_exposures=None, factor_risk_summary=None, portfolio_engine_result=None, ml_time_series_result=None):
    charts = {}
    if not pivot.empty:
        clean_pivot = pivot.sort_index().replace([np.inf, -np.inf], np.nan).ffill().dropna(how="all")
        if not clean_pivot.empty:
            norm = clean_pivot / clean_pivot.iloc[0]
            drawdown = norm / norm.cummax() - 1
            charts["equity_curve"] = _style_figure(px.line(norm, title="Equity Curve / Normalized Price Index", template=PLOTLY_TEMPLATE, labels={"value": "Index", "date": "Date"}), height=460)
            charts["drawdown"] = _style_figure(px.line(drawdown, title="Drawdown by Ticker", template=PLOTLY_TEMPLATE, labels={"value": "Drawdown", "date": "Date"}), percent_y=True, height=460)
    if not val.empty:
        plot_val = val.copy()
        plot_val["role"] = plot_val["ticker"].map(_ticker_roles(plot_val["ticker"].tolist())).fillna("watchlist")
        for column, default in [
            ("volatility", 0.0),
            ("annual_return", 0.0),
            ("composite_score", 0.0),
            ("return_on_equity", 0.0),
            ("profit_margin", 0.0),
            ("trailing_pe", 0.0),
            ("price_to_sales", 0.0),
            ("market_cap", 1.0),
        ]:
            plot_val[column] = _safe_numeric_column(plot_val, column, default)
        plot_val["marker_size"] = _clean_plotly_marker_size(plot_val["market_cap"])
        score_cols = ["value_score", "quality_score", "momentum_score", "risk_score", "growth_score"]
        for column in score_cols:
            plot_val[column] = _safe_numeric_column(plot_val, column, 0.5)
        charts["ranking"] = _style_figure(px.bar(plot_val.sort_values("composite_score", ascending=True), x="composite_score", y="ticker", orientation="h", title="Final Ranking Score", template=PLOTLY_TEMPLATE, color="composite_score", color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_x=False)
        charts["risk_return"] = _style_figure(px.scatter(plot_val, x="volatility", y="annual_return", size="marker_size", color="composite_score", symbol="role", hover_name="ticker", hover_data=["role", "market_cap", "composite_score"], title="Risk-Return Scatter", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_x=True, percent_y=True)
        peer_scope = [UNIVERSE["main_ticker"]] + UNIVERSE["peers"]
        peer_val = plot_val[plot_val["ticker"].isin(peer_scope)].copy()
        if peer_val.empty:
            peer_val = plot_val.copy()
        charts["peer_comparison_scatter"] = _style_figure(px.scatter(peer_val, x="trailing_pe", y="return_on_equity", size="marker_size", color="composite_score", symbol="role", hover_name="ticker", hover_data=["profit_margin", "annual_return", "volatility"], title="Peer Comparison Scatter - Valuation vs ROE", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_y=True)
        multiples = ["trailing_pe", "forward_pe", "price_to_sales", "price_to_book", "ev_to_ebitda_proxy"]
        available_multiples = [c for c in multiples if c in plot_val.columns]
        if available_multiples:
            m = plot_val[["ticker"] + available_multiples].melt("ticker", var_name="metric", value_name="value")
            charts["valuation_multiples"] = _style_figure(px.bar(m, x="ticker", y="value", color="metric", barmode="group", title="Valuation Multiples", template=PLOTLY_TEMPLATE))
        s = plot_val[["ticker"] + score_cols].melt("ticker", var_name="score", value_name="value")
        charts["factor_exposure"] = _style_figure(px.bar(s, x="ticker", y="value", color="score", barmode="stack", title="Factor Exposure Chart", template=PLOTLY_TEMPLATE), percent_y=True)
        score_matrix = plot_val.set_index("ticker")[score_cols + ["composite_score"]]
        charts["score_heatmap"] = _style_figure(px.imshow(score_matrix, text_auto=".2f", title="Factor Score Heatmap", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), height=500)
    if scenarios is not None and not scenarios.empty:
        scen_targets = scenarios.melt("ticker", value_vars=["current_price", "bear_target", "base_target", "bull_target"], var_name="scenario", value_name="target_price")
        charts["valuation_scenario_chart"] = _style_figure(px.bar(scen_targets, x="ticker", y="target_price", color="scenario", barmode="group", title="Valuation Scenario Chart - Target Price Range", template=PLOTLY_TEMPLATE))
        scen_returns = scenarios.melt("ticker", value_vars=["bear_return", "base_return", "bull_return"], var_name="scenario", value_name="return")
        charts["scenario"] = _style_figure(px.bar(scen_returns, x="ticker", y="return", color="scenario", barmode="group", title="Scenario Returns", template=PLOTLY_TEMPLATE), percent_y=True)
    if sensitivity is not None and not sensitivity.empty:
        charts["sensitivity"] = _style_figure(px.density_heatmap(sensitivity, x="discount_rate_delta", y="multiple_rerating", z="target_proxy", histfunc="avg", title="Valuation Sensitivity Heatmap", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_x=True, percent_y=True)
    if allocation is not None and not allocation.empty:
        charts["allocation_pie"] = _style_figure(px.pie(allocation, names="ticker", values="target_weight_capped", title="Target Allocation Weights", template=PLOTLY_TEMPLATE), height=500)
    if optimization is not None and not optimization.empty:
        opt_melt = optimization.melt("ticker", value_vars=[c for c in ["optimized_weight", "target_weight_capped"] if c in optimization.columns], var_name="weight_type", value_name="weight")
        charts["optimization_weights"] = _style_figure(px.bar(opt_melt, x="ticker", y="weight", color="weight_type", barmode="group", title="Optimization - Target vs Optimized Weights", template=PLOTLY_TEMPLATE), percent_y=True)
        if {"volatility", "annual_return", "optimized_weight"}.issubset(optimization.columns):
            charts["efficient_frontier_proxy"] = _style_figure(px.scatter(optimization, x="volatility", y="annual_return", size="optimized_weight", color="risk_adjusted_score", hover_name="ticker", title="Efficient Frontier Proxy", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_x=True, percent_y=True)
    if portfolio_scenarios is not None and not portfolio_scenarios.empty:
        charts["portfolio_scenario"] = _style_figure(px.bar(portfolio_scenarios, x="scenario", y="portfolio_return", color="scenario", title="Portfolio Scenario Analysis", template=PLOTLY_TEMPLATE), percent_y=True)
    if portfolio_factor_exposure is not None and not portfolio_factor_exposure.empty:
        charts["portfolio_factor_exposure"] = _style_figure(px.bar(portfolio_factor_exposure, x="factor", y="weighted_exposure", color="weighted_exposure", title="Weighted Portfolio Factor Exposure", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_y=True)
    if risk_factor_exposures is not None and not risk_factor_exposures.empty:
        rf = risk_factor_exposures.copy()
        for col in ["beta", "correlation", "risk_contribution_proxy"]:
            rf[col] = _safe_numeric_column(rf, col, 0.0)
        charts["risk_factor_beta"] = _style_figure(px.bar(rf, x="factor", y="beta", color="family", title="Risk Factor Betas - Fama-French, FX, Commodities", template=PLOTLY_TEMPLATE))
        charts["risk_factor_contribution"] = _style_figure(px.bar(rf, x="factor", y="risk_contribution_proxy", color="family", title="Risk Factor Contribution Proxy", template=PLOTLY_TEMPLATE), percent_y=True)
    if factor_risk_summary is not None and not factor_risk_summary.empty:
        frs = factor_risk_summary.copy()
        frs["max_abs_risk_contribution"] = _safe_numeric_column(frs, "max_abs_risk_contribution", 0.0)
        charts["factor_family_risk"] = _style_figure(px.bar(frs, x="family", y="max_abs_risk_contribution", color="family", title="Risk Contribution by Factor Family", template=PLOTLY_TEMPLATE), percent_y=True)
    if benchmark_comparison is not None and not benchmark_comparison.empty:
        bc = benchmark_comparison.copy()
        bc["value"] = _safe_numeric_column(bc, "value", 0.0)
        charts["benchmark_comparison"] = _style_figure(px.bar(bc, x="metric", y="value", color="source", title="Benchmark Comparison Metrics", template=PLOTLY_TEMPLATE), percent_y=True)
    if portfolio_engine_result:
        weights = portfolio_engine_result.get("weights", pd.DataFrame())
        frontier = portfolio_engine_result.get("frontier", pd.DataFrame())
        backtest = portfolio_engine_result.get("backtest", pd.DataFrame())
        if weights is not None and not weights.empty:
            charts["engine_weights_bar"] = _style_figure(px.bar(weights, x="ticker", y="weight", color="weight", title="Portfolio Engine Allocation Weights", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_y=True)
            charts["engine_weights_treemap"] = _style_figure(px.treemap(weights, path=["ticker"], values="weight", color="weight", title="Portfolio Engine Allocation Treemap", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), height=520)
        if frontier is not None and not frontier.empty and {"volatility", "expected_return"}.issubset(frontier.columns):
            charts["engine_frontier"] = _style_figure(px.scatter(frontier, x="volatility", y="expected_return", color="sharpe" if "sharpe" in frontier.columns else None, symbol="portfolio" if "portfolio" in frontier.columns else None, title="Portfolio Engine Efficient Frontier", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["bg"], COLORS["accent"], COLORS["primary"]]), percent_x=True, percent_y=True)
        if backtest is not None and not backtest.empty and "date" in backtest.columns:
            y_cols = [c for c in ["portfolio_equity", "benchmark_equity"] if c in backtest.columns]
            if y_cols:
                charts["engine_backtest"] = _style_figure(px.line(backtest, x="date", y=y_cols, title="Portfolio Engine Backtest", template=PLOTLY_TEMPLATE), height=460)
    if ml_time_series_result:
        ts_result = ml_time_series_result.get("time_series", {}) or {}
        deep_result = ml_time_series_result.get("deep_stock", {}) or {}
        crypto_result = ml_time_series_result.get("crypto", {}) or {}
        ts_forecast = ts_result.get("forecast", pd.DataFrame())
        ts_regimes = ts_result.get("regimes", pd.DataFrame())
        ts_anomalies = ts_result.get("anomalies", pd.DataFrame())
        deep_signals = deep_result.get("signals", pd.DataFrame())
        crypto_signals = crypto_result.get("signals", pd.DataFrame())
        if ts_forecast is not None and not ts_forecast.empty and {"date", "ticker", "actual", "forecast"}.issubset(ts_forecast.columns):
            ts_plot = ts_forecast.copy().sort_values(["ticker", "date"]).groupby("ticker").tail(80)
            ts_long = ts_plot.melt(["date", "ticker"], value_vars=["actual", "forecast"], var_name="series", value_name="return")
            charts["ts_forecast_actual"] = _style_figure(px.line(ts_long, x="date", y="return", color="ticker", line_dash="series", title="Time Series Lab - Forecast vs Actual", template=PLOTLY_TEMPLATE), percent_y=True, height=500)
        if ts_regimes is not None and not ts_regimes.empty and {"date", "ticker", "rolling_vol"}.issubset(ts_regimes.columns):
            reg_plot = ts_regimes.copy().sort_values(["ticker", "date"]).groupby("ticker").tail(120)
            charts["ts_regime_detection"] = _style_figure(px.line(reg_plot, x="date", y="rolling_vol", color="ticker", line_dash="regime" if "regime" in reg_plot.columns else None, title="Regime Detection - Rolling Volatility", template=PLOTLY_TEMPLATE), percent_y=True, height=500)
        if ts_anomalies is not None and not ts_anomalies.empty and {"date", "ticker", "return", "z_score"}.issubset(ts_anomalies.columns):
            anom_plot = ts_anomalies.copy().tail(300)
            charts["ts_anomaly_flags"] = _style_figure(px.scatter(anom_plot, x="date", y="return", color="ticker", size=anom_plot["z_score"].abs().clip(lower=0.0), hover_data=["z_score"], title="Anomaly Flags - Return Z-Scores", template=PLOTLY_TEMPLATE), percent_y=True, height=500)
        if deep_signals is not None and not deep_signals.empty and {"ticker", "ml_score"}.issubset(deep_signals.columns):
            ds_plot = deep_signals.copy().sort_values("ml_score", ascending=False)
            charts["deep_stock_scores"] = _style_figure(px.bar(ds_plot, x="ticker", y="ml_score", color="signal" if "signal" in ds_plot.columns else "ml_score", title="Stock Picking Deep Models - ML Score Ranking", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["danger"], COLORS["bg"], COLORS["primary"]] if "signal" not in ds_plot.columns else None), percent_y=True)
        if crypto_signals is not None and not crypto_signals.empty and {"ticker", "forecast_return"}.issubset(crypto_signals.columns):
            charts["crypto_ml_signals"] = _style_figure(px.bar(crypto_signals.sort_values("forecast_return", ascending=False), x="ticker", y="forecast_return", color="signal" if "signal" in crypto_signals.columns else "forecast_return", title="Crypto ML Signals - Forecast Return by Asset", template=PLOTLY_TEMPLATE), percent_y=True)
    if backtest_curve is not None and not backtest_curve.empty:
        y_cols = [c for c in backtest_curve.columns if c != "date"]
        charts["backtest"] = _style_figure(px.line(backtest_curve, x="date", y=y_cols, title="Backtest Lab - Strategy vs Benchmark", template=PLOTLY_TEMPLATE), height=460)
        charts.update(_build_rolling_charts(backtest_curve))
    if model_cmp is not None and not model_cmp.empty and "model" in model_cmp.columns:
        model_plot = model_cmp.copy()
        model_plot["mae"] = _safe_numeric_column(model_plot, "mae", 0.0)
        charts["model_comparison"] = _style_figure(px.bar(model_plot, x="model", y="mae", color="model", title="Model Comparison Chart - MAE", template=PLOTLY_TEMPLATE))
    if feat_imp is not None and not feat_imp.empty:
        charts["feature_importance"] = _style_figure(px.bar(feat_imp, x="importance", y="feature", orientation="h", title="Explainability - Feature Contribution", template=PLOTLY_TEMPLATE))
    if risk_table is not None and not risk_table.empty:
        risk_plot = risk_table.copy()
        risk_plot["score"] = _safe_numeric_column(risk_plot, "score", 0.0)
        charts["risk_dashboard"] = _style_figure(px.bar(risk_plot, x="risk_family", y="score", color="status", title="Risk Dashboard by Family", template=PLOTLY_TEMPLATE))
    if data_quality is not None and not data_quality.empty:
        dq = data_quality.copy()
        dq["price_rows"] = _safe_numeric_column(dq, "price_rows", 0.0)
        dq["fundamental_missing_ratio"] = _safe_numeric_column(dq, "fundamental_missing_ratio", 1.0).clip(0, 1)
        charts["data_coverage"] = _style_figure(px.bar(dq, x="ticker", y="price_rows", color="fundamental_missing_ratio", hover_data=[c for c in ["first_date", "last_date", "data_status", "source"] if c in dq.columns], title="Data Coverage Chart", template=PLOTLY_TEMPLATE, color_continuous_scale=[COLORS["primary"], COLORS["accent"], COLORS["danger"]]), percent_y=False)
    if not val.empty:
        try:
            sws = table_sws_scorecard(val)
            axes = ["value_axis", "future_axis", "past_axis", "health_axis", "income_axis"]
            focus = sws[sws["ticker"].eq(UNIVERSE.get("main_ticker"))].head(1)
            if focus.empty:
                focus = sws.head(1)
            radar = focus.melt("ticker", value_vars=axes, var_name="axis", value_name="score")
            charts["sws_snowflake"] = _style_figure(px.line_polar(radar, r="score", theta="axis", line_close=True, color="ticker", range_r=[0, 6], title="SWS-Style Portfolio Snowflake", template=PLOTLY_TEMPLATE), height=500)
        except Exception as exc:
            logger.warning("SWS snowflake chart unavailable: %s", exc)
    return charts


def _fmt_styler(df, percent_cols=None, number_cols=None, gradient_cols=None):
    percent_cols = [c for c in (percent_cols or []) if c in df.columns]
    number_cols = [c for c in (number_cols or []) if c in df.columns]
    gradient_cols = [c for c in (gradient_cols or []) if c in df.columns]
    fmt = {c: "{:.2%}" for c in percent_cols}
    fmt.update({c: "{:,.2f}" for c in number_cols})
    styler = df.style.format(fmt)
    if gradient_cols:
        styler = styler.background_gradient(subset=gradient_cols, cmap="YlGnBu")
    return styler



def build_config_snapshot():
    return {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "project_root": str(PROJECT_ROOT),
        "db_base": str(DB_BASE),
        "data_path": str(DATA_PATH),
        "masterrequest": MASTERREQUEST,
        "experiment": EXPERIMENT,
        "universe": UNIVERSE,
        "portfolio_config": PORTFOLIO_CONFIG,
        "valuation_config": VALUATION_CONFIG,
        "ml_config": ML_CONFIG,
        "risk_config": RISK_CONFIG,
        "governance_config": GOVERNANCE_CONFIG,
        "reporting_config": REPORTING_CONFIG,
        "project_database_config": PROJECT_DATABASE_CONFIG,
        "model_depth_preset": MODEL_DEPTH_PRESETS.get(ML_CONFIG.get("model_depth", "standard"), {}),
    }


def save_config_snapshot(snapshot, output_dir=None):
    output_dir = output_dir or SNAPSHOT_DIR
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"config_snapshot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    path.write_text(json.dumps(snapshot, indent=2, default=str), encoding="utf-8")
    return path


def export_static_dashboard(bundle, output_paths):
    """Export navigable dashboard using the portfolio dashboard module.

    This mirrors the Company Valuation notebook pattern: notebook variables and
    result bundles remain canonical, while final HTML/report generation lives in
    a reusable module.
    """
    module_roots = [PROJECT_ROOT / "portfolio_analysis" / "src", PROJECT_ROOT / "company_valuation" / "src"]
    for module_root in module_roots:
        if str(module_root) not in sys.path:
            sys.path.insert(0, str(module_root))
    try:
        from portfolio_dashboard import export_dashboard_and_report
        namespace = dict(globals())
        namespace["RESEARCH_RESULT"] = {"tables": bundle.get("tables", {}), "charts": bundle.get("charts", {}), "outputs": output_paths}
        artifacts = export_dashboard_and_report(namespace)
        output_paths["research_report"] = str(artifacts.report_path)
        return artifacts.dashboard_path
    except Exception as exc:
        logger.warning("Module-backed portfolio dashboard export failed: %s", exc)
        DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
        path = DASHBOARD_DIR / f"research_platform_dashboard_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
        tables = bundle.get("tables", {})
        charts = output_paths.get("charts", {})
        kpi = tables.get("sws_portfolio_snapshot", pd.DataFrame())
        kpi_html = kpi.to_html(index=False, classes="table") if kpi is not None and not kpi.empty else "<p>No KPI snapshot available.</p>"
        chart_links = "".join([f"<li><a href='{chart_path}'>{name}</a></li>" for name, chart_path in charts.items()])
        html = f"""
        <html><head><title>Investment Research Platform Pro</title></head><body>
        <h1>Investment Research Platform Pro</h1>
        <h2>KPI Snapshot</h2>{kpi_html}
        <h2>Interactive Plotly Charts</h2><ul>{chart_links}</ul>
        <p>Output root: {OUTPUT_ROOT}</p>
        </body></html>
        """
        path.write_text(html, encoding="utf-8")
        return path

def save_outputs(bundle):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    paths = {}
    for name, obj in bundle["tables"].items():
        if isinstance(obj, pd.DataFrame):
            path = TABLES_DIR / f"{name}_{ts}.csv"
            obj.to_csv(path, index=False)
        else:
            path = TABLES_DIR / f"{name}_{ts}.json"
            try:
                path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
            except TypeError:
                path.write_text(json.dumps(str(obj), indent=2), encoding="utf-8")
        paths[name] = str(path)
    chart_paths = {}
    for name, fig in bundle["charts"].items():
        path = FIGURES_DIR / f"{name}_{ts}.html"
        fig.write_html(str(path))
        chart_paths[name] = str(path)
    config_path = CONFIG_DIR / f"masterrequest_{ts}.json"
    config_path.write_text(json.dumps(MASTERREQUEST, indent=2, default=str), encoding="utf-8")
    report_path = REPORTS_DIR / f"investment_research_platform_pro_{ts}.md"
    report_path.write_text(f"""# Investment Research Platform Pro Report

Generated: {datetime.now().isoformat(timespec='seconds')}

Main ticker: {UNIVERSE['main_ticker']}
Benchmark: {UNIVERSE['benchmark']}
Peer mode: {UNIVERSE['peer_mode']}
Peers: {', '.join(UNIVERSE['peers'])}
Universe: {', '.join(UNIVERSE['all_tickers'])}

Tables:
{json.dumps({k: v for k, v in paths.items()}, indent=2)}

Charts:
{json.dumps(chart_paths, indent=2)}
""", encoding="utf-8")
    paths["charts"] = chart_paths
    paths["masterrequest"] = str(config_path)
    snapshot_path = save_config_snapshot(build_config_snapshot())
    paths["config_snapshot"] = str(snapshot_path)
    paths["markdown_report"] = str(report_path)
    dashboard_path = export_static_dashboard(bundle, paths)
    paths["static_dashboard"] = str(dashboard_path)
    return paths


def _safe_pct(value):
    return "n/a" if pd.isna(value) else f"{value:.2%}"


def _safe_num(value, digits=2):
    return "n/a" if pd.isna(value) else f"{value:,.{digits}f}"


def _card_grid(cards):
    html = ["<div class='ir-grid-4'>"]
    for title, value, subtitle in cards:
        html.append(f"<div class='ir-card'><div class='ir-card-title'>{title}</div><div class='ir-card-value'>{value}</div><div style='color:{COLORS['muted']};font-size:12px;margin-top:6px'>{subtitle}</div></div>")
    html.append("</div>")
    display(HTML("".join(html)))


def _display_table(df, percent_cols=None, number_cols=None, gradient_cols=None, rows=50):
    if df is None or df.empty:
        display(HTML("<div class='ir-status-warn'>No data available for this view.</div>"))
        return
    display(_fmt_styler(df.head(rows), percent_cols=percent_cols, number_cols=number_cols, gradient_cols=gradient_cols))


def _show_chart(charts, key):
    fig = charts.get(key)
    if fig is None:
        display(HTML(f"<div class='ir-status-warn'>Chart not available: {key}</div>"))
        return
    fig.show()


def _show_chart_group(charts, keys):
    for key in keys:
        if key in charts:
            _show_chart(charts, key)


def _chart_inventory(charts):
    rows = []
    for name, fig in charts.items():
        title = ""
        try:
            title = fig.layout.title.text or ""
        except Exception:
            title = ""
        rows.append({"chart_key": name, "title": title, "interactive": True})
    return pd.DataFrame(rows)




def build_research_section_map():
    rows = []
    enabled = set(EXPERIMENT.get("enabled_sections", RESEARCH_SECTION_CATALOG.keys()))
    detail = EXPERIMENT.get("detail_level", "institutional")
    for key, meta in RESEARCH_SECTION_CATALOG.items():
        chapter_path = PROJECT_ROOT / meta["chapter"]
        rows.append({
            "enabled": key in enabled,
            "section": key,
            "title": meta["title"],
            "project_chapter": meta["chapter"],
            "chapter_exists": chapter_path.exists(),
            "formula": meta["formula"],
            "outputs": ", ".join(meta["outputs"]),
            "detail_level": detail,
        })
    return pd.DataFrame(rows)


def build_data_integration_plan():
    selected = EXPERIMENT.get("enabled_data_layers", list(DATA_INTEGRATION_LAYERS.keys()))
    rows = []
    for key, meta in DATA_INTEGRATION_LAYERS.items():
        raw_path = meta["path"]
        exists = Path(raw_path).exists() if isinstance(raw_path, str) and raw_path.startswith("/") else key in ["yfinance", "api_registry", "synthetic_fallback"]
        rows.append({
            "enabled": key in selected,
            "layer": key,
            "label": meta["label"],
            "priority": meta["priority"],
            "path_or_source": raw_path,
            "available": bool(exists),
            "refresh_policy": "weekly if missing/stale" if EXPERIMENT.get("weekly_refresh_missing", True) else "manual",
            "max_staleness_days": EXPERIMENT.get("max_data_staleness_days", 7),
        })
    return pd.DataFrame(rows).sort_values(["enabled", "priority"], ascending=[False, True])


def build_model_family_plan():
    enabled = set(ML_CONFIG.get("model_families", MODEL_FAMILY_CATALOG.keys()))
    rows = []
    for family, members in MODEL_FAMILY_CATALOG.items():
        rows.append({
            "enabled": family in enabled,
            "family": family,
            "models_or_methods": ", ".join(members),
            "implementation_status": "active" if family in ["linear", "machine_learning", "gradient_boosting", "time_series", "factor"] else "project integration / governed proxy",
            "project_reference": next((x["chapter"] for x in PROJECT_CHAPTER_LINKS if family.replace("_", " ").split()[0] in x["topic"] or family in x["topic"].replace(" ", "_")), "see project chapters"),
        })
    return pd.DataFrame(rows)


def build_project_chapter_table():
    rows = []
    for item in PROJECT_CHAPTER_LINKS:
        path = Path(item["path"])
        rows.append({**item, "exists": path.exists(), "notebooks": len(list(path.rglob("*.ipynb"))) if path.exists() else 0})
    return pd.DataFrame(rows)


def build_portfolio_methodology_html():
    return """
    <div class='ir-card'>
      <div class='ir-card-title'>Methodology</div>
      <details open><summary><b>Composite score</b></summary><p>Composite = 0.22 Value + 0.28 Quality + 0.22 Momentum + 0.18 Risk + 0.10 Growth.</p></details>
      <details><summary><b>SWS-style axes</b></summary><p>Value, Future, Past, Health and Income are transparent 0-6 percentile proxies built from valuation, upside, return, leverage/risk and cash-yield fields.</p></details>
      <details><summary><b>Scenario valuation</b></summary><p>Bear/base/bull targets use analyst targets when available, otherwise price, return and volatility proxies.</p></details>
      <details><summary><b>Risk and backtest</b></summary><p>Volatility, drawdown, rolling Sharpe and net return are computed from daily returns with transaction-cost drag.</p></details>
      <details><summary><b>Governance</b></summary><p>Every run exports MASTERREQUEST, config snapshot, source summary, feature missingness, robustness checks and static dashboard HTML.</p></details>
      <details><summary><b>Model families</b></summary><p>Linear models, tree machine learning, gradient boosting, time-series diagnostics and factor models are active. Unsupervised, deep learning and recurrent-net views are governed project integrations that link to the relevant chapters and use proxies unless trained artifacts are available.</p></details>
    </div>
    """


def build_warning_panel_html(data_quality=None, robustness_checks=None):
    dq = data_quality if data_quality is not None else pd.DataFrame()
    rb = robustness_checks if robustness_checks is not None else pd.DataFrame()
    missing = dq.get("fundamental_missing_ratio", pd.Series(dtype=float)).mean() if not dq.empty else np.nan
    missing_txt = "n/a" if pd.isna(missing) else f"{missing:.2%}"
    warn_count = int((rb.get("status", pd.Series(dtype=str)) == "WARN").sum()) if not rb.empty else 0
    return f"""
    <div style='margin-top: 14px; padding: 14px; background-color: #fff7e6; border-left: 5px solid {COLORS.get('danger', '#c0392b')}; border-radius: 8px;'>
      <h3 style='margin-top:0;color:{COLORS.get('danger', '#c0392b')}'>Best Practices & Data Warnings</h3>
      <ul>
        <li><b>Data coverage:</b> average fundamental missingness is {missing_txt}.</li>
        <li><b>Robustness:</b> {warn_count} warning checks are active.</li>
        <li><b>Interpretation:</b> SWS-style axes are transparent research proxies, not official SWS methodology.</li>
        <li><b>Persistence:</b> heavy datasets remain outside git under DB_BASE / project data cache.</li>
      </ul>
    </div>
    """

def render_dashboard(result):
    t, charts, outputs = result["tables"], result["charts"], result["outputs"]
    val = t["valuation_ranking"].copy()
    company = t["company_overview"].copy()
    main = UNIVERSE["main_ticker"]
    tickers = val["ticker"].tolist() if not val.empty and "ticker" in val.columns else [main]
    main_row_df = val[val["ticker"].eq(main)] if not val.empty and "ticker" in val.columns else pd.DataFrame()
    main_row = main_row_df.iloc[0] if not main_row_df.empty else (company.iloc[0] if not company.empty else pd.Series(dtype=object))
    top = val.iloc[0]["ticker"] if not val.empty and "ticker" in val.columns else "n/a"
    base_upside = main_row.get("upside_base", np.nan)
    score = main_row.get("composite_score", np.nan)
    risk_status = t["risk_dashboard"]["status"].value_counts().to_dict() if not t["risk_dashboard"].empty and "status" in t["risk_dashboard"].columns else {}

    display(HTML("""
    <div class='ir-dashboard-hero'>
      <h2>Interactive Results Dashboard</h2>
      <p><b>Visual Analytics Dashboard:</b> KPI, tables, diagnostics and Plotly charts are grouped into eight result views so the analysis reads like a platform, not scattered notebook output.</p>
      <div class='ir-dashboard-navhint'><div>Executive</div><div>Allocation</div><div>Performance</div><div>Benchmark</div><div>Factors</div><div>Optimization</div><div>Scenarios</div><div>Outputs</div></div>
    </div>
    """))
    _card_grid([
        ("Main ticker", main, f"Benchmark {UNIVERSE['benchmark']}"),
        ("Top ranked", top, f"Universe {len(UNIVERSE['all_tickers'])} tickers"),
        ("Base upside", _safe_pct(base_upside), "Base scenario vs current price"),
        ("Composite score", _safe_num(score, 2), f"Risk warnings {risk_status.get('WARN', 0)}"),
    ])

    selected_value = main if main in tickers else tickers[0]
    selected_ticker_w = widgets.Dropdown(options=tickers, value=selected_value, description="Focus ticker", style={"description_width": "92px"}, layout=widgets.Layout(width="280px"))
    min_score_w = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description="Min score", style={"description_width": "92px"}, layout=widgets.Layout(width="320px"))
    chart_options = list(charts.keys()) or ["no_charts_available"]
    chart_selector_w = widgets.Dropdown(options=chart_options, value=chart_options[0], description="Spotlight", style={"description_width": "92px"}, layout=widgets.Layout(width="360px"))
    refresh_view_btn = widgets.Button(description="Refresh", icon="refresh", button_style="info", layout=widgets.Layout(width="120px"))

    view_names = ["Executive Summary", "Allocation", "Performance", "Benchmark", "Factor Exposures", "Optimization", "Engine Lab", "ML & Time Series", "Scenarios", "Risk", "Backtest", "Diagnostics", "Outputs"]
    view_tabs = widgets.Tab(children=[widgets.Output() for _ in view_names])
    for i, name in enumerate(view_names):
        view_tabs.set_title(i, name)
    tab = {name: view_tabs.children[i] for i, name in enumerate(view_names)}

    def _filtered_val():
        if val.empty or "composite_score" not in val.columns:
            return val
        return val[val["composite_score"].fillna(0) >= min_score_w.value].copy()

    def _role_table():
        return pd.DataFrame({
            "ticker": UNIVERSE["all_tickers"],
            "role": ["main ticker" if x == main else "benchmark" if x == UNIVERSE["benchmark"] else "peer" if x in UNIVERSE["peers"] else "watchlist" for x in UNIVERSE["all_tickers"]],
        })

    def _panel_title(title, copy=""):
        display(HTML(f"""
        <div class='ir-chart-panel'>
          <div class='ir-chart-title'>{title}</div>
          <div class='ir-chart-copy'>{copy}</div>
        </div>
        """))

    def _chart_block(key, title, copy=""):
        if key not in charts:
            display(HTML(f"<div class='ir-status-warn'>Chart not available: {key}</div>"))
            return
        display(HTML(f"""
        <div class='ir-chart-panel'>
          <div class='ir-chart-title'>{title}</div>
          <div class='ir-chart-copy'>{copy}</div>
        </div>
        """))
        display(HTML("<div class='ir-chart-stage'>"))
        charts[key].show()
        display(HTML("</div>"))

    def _chart_pack(items):
        for key, title, copy in items:
            _chart_block(key, title, copy)

    def _diagnostics_snapshot():
        diag = t.get("diagnostics", pd.DataFrame())
        if diag is None or diag.empty or "status" not in diag.columns:
            return pd.DataFrame()
        return diag.groupby("status", dropna=False).size().reset_index(name="checks")

    def _render_all(_=None):
        focus = selected_ticker_w.value
        focus_row = val[val["ticker"].eq(focus)].iloc[0] if not val.empty and focus in val["ticker"].tolist() else pd.Series(dtype=object)
        filtered = _filtered_val()

        for child in view_tabs.children:
            child.clear_output(wait=True)

        with tab["Executive Summary"]:
            display(HTML(f"""
            <div class='ir-card'>
              <div class='ir-card-title'>Executive summary</div>
              <p><b>{main}</b> is analyzed against <b>{UNIVERSE['benchmark']}</b> using <b>{UNIVERSE['peer_mode']}</b> peer logic. The top ranked security is <b>{top}</b>. Current focus is <b>{focus}</b>, with composite score <b>{_safe_num(focus_row.get('composite_score', np.nan), 2)}</b> and base upside <b>{_safe_pct(focus_row.get('upside_base', np.nan))}</b>.</p>
            </div>
            """))
            _card_grid([
                ("Universe", str(len(UNIVERSE["all_tickers"])), "main + benchmark + peers + watchlist"),
                ("Peers", str(len(UNIVERSE["peers"])), UNIVERSE["peer_mode"]),
                ("Model depth", ML_CONFIG.get("model_depth", "n/a"), ML_CONFIG.get("depth_description", "")[:44]),
                ("Data source", UNIVERSE.get("data_source", "auto"), "project DB -> yfinance -> fallback"),
            ])
            _panel_title("Configuration and diagnostics snapshot", "The dashboard keeps the applied MASTERREQUEST separate from results. Diagnostics are summarized here and detailed in the Outputs tab.")
            _display_table(_role_table())
            _display_table(_diagnostics_snapshot())
            _panel_title("Data quality snapshot", "Coverage and source diagnostics are surfaced before the charts so interpretation is tied to data reliability.")
            _display_table(t["data_quality"], percent_cols=["fundamental_missing_ratio"], number_cols=["price_rows", "missing_price"], rows=12)
            _chart_block("data_coverage", "Data coverage chart", "Price-row coverage and fundamental missingness by ticker.")
            _chart_block("ranking", "Ranking scorecard", "Composite score across the filtered universe.")
            _chart_block(chart_selector_w.value, "Spotlight interactive chart", "Use the Spotlight selector above to bring any chart into the executive view.")

        with tab["Performance"]:
            _panel_title("Performance analytics", "Net strategy performance, benchmark comparison, rolling volatility, rolling Sharpe and drawdown are grouped here as the portfolio performance cockpit.")
            _display_table(t.get("performance_summary", t.get("backtest_performance", pd.DataFrame())), percent_cols=["annual_return_net", "annual_vol_net", "max_drawdown_net", "rolling_vol_latest", "rolling_sharpe_latest", "estimated_monthly_cost_drag"])
            _display_table(t.get("benchmark_comparison", pd.DataFrame()), percent_cols=["value"], rows=80)
            _chart_pack([
                ("backtest", "Strategy vs benchmark", "Top-ranked allocation strategy net/gross versus benchmark."),
                ("benchmark_comparison", "Benchmark comparison", "Return, risk, drawdown and active-return metrics."),
                ("drawdown", "Drawdown", "Peak-to-trough loss paths by ticker."),
                ("rolling_volatility", "Rolling volatility", "63-day annualized volatility."),
                ("rolling_sharpe", "Rolling Sharpe", "63-day rolling risk-adjusted return."),
            ])

        with tab["Allocation"]:
            _panel_title("Allocation analytics", "Target weights translate company valuation-style scoring into portfolio allocation, active weights and risk-return context.")
            _display_table(t.get("portfolio_allocation", pd.DataFrame()), percent_cols=["target_weight_capped", "current_weight_proxy", "active_weight", "annual_return", "volatility", "max_drawdown", "upside_base"], number_cols=["composite_score"], gradient_cols=["target_weight_capped", "composite_score"], rows=100)
            _display_table(filtered[[c for c in ["ticker", "rank", "composite_score", "annual_return", "volatility", "max_drawdown", "upside_base"] if c in filtered.columns]], percent_cols=["annual_return", "volatility", "max_drawdown", "upside_base"], number_cols=["composite_score"], gradient_cols=["composite_score"])
            _chart_pack([
                ("allocation_pie", "Allocation weights", "Target allocation by ticker."),
                ("equity_curve", "Equity curve", "Normalized price path for portfolio candidates and benchmark."),
                ("drawdown", "Drawdown", "Peak-to-trough loss paths by ticker."),
                ("risk_return", "Risk-return scatter", "Annualized return versus annualized volatility, sized by market cap proxy."),
                ("factor_exposure", "Factor exposure", "Stacked factor score composition by ticker."),
                ("sws_snowflake", "SWS-style snowflake", "Value, future, past, health and income axes on a 0-6 scale."),
            ])

        with tab["Benchmark"]:
            _panel_title("Benchmark and peer comparison", "Portfolio performance is compared against the benchmark, then peer intelligence explains cross-sectional positioning.")
            _display_table(t.get("benchmark_comparison", pd.DataFrame()), percent_cols=["value"], rows=80)
            _chart_block("benchmark_comparison", "Benchmark comparison chart", "Strategy, benchmark and active return/risk metrics.")
            _display_table(t["peer_comparison"], percent_cols=["profit_margin", "return_on_equity", "annual_return", "volatility", "max_drawdown"], number_cols=["market_cap", "revenue", "trailing_pe", "forward_pe", "price_to_sales", "price_to_book", "ev_to_ebitda_proxy"], gradient_cols=["composite_score", "quality_score", "value_score"])
            _display_table(t["peer_similarity"], number_cols=["similarity_distance", "similarity_score"])
            _chart_pack([
                ("peer_comparison_scatter", "Peer comparison scatter", "Valuation multiple versus return on equity with role and score encoding."),
                ("score_heatmap", "Peer score heatmap", "Factor score intensity across the peer universe."),
            ])

        with tab["Optimization"]:
            _panel_title("Optimization", "Risk-adjusted optimized weights are shown alongside score-based target weights. This is a governed optimization proxy, not a hidden black box.")
            _display_table(t.get("optimization_summary", pd.DataFrame()), percent_cols=["optimized_weight", "target_weight_capped", "weight_change_vs_target", "annual_return", "volatility"], number_cols=["risk_adjusted_score", "composite_score"], gradient_cols=["optimized_weight", "risk_adjusted_score"], rows=100)
            _chart_pack([
                ("optimization_weights", "Optimized vs target weights", "Compare optimized allocation with score-based target allocation."),
                ("efficient_frontier_proxy", "Efficient frontier proxy", "Return/volatility positioning sized by optimized weight."),
            ])

        with tab["Engine Lab"]:
            _panel_title("Portfolio Optimization & Allocation Lab", "PyPortfolioOpt, Riskfolio-Lib and cvxportfolio-style allocation share one unified API and dashboard output.")
            engine_result = t.get("portfolio_engine_result", {})
            _display_table(t.get("portfolio_engine_weights", pd.DataFrame()), percent_cols=["weight"], rows=100)
            _display_table(t.get("portfolio_engine_frontier", pd.DataFrame()), percent_cols=["expected_return", "volatility", "sharpe"], rows=100)
            _display_table(t.get("portfolio_engine_metrics", pd.DataFrame()), percent_cols=["value"], rows=50)
            _display_table(t.get("portfolio_engine_diagnostics", pd.DataFrame()), rows=50)
            _chart_pack([
                ("engine_weights_bar", "Engine allocation weights", "Optimized weights from the selected portfolio engine."),
                ("engine_weights_treemap", "Engine allocation treemap", "Allocation concentration and relative exposure."),
                ("engine_frontier", "Efficient frontier", "Expected return versus volatility with Sharpe coloring when available."),
                ("engine_backtest", "Engine backtest", "Static-weight or multi-period style backtest from the selected engine."),
            ])

        with tab["ML & Time Series"]:
            _panel_title("ML & Time Series Engine", "Forecasting, anomaly flags, regime detection, stock-picking sequence models and crypto signals are grouped in one governed research lab.")
            _display_table(t.get("time_series_diagnostics", pd.DataFrame()), rows=80)
            _display_table(t.get("time_series_forecast", pd.DataFrame()).tail(40), percent_cols=["actual", "forecast", "error"], rows=80)
            _display_table(t.get("time_series_regimes", pd.DataFrame()).tail(40), percent_cols=["rolling_return", "rolling_vol"], rows=80)
            _display_table(t.get("time_series_anomalies", pd.DataFrame()).tail(40), percent_cols=["return"], number_cols=["z_score"], rows=80)
            _display_table(t.get("deep_stock_signals", pd.DataFrame()), percent_cols=["forecast_return", "ml_score"], rows=100)
            _display_table(t.get("deep_stock_diagnostics", pd.DataFrame()), rows=80)
            _display_table(t.get("crypto_signals", pd.DataFrame()), percent_cols=["forecast_return"], rows=100)
            _display_table(t.get("crypto_diagnostics", pd.DataFrame()), rows=80)
            _chart_pack([
                ("ts_forecast_actual", "Forecast vs actual", "Predicted versus realized returns from the Time Series Lab."),
                ("ts_regime_detection", "Regime detection", "Rolling volatility regimes for the selected universe."),
                ("ts_anomaly_flags", "Anomaly flags", "Return outliers by ticker and z-score magnitude."),
                ("deep_stock_scores", "Deep stock ML score ranking", "Ticker ranking from sequence-model or light fallback forecasts."),
                ("crypto_ml_signals", "Crypto ML signals", "Buy/hold/sell signal layer when crypto price data is supplied."),
            ])

        with tab["Scenarios"]:
            _panel_title("Scenario analysis", "Portfolio-level bear/base/bull outcomes are built from weighted security scenarios and valuation sensitivity.")
            _display_table(t.get("portfolio_scenarios", pd.DataFrame()), percent_cols=["portfolio_return", "worst_security_return", "best_security_return", "coverage_weight"], rows=40)
            _chart_block("portfolio_scenario", "Portfolio scenario chart", "Weighted bear, base and bull portfolio returns.")
            _display_table(t["scenario_framework"], percent_cols=["bear_return", "base_return", "bull_return", "valuation_risk"], number_cols=["current_price", "bear_target", "base_target", "bull_target", "asymmetry"], gradient_cols=["base_return", "asymmetry"])
            _chart_pack([
                ("valuation_scenario_chart", "Valuation scenario chart", "Current, bear, base and bull target prices by ticker."),
                ("scenario", "Scenario return chart", "Upside and downside returns implied by the scenario framework."),
                ("valuation_multiples", "Valuation multiples", "Relative multiples for the selected universe."),
                ("sensitivity", "Sensitivity heatmap", "Target proxy across discount-rate and multiple re-rating assumptions."),
            ])

        with tab["Risk"]:
            _panel_title("Risk dashboard", "Market, factor, valuation, accounting/governance and data/model risk views.")
            _display_table(t["risk_dashboard"])
            _display_table(t.get("factor_risk_summary", pd.DataFrame()), percent_cols=["avg_abs_beta", "max_abs_risk_contribution"], rows=50)
            _chart_pack([
                ("factor_family_risk", "Factor family risk", "Aggregated Fama-French, rates, FX and commodity factor risk."),
                ("risk_dashboard", "Risk family dashboard", "Risk score and status by risk family."),
                ("risk_return", "Risk-return scatter", "Cross-sectional return/volatility trade-off."),
                ("drawdown", "Drawdown", "Loss path and recovery context."),
                ("rolling_volatility", "Rolling volatility", "63-day annualized volatility for strategy, net strategy and benchmark."),
                ("rolling_sharpe", "Rolling Sharpe", "63-day rolling risk-adjusted return."),
            ])

        with tab["Factor Exposures"]:
            _panel_title("Factor exposures and model outputs", "Weighted portfolio factor exposure, security factor map, model factory outputs and explainability diagnostics.")
            _display_table(t.get("portfolio_factor_exposure", pd.DataFrame()), percent_cols=["weighted_exposure", "min_security_score", "max_security_score"], rows=50)
            _display_table(t.get("risk_factor_exposures", pd.DataFrame()), percent_cols=["beta", "correlation", "risk_contribution_proxy"], number_cols=["t_stat_proxy", "observations"], gradient_cols=["risk_contribution_proxy"], rows=100)
            _display_table(t.get("factor_risk_summary", pd.DataFrame()), percent_cols=["avg_abs_beta", "max_abs_risk_contribution"], rows=50)
            _chart_block("portfolio_factor_exposure", "Weighted internal factor exposure", "Portfolio-level value, quality, momentum, risk, growth and size exposure.")
            _chart_block("risk_factor_beta", "Risk factor betas", "Exposure to Fama-French-style, currency, FX-rate and commodity proxies.")
            _chart_block("risk_factor_contribution", "Risk factor contribution", "Volatility-normalized contribution proxy by factor.")
            _chart_block("factor_family_risk", "Factor family risk", "Aggregated risk by equity, rates, FX and commodity families.")
            _display_table(t.get("sws_scorecard", pd.DataFrame()), number_cols=["value_axis", "future_axis", "past_axis", "health_axis", "income_axis", "sws_total_score"], gradient_cols=["sws_total_score"])
            _display_table(build_research_section_map(), rows=100)
            _display_table(build_model_family_plan(), rows=100)
            _display_table(build_project_chapter_table(), rows=100)
            _display_table(t["factor_signals"], number_cols=["value_score", "quality_score", "momentum_score", "risk_score", "growth_score", "composite_score"], gradient_cols=["composite_score", "quality_score"])
            _display_table(t.get("model_leaderboard", pd.DataFrame()), number_cols=["mae", "r2", "leaderboard_score"], gradient_cols=["leaderboard_score"])
            _display_table(t["model_comparison"])
            _display_table(t["feature_importance"])
            _chart_pack([
                ("model_comparison", "Model comparison chart", "Model MAE comparison from the canonical model factory."),
                ("feature_importance", "Feature importance", "Explainability contribution by factor input."),
                ("factor_exposure", "Factor exposure", "Factor block composition for the selected universe."),
            ])

        with tab["Backtest"]:
            _panel_title("Backtest", "Net strategy curve, benchmark comparison, rolling volatility and rolling Sharpe.")
            _display_table(t["backtest_performance"], percent_cols=["annual_return_net", "annual_vol_net", "max_drawdown_net", "rolling_vol_latest", "rolling_sharpe_latest", "estimated_monthly_cost_drag"])
            _display_table(t["backtest_curve"].tail(20))
            _chart_pack([
                ("backtest", "Backtest curve", "Top-ranked strategy gross/net versus benchmark."),
                ("equity_curve", "Universe equity curve", "Normalized price reference for all tickers."),
                ("drawdown", "Drawdown", "Drawdown context for the universe."),
                ("rolling_volatility", "Rolling volatility", "Rolling 63-day annualized volatility."),
                ("rolling_sharpe", "Rolling Sharpe", "Rolling 63-day risk-adjusted performance."),
            ])

        with tab["Diagnostics"]:
            _panel_title("Diagnostics-first workflow", "Configuration, data coverage, source mix, missingness and governance diagnostics are kept visible before relying on the dashboard.")
            _display_table(t.get("diagnostics", pd.DataFrame()), rows=100)
            _display_table(t.get("data_quality", pd.DataFrame()), percent_cols=["fundamental_missing_ratio"], number_cols=["price_rows", "missing_price"], rows=100)
            _display_table(t.get("data_source_summary", pd.DataFrame()), rows=100)
            _display_table(t.get("risk_factor_metadata", pd.DataFrame()), rows=100)
            _display_table(t.get("feature_missingness", pd.DataFrame()), percent_cols=["missing_pct"], rows=100)
            _chart_block("data_coverage", "Data coverage chart", "Coverage and missingness by ticker.")
            display_diagnostics(t["diagnostics"])

        with tab["Outputs"]:
            _panel_title("Saved outputs and audit trail", "CSV tables, HTML Plotly charts, Markdown report and MASTERREQUEST snapshot created by this run.")
            output_rows = []
            for key, value in outputs.items():
                if isinstance(value, dict):
                    for subkey, subvalue in value.items():
                        output_rows.append({"kind": key, "name": subkey, "path": subvalue})
                else:
                    output_rows.append({"kind": "table/report/config", "name": key, "path": value})
            _display_table(pd.DataFrame(output_rows), rows=200)
            _display_table(_chart_inventory(charts), rows=200)
            snapshot = build_config_snapshot()
            _panel_title("Configuration snapshot preview", "Reproducibility metadata saved with every run.")
            _display_table(pd.DataFrame([
                {"section": "MASTERREQUEST", "keys": len(snapshot.get("masterrequest", {}) or {})},
                {"section": "EXPERIMENT", "keys": len(snapshot.get("experiment", {}) or {})},
                {"section": "UNIVERSE", "keys": len(snapshot.get("universe", {}) or {})},
                {"section": "ML_CONFIG", "keys": len(snapshot.get("ml_config", {}) or {})},
                {"section": "PROJECT_DATABASE_CONFIG", "keys": len(snapshot.get("project_database_config", {}) or {})},
            ]))
            _panel_title("Project database integration", "Local repo catalog and external DB_BASE are surfaced together for auditability.")
            _display_table(t.get("project_database_overview", pd.DataFrame()), rows=100)
            _display_table(t.get("sws_portfolio_snapshot", pd.DataFrame()), rows=100)
            _panel_title("Company Valuation Standard Tables", "Data source summary, feature missingness and robustness checks mirror the company valuation notebook standard.")
            _display_table(t.get("data_source_summary", pd.DataFrame()), rows=100)
            _display_table(build_data_integration_plan(), rows=100)
            _display_table(t.get("feature_missingness", pd.DataFrame()), percent_cols=["missing_pct"], rows=100)
            _display_table(t.get("robustness_checks", pd.DataFrame()), rows=100)
            _panel_title("Methodology and warnings", "Transparent formulas and governance warnings are part of the exported research artifact.")
            display(HTML(build_portfolio_methodology_html()))
            display(HTML(build_warning_panel_html(t.get("data_quality", pd.DataFrame()), t.get("robustness_checks", pd.DataFrame()))))
            _panel_title("Data quality", "Coverage and source diagnostics are kept visible here so chart interpretation is tied to data reliability.")
            _display_table(t["data_quality"], percent_cols=["fundamental_missing_ratio"], number_cols=["price_rows", "missing_price"])
            _display_table(t["source_mix"])
            _display_table(t["coverage_audit"])
            _chart_block("data_coverage", "Data coverage chart", "Price-row coverage and fundamental missingness by ticker.")
            display_diagnostics(t["diagnostics"])
            display(HTML(f"<div class='ir-card'><b>Output root:</b><br>{OUTPUT_ROOT}</div>"))

    for widget in [selected_ticker_w, min_score_w, chart_selector_w]:
        widget.observe(_render_all, names="value")
    refresh_view_btn.on_click(_render_all)

    display(HTML("<div class='ir-dashboard-shell'><b>Dashboard controls:</b> choose focus ticker, minimum score threshold and spotlight chart. All charts below are interactive Plotly figures.</div>"))
    display(widgets.HBox([selected_ticker_w, min_score_w, chart_selector_w, refresh_view_btn]))
    display(view_tabs)
    _render_all()


In [ ]:
# 3.4 Run workflow

def run_research():
    global PORTFOLIOSELECTIONCONFIG
    if "PORTFOLIOSELECTIONCONFIG" not in globals():
        PORTFOLIOSELECTIONCONFIG = {"preset": "Top ranked allocation", "filters": [], "ranking": {"mode": "raw_sort", "sort_by": "composite_score", "ascending": False}}
    diagnostics = assert_ready_to_run()
    price_df, loaded, failed = load_price_layer(UNIVERSE["all_tickers"], EXPERIMENT["start_date"], EXPERIMENT["end_date"])
    price_metrics, pivot, returns = build_price_metrics(price_df)
    risk_factors, risk_factor_metadata = load_risk_factor_layer(EXPERIMENT["start_date"], EXPERIMENT["end_date"])
    returns_df = build_returns_matrix_from_universe(price_df, UNIVERSE, EXPERIMENT)
    portfolio_engine_result = unified_run_portfolio_engine(
        price_df,
        returns_df,
        PORTFOLIO_ENGINE_CONFIG,
        universe=UNIVERSE,
        experiment=EXPERIMENT,
        portfolio_config=PORTFOLIO_CONFIG,
        risk_config=RISK_CONFIG,
    )
    portfolio_engine_weights = portfolio_engine_result.get("weights", pd.DataFrame())
    portfolio_engine_frontier = portfolio_engine_result.get("frontier", pd.DataFrame())
    portfolio_engine_backtest = portfolio_engine_result.get("backtest", pd.DataFrame())
    portfolio_engine_metrics = pd.DataFrame([{"metric": k, "value": v} for k, v in portfolio_engine_result.get("metrics", {}).items()])
    portfolio_engine_diagnostics = pd.DataFrame([portfolio_engine_result.get("diagnostics", {})])
    ml_time_series_result = unified_run_ml_time_series_engine(
        price_df=price_df,
        returns_df=returns_df,
        universe=UNIVERSE,
        experiment=EXPERIMENT,
        time_series_config=TIME_SERIES_ENGINE_CONFIG,
        deep_stock_config=DEEP_STOCK_ENGINE_CONFIG,
        crypto_config=CRYPTO_ENGINE_CONFIG,
        crypto_price_df=None,
    )
    time_series_result = ml_time_series_result.get("time_series", {}) or {}
    deep_stock_result = ml_time_series_result.get("deep_stock", {}) or {}
    crypto_result = ml_time_series_result.get("crypto", {}) or {}
    time_series_forecast = time_series_result.get("forecast", pd.DataFrame())
    time_series_diagnostics = time_series_result.get("diagnostics", pd.DataFrame())
    time_series_anomalies = time_series_result.get("anomalies", pd.DataFrame())
    time_series_regimes = time_series_result.get("regimes", pd.DataFrame())
    deep_stock_signals = deep_stock_result.get("signals", pd.DataFrame())
    deep_stock_forecast = deep_stock_result.get("forecast", pd.DataFrame())
    deep_stock_diagnostics = deep_stock_result.get("diagnostics", pd.DataFrame())
    crypto_signals = crypto_result.get("signals", pd.DataFrame())
    crypto_forecast = crypto_result.get("forecast", pd.DataFrame())
    crypto_diagnostics = crypto_result.get("diagnostics", pd.DataFrame())
    fund_df = load_fundamental_layer(UNIVERSE["all_tickers"])
    data_quality, source_mix, coverage_audit = build_data_quality_layer(price_df, fund_df, loaded, failed)

    valuation_df = build_valuation_metrics(fund_df, price_metrics)
    refined_peers, similarity_df = refine_peers_with_similarity(valuation_df)
    if UNIVERSE["peer_mode"] in ["clustered", "factor-similar", "custom-screened"] and refined_peers:
        UNIVERSE["peers"] = refined_peers

    company = table_company_overview(valuation_df)
    peers = table_peer_comparison(valuation_df)
    scenarios = table_scenarios(valuation_df)
    sensitivity = table_sensitivity(valuation_df)
    factor_signals = table_factor_signals(valuation_df)
    sws_scorecard = table_sws_scorecard(valuation_df)
    feature_missingness = table_feature_missingness(valuation_df)
    data_source_summary = table_data_source_summary(price_df, fund_df, loaded, failed, source_mix)
    model_cmp, feat_imp = table_model_lab(valuation_df)
    model_leaderboard = build_model_leaderboard(model_cmp, feat_imp)
    bt_curve, bt_perf = table_backtest(valuation_df, pivot)
    sws_snapshot = build_sws_portfolio_snapshot(valuation_df, bt_perf, data_quality)
    portfolio_allocation = table_portfolio_allocation(valuation_df)
    portfolio_factor_exposure = table_portfolio_factor_exposure(portfolio_allocation, factor_signals)
    risk_factor_exposures = table_risk_factor_exposures(portfolio_allocation, pivot, risk_factors)
    factor_risk_summary = table_factor_risk_summary(risk_factor_exposures)
    risk_table = table_risk_dashboard(valuation_df, failed, factor_risk_summary=factor_risk_summary)
    robustness_checks = table_robustness_checks(valuation_df, data_quality, model_leaderboard, risk_table)
    benchmark_comparison = table_benchmark_comparison(bt_perf, bt_curve)
    optimization_summary = table_optimization_summary(valuation_df, portfolio_allocation)
    portfolio_scenarios = table_portfolio_scenarios(portfolio_allocation, scenarios)
    performance_summary = bt_perf.copy()
    project_database_overview = build_project_database_overview()

    diagnostics_extra = pd.DataFrame([
        {"check": "Price columns", "status": validate_required_columns(price_df, GOVERNANCE_CONFIG["required_price_columns"])["status"], "detail": str(validate_required_columns(price_df, GOVERNANCE_CONFIG["required_price_columns"]))},
        {"check": "Fundamental columns", "status": validate_required_columns(fund_df, GOVERNANCE_CONFIG["required_fundamental_columns"])["status"], "detail": str(validate_required_columns(fund_df, GOVERNANCE_CONFIG["required_fundamental_columns"]))},
        {"check": "Duplicate price keys", "status": check_duplicate_keys(price_df, GOVERNANCE_CONFIG["key_columns"])["status"], "detail": str(check_duplicate_keys(price_df, GOVERNANCE_CONFIG["key_columns"]))},
        {"check": "Loaded tickers", "status": "PASS" if len(loaded) > 0 else "WARN", "detail": ", ".join(loaded) if loaded else "synthetic fallback"},
        {"check": "Failed/fallback tickers", "status": "PASS" if len(failed) == 0 else "WARN", "detail": ", ".join(failed) if failed else "none"},
    ])
    diagnostics = pd.concat([diagnostics, diagnostics_extra], ignore_index=True)

    charts = build_charts(
        valuation_df,
        pivot,
        scenarios,
        sensitivity,
        bt_curve,
        model_cmp,
        feat_imp,
        risk_table,
        data_quality=data_quality,
        allocation=portfolio_allocation,
        optimization=optimization_summary,
        portfolio_scenarios=portfolio_scenarios,
        portfolio_factor_exposure=portfolio_factor_exposure,
        benchmark_comparison=benchmark_comparison,
        risk_factor_exposures=risk_factor_exposures,
        factor_risk_summary=factor_risk_summary,
        portfolio_engine_result=portfolio_engine_result,
        ml_time_series_result=ml_time_series_result,
    )

    tables = {
        "source_mix": source_mix,
        "coverage_audit": coverage_audit,
        "data_quality": data_quality,
        "company_overview": company,
        "peer_comparison": peers,
        "peer_similarity": similarity_df,
        "valuation_ranking": valuation_df,
        "scenario_framework": scenarios,
        "sensitivity": sensitivity,
        "factor_signals": factor_signals,
        "sws_scorecard": sws_scorecard,
        "feature_missingness": feature_missingness,
        "data_source_summary": data_source_summary,
        "robustness_checks": robustness_checks,
        "model_comparison": model_cmp,
        "model_leaderboard": model_leaderboard,
        "feature_importance": feat_imp,
        "backtest_curve": bt_curve,
        "backtest_performance": bt_perf,
        "risk_dashboard": risk_table,
        "diagnostics": diagnostics,
        "sws_portfolio_snapshot": sws_snapshot,
        "portfolio_allocation": portfolio_allocation,
        "portfolio_factor_exposure": portfolio_factor_exposure,
        "risk_factors": risk_factors,
        "risk_factor_metadata": risk_factor_metadata,
        "risk_factor_exposures": risk_factor_exposures,
        "factor_risk_summary": factor_risk_summary,
        "benchmark_comparison": benchmark_comparison,
        "optimization_summary": optimization_summary,
        "portfolio_scenarios": portfolio_scenarios,
        "performance_summary": performance_summary,
        "portfolio_engine_result": portfolio_engine_result,
        "portfolio_engine_weights": portfolio_engine_weights,
        "portfolio_engine_frontier": portfolio_engine_frontier,
        "portfolio_engine_backtest": portfolio_engine_backtest,
        "portfolio_engine_metrics": portfolio_engine_metrics,
        "portfolio_engine_diagnostics": portfolio_engine_diagnostics,
        "ml_time_series_result": ml_time_series_result,
        "time_series_forecast": time_series_forecast,
        "time_series_diagnostics": time_series_diagnostics,
        "time_series_anomalies": time_series_anomalies,
        "time_series_regimes": time_series_regimes,
        "deep_stock_signals": deep_stock_signals,
        "deep_stock_forecast": deep_stock_forecast,
        "deep_stock_diagnostics": deep_stock_diagnostics,
        "crypto_signals": crypto_signals,
        "crypto_forecast": crypto_forecast,
        "crypto_diagnostics": crypto_diagnostics,
        "project_database_overview": project_database_overview,
        "price_data": price_df,
        "fundamentals": fund_df,
    }
    # Portfolio-native Finviz-style selection layer: allocation-aware filters, ranking, audit and exports.
    globals()["RESEARCH_RESULT"] = {"tables": tables, "charts": charts, "outputs": {}}
    try:
        run_portfolio_selection_layer(globals(), PORTFOLIOSELECTIONCONFIG)
    except Exception as exc:
        logger.warning("Portfolio selection layer failed: %s", exc)
        tables["portfolio_selection_summary"] = pd.DataFrame([{"status": "WARN", "detail": str(exc)}])

    outputs = save_outputs({"tables": tables, "charts": charts})
    result = {"tables": tables, "charts": charts, "outputs": outputs}
    globals()["RESEARCH_RESULT"] = result
    render_dashboard(result)
    return result

def _run_button_clicked(_):
    with results_out:
        clear_output(wait=True)
        try:
            globals()["RESEARCH_RESULT"] = run_research()
        except Exception as exc:
            display(HTML(f"<div class='ir-status-fail'>Run error: {exc}</div>"))

run_btn.on_click(_run_button_clicked)
display(HTML("<div class='ir-status-pass'>Workflow callbacks are registered. Use the Control Center tabs above to apply inputs, run research and read the dashboard.</div>"))


## 3.5 Portfolio Selection Lab

Notebook-friendly filtering, ranking, audit and export layer for allocation candidates.


In [ ]:
# 3.5 Portfolio Selection Lab - notebook UI and dashboard hooks
if "PORTFOLIOSELECTIONCONFIG" not in globals():
    PORTFOLIOSELECTIONCONFIG = {"preset": "Top ranked allocation", "filters": [], "ranking": {"mode": "raw_sort", "sort_by": "composite_score", "ascending": False}}

portfolio_selection_schema = portfolio_selection_schema_frame()
portfolio_selection_presets = portfolio_selection_presets_frame()

if "RESEARCH_RESULT" in globals():
    portfolio_selection_run = run_portfolio_selection_layer(globals(), PORTFOLIOSELECTIONCONFIG)
else:
    portfolio_selection_run = None
    print("Run research first to populate RESEARCH_RESULT, then rerun this cell for selection outputs.")

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output

    preset_options = list(PORTFOLIO_PRESETS.keys()) or ["Top ranked allocation"]
    preset_widget = widgets.Dropdown(
        options=preset_options,
        value=PORTFOLIOSELECTIONCONFIG.get("preset") if PORTFOLIOSELECTIONCONFIG.get("preset") in preset_options else preset_options[0],
        description="Preset",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "100px"},
    )
    field_options = sorted({field for group in PORTFOLIO_FILTER_SCHEMA.values() for field in group.keys()}) if PORTFOLIO_FILTER_SCHEMA else ["composite_score"]
    field_widget = widgets.Dropdown(options=field_options, description="Field", layout=widgets.Layout(width="320px"), style={"description_width": "70px"})
    op_widget = widgets.Dropdown(options=["gt", "lt", "between", "equals", "contains", "top_pct", "bottom_pct"], value="gt", description="Op", layout=widgets.Layout(width="180px"), style={"description_width": "40px"})
    value_widget = widgets.Text(value="", placeholder="0.5 or 0,0.3 or Technology", description="Value", layout=widgets.Layout(width="340px"), style={"description_width": "60px"})
    ranking_widget = widgets.Dropdown(options=["preset", "raw_sort", "weighted_score", "risk_budget"], value="preset", description="Ranking", layout=widgets.Layout(width="260px"), style={"description_width": "80px"})
    run_selection_button = widgets.Button(description="Run selection", button_style="success", icon="filter")
    export_dashboard_button = widgets.Button(description="Export dashboard", button_style="info", icon="external-link")
    out = widgets.Output()
    active_filters = list(PORTFOLIOSELECTIONCONFIG.get("filters", []))

    def _parse_value(raw):
        raw = str(raw).strip()
        if not raw:
            return None
        if op_widget.value == "between":
            parts = [x.strip() for x in raw.replace(";", ",").split(",") if x.strip()]
            return [float(parts[0]), float(parts[1])] if len(parts) >= 2 else None
        if op_widget.value in {"in", "in_list"}:
            return [x.strip() for x in raw.replace(";", ",").split(",") if x.strip()]
        try:
            return float(raw)
        except Exception:
            return raw

    def _config():
        filters = list(active_filters)
        parsed = _parse_value(value_widget.value)
        if parsed is not None:
            filters.append({"field": field_widget.value, "op": op_widget.value, "value": parsed})
        cfg = {"preset": preset_widget.value, "filters": filters}
        if ranking_widget.value == "raw_sort":
            cfg["ranking"] = {"mode": "raw_sort", "sort_by": "composite_score", "ascending": False}
        elif ranking_widget.value == "weighted_score":
            cfg["ranking"] = {"mode": "weighted_score", "weights": {"composite_score": 0.35, "quality_score": 0.25, "risk_score": 0.25, "momentum_score": 0.15}}
        elif ranking_widget.value == "risk_budget":
            cfg["ranking"] = {"mode": "risk_budget"}
        return cfg

    def _render(_=None):
        global PORTFOLIOSELECTIONCONFIG, portfolio_selection_run
        with out:
            clear_output(wait=True)
            if "RESEARCH_RESULT" not in globals():
                display(HTML("<div class='ir-status-warn'>Run research first, then rerun this selection lab.</div>"))
                return
            PORTFOLIOSELECTIONCONFIG = _config()
            portfolio_selection_run = run_portfolio_selection_layer(globals(), PORTFOLIOSELECTIONCONFIG)
            display(HTML(f"<div class='ir-status-pass'>Selection ready: {len(portfolio_selection_results)} rows · preset {PORTFOLIOSELECTIONCONFIG.get('preset')}</div>"))
            display(portfolio_selection_summary)
            display(portfolio_selection_results.head(50))
            display(portfolio_selection_audit)

    def _export(_=None):
        _render()
        try:
            from portfolio_dashboard import export_dashboard_and_report
            artifacts = export_dashboard_and_report(globals())
            with out:
                display(HTML(f"<div class='ir-status-pass'>Dashboard exported: <a href='{artifacts.dashboard_path}' target='_blank'>{artifacts.dashboard_path}</a></div>"))
        except Exception as exc:
            with out:
                display(HTML(f"<div class='ir-status-warn'>Dashboard export failed: {exc}</div>"))

    run_selection_button.on_click(_render)
    export_dashboard_button.on_click(_export)
    display(HTML("<div class='ir-section'>Portfolio Selection Lab</div>"))
    display(widgets.VBox([
        preset_widget,
        widgets.HBox([field_widget, op_widget, value_widget]),
        ranking_widget,
        widgets.HBox([run_selection_button, export_dashboard_button]),
        out,
    ]))
    _render()
except Exception as exc:
    print("Portfolio selection UI unavailable:", exc)
    if "portfolio_selection_summary" in globals():
        display(portfolio_selection_summary)
        display(portfolio_selection_results.head(50))


## 4. Quick run

In [ ]:
# Optional non-interactive run

def quick_run():
    global MASTERREQUEST
    MASTERREQUEST = {
        "profile": "Quality",
        "main_ticker": "AAPL",
        "watchlist": WATCHLISTS["US Mega Cap"],
        "benchmark": "SPY",
        "peers": [],
        "peer_mode": "clustered",
        "start_date": date(2020, 1, 1),
        "end_date": date.today(),
        "test_start": date(2023, 1, 1),
        "market": "US",
        "currency": "USD",
        "horizon_months": 12,
        "models": ["ridge", "random_forest", "gradient_boosting"],
        "cost_bps": 12.0,
        "slippage_bps": 5.0,
        "tax_rate": 0.26,
        "turnover_limit": 0.30,
        "run_backtest": True,
        "run_explainability": True,
        "run_scenarios": True,
        "run_sensitivity": True,
    }
    sync_all_configs_from_user_selection()
    return run_research()

# RESEARCH_RESULT = quick_run()

## Smart Money Government Data Engine

Official-source-first layer for SEC 13F, SEC Form 4, 13D/13G, CFTC COT, Treasury TIC, USAspending, ESMA/ECB/TED and EU/Italian proxy coverage.

This section does not fabricate smart-money signals. If local official datasets are missing, it emits schema-compliant empty tables, coverage diagnostics and caveats. USA coverage is more centralized; EU/Italy coverage is federated and must be interpreted as complete, partial, proxy or unavailable depending on connector coverage.


In [ ]:
# Smart Money Government Data Engine - notebook bridge
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / 'src' / 'smart_money_engine').exists():
        PROJECT_ROOT = parent
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.smart_money_engine import run_smart_money_engine
except Exception:
    from smart_money_engine import run_smart_money_engine

financial_db_root = globals().get('FINANCIAL_DB_ROOT', None) or globals().get('DB_BASE', None)
output_root = Path(globals().get('OUTPUTROOT', PROJECT_ROOT / 'output')) / 'smart_money'

smart_money_outputs = run_smart_money_engine(
    financial_db_root=financial_db_root,
    output_root=output_root,
    max_files_per_source=5,
)

smart_money_scores = smart_money_outputs['smart_money_scores']
smart_money_event_feed = smart_money_outputs['event_feed']
smart_money_coverage = smart_money_outputs['coverage']
smart_money_source_registry = smart_money_outputs['source_registry']

print('Smart Money artifacts:', smart_money_outputs['manifest']['output_root'])
print('Score rows:', len(smart_money_scores), '| Event rows:', len(smart_money_event_feed))
display(smart_money_coverage)
display(smart_money_scores.head(25))


## ML Stock Lab Integration

This section delegates ML fair-value estimation, mispricing, ranking and quintile portfolio diagnostics to `ml_stock_lab`. The notebook remains the analytical authoring layer; `ml_stock_lab` provides reusable sklearn-like APIs and stable `MLStockLab_*` artifacts consumed by Streamlit.


In [ ]:
# ML Stock Lab bridge - fair value, mispricing and quintile diagnostics
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / "src" / "ml_stock_lab").exists():
        PROJECT_ROOT = parent
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ml_stock_lab import run_ml_stock_lab_experiment

ml_stock_lab_output_root = PROJECT_ROOT / "output" / "ml_stock_lab"
ml_stock_lab_result = run_ml_stock_lab_experiment(
    output_root=ml_stock_lab_output_root,
    financial_db_root=globals().get("FINANCIAL_DB_ROOT", None) or globals().get("DB_BASE", None),
    model="ols",
    max_rows=2000,
)
ml_stock_lab_metrics = ml_stock_lab_result.get("metrics")
ml_stock_lab_signals = ml_stock_lab_result.get("signals")
ml_stock_lab_quintiles = ml_stock_lab_result.get("quintiles")
print("ML Stock Lab status:", ml_stock_lab_result.get("status"))
print("ML Stock Lab output:", ml_stock_lab_output_root)
display(ml_stock_lab_metrics)
if ml_stock_lab_signals is not None:
    display(ml_stock_lab_signals.head(25))
